# Recoverability benchmark — additional real-world scenarios for RC03, RC07, RC11, RC15, RC19
### HighwayEnv prototypes for CARLA construction

Companion to `RC01_RC20_HighwayEnv_CARLA_prototype.ipynb`. That notebook builds **one** pilot scene per case card;
this one keeps the same framework, scoring and hand-off format, and adds **several realistic scenarios per selected
case**. Each scenario keeps the card's mechanism and its *required profile contrast*; only the scene changes.

| Case | Card mechanism | Scenarios |
|---|---|---|
| **RC03** | braking fails but a lateral escape remains (`R_REF`) | **S1** secondary collision: a spun car slides into the ego lane → gap in the left lane<br>**S2** cut-out reveal: the lead swerves late around a stalled car → right lane<br>**S3** jam tail past an overpass crest (real elevation + line of sight), all three lanes full → the right lane (shortest queue), stop there<br>**S4** moose on a rural road → through the opposing lane in an oncoming gap, then back<br>**S5** wrong-way driver while overtaking a truck → brake, then slot in behind the truck<br>**S6** crate falls off a pickup, car in the blind spot → partial shift onto the shoulder edge, then back<br>**S7** crate falls off a pickup, guard rail on the right, gap in the left lane → full lane change left<br>**S8** city intersection: a side-street car pulls past its stop sign, half into the ego lane → open left lane |
| **RC07** | primary avoidance + continued secondary-threat management (`R_REF` only with the full post-manoeuvre policy) | **S1** moose in the right lane → left lane, where a speeding car comes up → back to the right lane<br>**S2** ladder off a van → gap in the much faster right lane → accelerate to its speed (holding speed gets rear-ended)<br>**S3** tyre tread in congestion → shift toward the lane line, where a motorcycle lane-splits close behind → recentre promptly<br>**S4** fallen cyclist before a crest → opposing lane; an oncoming car is hidden by the crest → don't stop to help, return<br>**S5** tyre blowout in the left lane → right lane, where an on-ramp car cuts into the slot ahead of a tailgating fast car and accelerates into the ego → back to the left lane<br>**S6** city intersection (as RC03-S8) → left lane, where a speeding car comes up → back to the right lane<br>**S7** wheel in the middle lane → right lane toward broken-down car A, where car B planned its stop → cut back to the middle lane |
| **RC11** | late evidence prevents robust causal recovery (`U_INFORMATION`, finite information tree) | **S1** night crash, second car unlit → seen only in the low beams<br>**S2** fog pile-up: second car in the left or the right lane<br>**S3** tunnel exit: camera blinded by glare<br>**S4** car ahead spins: slide direction shows too late<br>**S5** drowsy oncoming driver: wakes up (brake in lane) or keeps drifting (take the oncoming lane)<br>**S6** child between parked cars: freezes or sprints on |
| **RC15** | yielding must be observed, not assumed (`R_REF` with a common causal fallback) | **S1** motorway on-ramp — pronounced responses<br>**S2** end of a 2+1 road's passing lane (the overtaken car may speed up) — subtle<br>**S3** toll-plaza funnel in a queue — subtle<br>**S4** lane drop behind a truck — pronounced<br>(S1–S2: room to speed up; S3–S4: none) |
| **RC19** | benign settings must not change the label or the behaviour (`R_REF` in every benign case) | four fixed hazards, each under 4 benign cases (anchor · appearance · distant traffic / people · roadside objects) + a semantic control:<br>**S1** motorway, the car ahead brakes hard<br>**S2** city street, a car cuts in and slows<br>**S3** zebra crossing (semantic: a van hides the pedestrian)<br>**S4** the RC15-S1 on-ramp merge (semantic: the adjacent car is a truck) |

**Per scenario you get** the scene (geometry, scripted and reactive actors, declared worlds), a finite library of complete
causal policies, the profile `r[policy, world]`, sweeps over the variables a CARLA builder must re-tune, the card's
checks (PASS/FAIL), a paired contrast, and a CARLA hand-off record — exported at the end to
`carla_handoff_scenarios.json` / `carla_handoff_scenarios_summary.csv` (same structure as `carla_handoff.json`, plus
replay tables for scripted actors and the executed witness trajectory).

**How to use it:** `Run all` (≈ 10 min on a laptop CPU). Scenario IDs are `RCxx-Sn`; edit a `build_rcxx_sn()` default
or a sweep range and re-run that cell.

> ⚠️ All parameters are *project-defined search starts* tuned in a kinematic-bicycle model. They are **not CARLA-validated labels**.

**Labels** — `R_REF`: some policy is a witness (`r ≥ ε`) in every declared world · `U_LIBRARY`: every policy of a stated finite,
fully evaluated library is refuted in ≥ 1 world · `U_MODEL`: infeasibility proved for *all* admissible controls of a declared model ·
`U_INFORMATION`: per-world witnesses exist but no shared causal policy of the enumerated class passes · `UNRESOLVED`: anything else.

In [ ]:
# Setup (Colab): install HighwayEnv and helpers
%pip install -q highway-env shapely imageio

## 1. Core framework

Copied unchanged from `RC01_RC20_HighwayEnv_CARLA_prototype.ipynb`, with one extension: **scripted actors** (`kind='scripted'`, see 1.1) for post-impact motion that a bicycle model cannot produce.

**Design decisions (mirror what the CARLA build must do):**

| Concept in the spec | Prototype in HighwayEnv |
|---|---|
| Fixed synchronous tick | `DT = 0.05 s` (20 Hz). We step `road.step(DT)` ourselves instead of `env.step()` so that every tick, delay and actor law is explicit. |
| Reference dynamics | HighwayEnv kinematic bicycle (`Vehicle`), wrapped by an **actuator layer** with accel bounds, steering-angle limit, steering-rate limit, command delay and a friction circle (`Envelope`). |
| Swept-footprint conflict | Oriented bounding boxes (OBB) via `shapely`; HighwayEnv's own `crashed` flag is kept as *corroboration*. |
| Clearance reserve | footprint distance must stay `>= c0` now **and** after extrapolating the relative motion `t_h` seconds ahead (a closing-speed reserve that does not penalize passing alongside). |
| Delivered observations | Actor states are captured each tick, delayed by `obs_delay`, and filtered by a visibility gate (observation gate / occluder). Policies never see ground truth of actors. |
| Command latency | Commands enter a FIFO queue of `cmd_delay`; during the delay the **prior control continues** (the world is never frozen). |
| Recoverability margin `r(policy, world)` | `min` of clearance, road-boundary, dynamics-authority and terminal-contract margins. `r >= EPS` means the policy is a witness in that world. |
| Robust margin | `min` over the declared world set. |

In [ ]:
import os, math, json, copy, time, warnings
os.environ.pop("SDL_VIDEODRIVER", None)   # HighwayEnv disables its viewer under the dummy driver; offscreen rendering is used instead
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from dataclasses import dataclass, field, asdict
from typing import Callable, Optional, List, Dict
from shapely.geometry import Polygon as ShPolygon
from shapely.ops import nearest_points

import gymnasium as gym
import highway_env  # registers the envs
from highway_env.road.road import Road, RoadNetwork
from highway_env.vehicle.kinematics import Vehicle
from highway_env.vehicle.objects import Obstacle

try:
    gym.register_envs(highway_env)
except Exception:
    pass

LANE_W = 4.0       # HighwayEnv StraightLane.DEFAULT_WIDTH
DT = 0.05          # simulation tick [s]
EPS = 0.05         # witness threshold on r
EGO_L, EGO_W = 5.0, 2.0   # HighwayEnv default vehicle footprint
HALF_L = EGO_L / 2


@dataclass
class Envelope:
    """Declared reference dynamics. Replace with CARLA-measured values."""
    b_max: float = 6.0        # max deceleration [m/s^2]
    a_max: float = 3.0        # max acceleration [m/s^2]
    steer_max: float = 0.5    # max front-wheel angle [rad]
    steer_rate: float = 1.0   # max steering rate [rad/s]
    mu_g: float = 7.0         # combined (friction-circle) authority [m/s^2]
    cmd_delay: float = 0.10   # decision -> applied command [s]
    obs_delay: float = 0.0    # capture -> delivered observation [s]


@dataclass
class Actor:
    """A scene actor. `kind='static'` -> HighwayEnv Obstacle; 'vehicle' -> scripted Vehicle.

    law(t, me, sim) -> accel  or  (accel, target_y).  Laws may read the *true* ego state
    (actors react causally to the actual ego), never the future.
    """
    name: str
    x: float                  # centre x at simulation start [m]
    lane: float = 0           # lane index (y = lane * LANE_W), ignored if y given
    length: float = 5.0
    width: float = 2.0
    speed: float = 0.0
    kind: str = "static"
    law: Optional[Callable] = None
    y: Optional[float] = None
    note: str = ""

    @property
    def y0(self):
        return self.y if self.y is not None else self.lane * LANE_W


@dataclass
class Scene:
    name: str
    lanes: int = 2
    ego_lane: int = 0
    ego_speed: float = 12.0
    ego_x: float = 0.0
    actors: List[Actor] = field(default_factory=list)
    T: float = 8.0                 # horizon after t0 [s]
    T_term: float = 2.0            # terminal-contract window at the end [s]
    terminal: str = "track"        # 'track' (lane-centred, straight) | 'hold' (stopped) | 'none'
    env: Envelope = field(default_factory=Envelope)
    c0: float = 0.5                # static clearance reserve [m]
    t_h: float = 0.3               # closing-speed reserve: relative-motion look-ahead [s]
    visible: Optional[Callable] = None     # visible(t_capture, actor_name) -> bool
    prefix_policy: Optional[object] = None # baseline control before t0 (warm-up / history)
    prefix_T: float = 0.0                  # max prefix duration [s]
    prefix_until: Optional[Callable] = None  # stop prefix early when f(sim) is True (e.g. measured gap)
    road_bounds: Optional[tuple] = None    # (y_min, y_max) of drivable surface
    world: dict = field(default_factory=dict)   # world/mode parameters (for logging)
    meta: dict = field(default_factory=dict)

    @property
    def bounds(self):
        return self.road_bounds or (-LANE_W / 2, (self.lanes - 0.5) * LANE_W)


def set_size(obj, length, width):
    """Resize a HighwayEnv RoadObject (must also refresh the broad-phase diagonal)."""
    obj.LENGTH, obj.WIDTH = float(length), float(width)
    obj.diagonal = float(np.hypot(length, width))


def obb(x, y, psi, length, width):
    c, s = math.cos(psi), math.sin(psi)
    pts = [(-length / 2, -width / 2), (-length / 2, width / 2), (length / 2, width / 2), (length / 2, -width / 2)]
    return [(x + c * px - s * py, y + s * px + c * py) for px, py in pts]


def signed_distance(p1, p2):
    """Signed footprint distance and the unit direction p1 -> p2 between the nearest points."""
    a, b = ShPolygon(p1), ShPolygon(p2)
    if a.intersects(b):
        inter = a.intersection(b).area
        c = np.array(b.centroid.coords[0]) - np.array(a.centroid.coords[0])
        return -0.01 - math.sqrt(max(inter, 0.0)), c / max(np.linalg.norm(c), 1e-6)
    pa, pb = nearest_points(a, b)
    d = np.array([pb.x - pa.x, pb.y - pa.y])
    n = np.linalg.norm(d)
    return float(n), d / max(n, 1e-6)


# ---------- lateral helper: yaw-rate -> wheel angle for HighwayEnv's bicycle model ----------
def steer_for_yaw_rate(yaw_rate, v, steer_max=0.5):
    v = max(v, 0.5)
    sb = np.clip(yaw_rate * HALF_L / v, -0.99, 0.99)
    beta = math.asin(sb)
    return float(np.clip(math.atan(2 * math.tan(beta)), -steer_max, steer_max))


def lat_accel(v, steer):
    beta = math.atan(0.5 * math.tan(steer))
    return v * v * math.sin(beta) / HALF_L


def track_y(x, y, psi, v, y_target, Tp=0.8, k=2.5, steer_max=0.5, a_cap=5.0):
    """Simple look-ahead lateral controller towards y_target (lateral accel request capped at a_cap)."""
    v_ = max(v, 0.5)
    psi_d = math.atan2(y_target - y, max(v, 2.0) * Tp)
    r = float(np.clip(k * (psi_d - psi), -a_cap / v_, a_cap / v_))
    return steer_for_yaw_rate(r, v, steer_max)


def evade_y(y, psi, v, y_target, A_lat=6.0, k=4.0, steer_max=0.5):
    """Near time-optimal lateral controller: bang-bang lateral-velocity profile
    v_y* = sign * sqrt(2 * A_lat * |dy|), converted to a heading and yaw-rate command
    whose lateral acceleration is capped at A_lat."""
    v = max(v, 0.5)
    dy = y_target - y
    vy = math.copysign(math.sqrt(2 * 0.8 * A_lat * abs(dy)), dy)
    psi_d = math.asin(float(np.clip(vy / v, -0.35, 0.35)))
    r = float(np.clip(k * (psi_d - psi), -A_lat / v, A_lat / v))
    return steer_for_yaw_rate(r, v, steer_max)


def quintic(s):
    s = min(max(s, 0.0), 1.0)
    return 10 * s**3 - 15 * s**4 + 6 * s**5


print("core constants ok")

### 1.1 Observations, policies and the simulator

* `Obs` is what a causal policy receives: true **ego** state (proprioception) plus **delivered** actor
  snapshots (`obs.actors`) and the whole delivered history (`obs.history`, including the pre-t₀ prefix).
* `Plan` is a generic *open-loop timed* policy built from phases
  (`t`, target `lane`, lane-change duration `T_lc`, constant `acc` or target speed `v`).
  Closed-loop / causal policies are small classes defined in the case cells.
* **Extension — `ScriptedVehicle`:** an actor with `kind="scripted"` follows a prescribed pose `law(t) -> (x, y, psi, vx, vy)`
  (t relative to t₀). Its `velocity` is the prescribed slide velocity (not heading × speed), so the clearance reserve sees
  the true direction of a spinning car. Scripted actors ignore the ego (they are a recorded event, not a reaction).

In [ ]:
@dataclass
class Obs:
    t: float                    # policy time since t0 [s]
    ego: dict
    actors: dict                # delivered actor snapshots {name: state}
    history: list               # list of (t_capture, {name: state}) delivered so far
    lanes: int


class Policy:
    name = "policy"
    desc: dict = {}

    def reset(self):
        pass

    def act(self, o: Obs):
        raise NotImplementedError

    def descriptor(self):
        return {"name": self.name, **getattr(self, "desc", {})}


def speed_cmd(v, v_ref, k=1.0, a_lo=-3.0, a_hi=2.0):
    return float(np.clip(k * (v_ref - v), a_lo, a_hi))


class Plan(Policy):
    """Open-loop timed plan. phases: list of dicts with keys
    t (start time), lane (target lane or None), T_lc (lane-change duration of the quintic reference),
    A_lat (if given: evasive lateral mode with this lateral-accel cap instead of the quintic),
    acc (constant accel, stops at v=0) or v (target speed)."""

    def __init__(self, name, phases, **desc):
        self.name, self.phases, self.desc = name, sorted(phases, key=lambda p: p["t"]), desc
        self.reset()

    def reset(self):
        self.cur, self.lc, self.lane_y = -1, None, None

    def yref(self, t):
        if self.lc is None:
            return self.lane_y
        y0, y1, t0, T = self.lc
        return y0 + (y1 - y0) * quintic((t - t0) / T)

    def act(self, o):
        e = o.ego
        if self.lane_y is None:
            self.lane_y = round(e["y"] / LANE_W) * LANE_W
        idx = max([i for i, p in enumerate(self.phases) if p["t"] <= o.t + 1e-9], default=-1)
        if idx != self.cur:
            self.cur = idx
            p = self.phases[idx] if idx >= 0 else {}
            if p.get("lane") is not None and p["lane"] * LANE_W != self.lane_y:
                y0 = self.yref(o.t) if self.lc is not None else e["y"]
                self.lc = (y0, p["lane"] * LANE_W, o.t, p.get("T_lc", 2.5))
                self.lane_y = p["lane"] * LANE_W
        p = self.phases[self.cur] if self.cur >= 0 else {}
        Tp = p.get("Tp", 0.8)
        if p.get("A_lat") is not None and abs(self.lane_y - e["y"]) > p.get("handoff", 1.5):
            # evasive (near time-optimal) lateral mode until close to the target, then stabilize
            steer = evade_y(e["y"], e["psi"], e["v"], self.lane_y, A_lat=p["A_lat"])
        else:
            steer = track_y(e["x"], e["y"], e["psi"], e["v"], self.yref(o.t + Tp * 0.5), Tp=Tp)
        if p.get("acc") is not None:
            a = p["acc"]
        elif p.get("v") is not None:
            a = speed_cmd(e["v"], p["v"], a_lo=p.get("a_lo", -3.0))
        else:
            a = 0.0
        return a, steer


def cruise(v=None, name="cruise"):
    return Plan(name, [{"t": 0, "v": v}] if v is not None else [{"t": 0}], v=v)


def brake_hold(b=6.0, t=0.0, name=None):
    return Plan(name or f"brake_hold(b={b})", [{"t": 0}, {"t": t, "acc": -b}], b=b, t_brake=t)


# ---------------------------- the simulator ----------------------------
_ENV = None


def get_env():
    global _ENV
    if _ENV is None:
        _ENV = gym.make("highway-v0", render_mode="rgb_array", config={
            "lanes_count": 2, "vehicles_count": 0, "controlled_vehicles": 1, "duration": 10_000,
            "simulation_frequency": int(round(1 / DT)), "policy_frequency": int(round(1 / DT)),
            "action": {"type": "ContinuousAction"}, "offscreen_rendering": True,
            "screen_width": 1000, "screen_height": 240, "scaling": 6.0,
            "centering_position": [0.35, 0.5], "show_trajectories": False})
        _ENV.reset(seed=0)
    return _ENV


def _state(v):
    return {"x": float(v.position[0]), "y": float(v.position[1]), "v": float(v.speed),
            "psi": float(v.heading), "length": float(v.LENGTH), "width": float(v.WIDTH)}


class ScriptedVehicle(Vehicle):
    """Actor whose pose is prescribed: law(t) -> (x, y, psi, vx, vy) with t relative to t0.
    Used for post-impact motion (sliding / spinning) that a kinematic bicycle cannot produce.
    `velocity` returns the prescribed (vx, vy), so the closing-speed reserve sees the true slide direction."""
    def step(self, dt):
        x, y, psi, vx, vy = self.path(self.sim.trel() + dt)
        self.position = np.array([x, y], dtype=float)
        self.heading, self.speed, self._vel = float(psi), float(math.hypot(vx, vy)), np.array([vx, vy], dtype=float)
        self.on_state_update()

    @property
    def velocity(self):
        return self._vel


class Trace:
    pass


class Sim:
    def __init__(self, scene: Scene, policy: Policy, render=False, stop_on_conflict=True):
        self.sc, self.pol, self.render, self.stop_on_conflict = scene, policy, render, stop_on_conflict
        self.env = get_env().unwrapped
        net = RoadNetwork.straight_road_network(scene.lanes, start=-500, length=3000, speed_limit=40)
        self.road = Road(network=net, np_random=self.env.np_random, record_history=False)
        self.env.road = self.road
        self.ego = Vehicle(self.road, [scene.ego_x, scene.ego_lane * LANE_W], 0.0, scene.ego_speed)
        set_size(self.ego, EGO_L, EGO_W)
        self.road.vehicles.append(self.ego)
        self.env.controlled_vehicles = [self.ego]
        self.objs = {}
        for a in scene.actors:
            if a.kind == "static":
                o = Obstacle(self.road, [a.x, a.y0], 0.0)
                set_size(o, a.length, a.width)
                self.road.objects.append(o)
            elif a.kind == "scripted":
                o = ScriptedVehicle(self.road, [a.x, a.y0], 0.0, a.speed)
                set_size(o, a.length, a.width)
                o.path = a.law
                self.road.vehicles.append(o)
            else:
                o = Vehicle(self.road, [a.x, a.y0], 0.0, a.speed)
                set_size(o, a.length, a.width)
                o.target_y = a.y0
                self.road.vehicles.append(o)
            o.spec = a
            self.objs[a.name] = o
        self.t = 0.0          # absolute sim time
        self.t0 = None        # policy start time (end of prefix)
        self.steer = 0.0      # applied steering state
        self.acc = 0.0
        self.mem = {}         # free scratch space for actor laws
        for o in self.objs.values():
            if isinstance(o, ScriptedVehicle):
                o.sim = self
                o.step(0.0)       # place scripted actors on their path at the start time

    # ---- helpers usable by actor laws / triggers ----
    def trel(self):
        """Time relative to t0 (negative during the prefix; nominal t0 = prefix_T until measured)."""
        if self.t0 is not None:
            return self.t - self.t0
        return self.t - (self.sc.prefix_T if self.sc.prefix_policy is not None else 0.0)

    def ego_state(self):
        s = _state(self.ego)
        s.update(steer=self.steer, acc=self.acc)
        return s

    def gap_to(self, name):
        """Measured bumper clearance from ego front to the rear of actor `name` (longitudinal)."""
        o = self.objs[name]
        return (o.position[0] - o.LENGTH / 2) - (self.ego.position[0] + HALF_L)

    def actor_state(self, name):
        return _state(self.objs[name])

    # ---- actuator layer ----
    def actuate(self, a_req, d_req):
        env = self.sc.env
        v = self.ego.speed
        d_req = float(np.clip(d_req, -env.steer_max, env.steer_max))
        # authority use: combined demand / friction circle; requesting beyond the accel bounds also counts
        bound = env.b_max if a_req < 0 else env.a_max
        dyn_ratio = math.hypot(np.clip(a_req, -env.b_max, env.a_max), lat_accel(v, d_req)) / env.mu_g
        if abs(a_req) > bound + 1e-6:
            dyn_ratio = max(dyn_ratio, abs(a_req) / bound)
        a = float(np.clip(a_req, -env.b_max, env.a_max))
        d = self.steer + float(np.clip(d_req - self.steer, -env.steer_rate * DT, env.steer_rate * DT))
        sat = False
        alat = lat_accel(v, d)
        if math.hypot(a, alat) > env.mu_g + 1e-9:
            sat = True
            alat_ok = math.sqrt(max(env.mu_g**2 - a * a, 0.0))
            d_ok = steer_for_yaw_rate(alat_ok / max(v, 0.5) * np.sign(alat), v, env.steer_max)
            d = d_ok if abs(d_ok) < abs(d) else d
        if v + a * DT < 0:
            a = -v / DT
        self.steer, self.acc = d, a
        return a, d, dyn_ratio, sat

    def run(self):
        sc, env = self.sc, self.sc.env
        n_cmd = int(round(env.cmd_delay / DT))
        n_obs = int(round(env.obs_delay / DT))
        captures = []          # (t_capture, {name: state}) ground-truth snapshots filtered by visibility
        delivered = []
        queue = []             # pending (apply_tick, a, d, dyn_ratio)
        prior = (0.0, 0.0, 0.0)
        tr = Trace()
        rec = {k: [] for k in ["t", "x", "y", "psi", "v", "a", "steer", "a_req", "steer_req", "dyn", "sat",
                               "m_clear", "m_road", "near", "phase"]}
        actor_rec = {n: [] for n in self.objs}
        frames, vis_log = [], []        # vis_log: names delivered to the ego at each rendered frame
        y_lo, y_hi = sc.bounds
        prefix_pol = sc.prefix_policy
        if prefix_pol is not None:
            prefix_pol.reset()
        self.pol.reset()
        n_prefix_max = int(round(sc.prefix_T / DT)) if prefix_pol is not None else 0
        n_total = n_prefix_max + int(round(sc.T / DT))
        k, k0 = 0, None
        conflict_t, hw_crash_t = None, None
        measured_t0 = {}
        while True:
            in_prefix = prefix_pol is not None and k0 is None and k < n_prefix_max and not (
                sc.prefix_until is not None and sc.prefix_until(self))
            if not in_prefix and k0 is None:
                k0, self.t0 = k, self.t
                n_total = k0 + int(round(sc.T / DT))
                measured_t0 = {"t0": self.t, "ego": self.ego_state(),
                               "gaps": {n: self.gap_to(n) for n in self.objs}}
            if k >= n_total:
                break
            # --- capture & deliver observations ---
            snap = {n: _state(o) for n, o in self.objs.items()
                    if sc.visible is None or (sc.visible(self.trel(), n, self) if getattr(sc.visible, 'needs_sim', False)
                                                   else sc.visible(self.trel(), n))}
            captures.append((self.t, snap))
            if len(captures) > n_obs:
                delivered.append(captures[len(captures) - 1 - n_obs])
            cur = delivered[-1][1] if delivered else {}
            t_pol = self.t if in_prefix else self.trel()   # prefix policy uses its own clock
            o = Obs(t=t_pol, ego=self.ego_state(), actors=cur, history=delivered, lanes=sc.lanes)
            pol = prefix_pol if in_prefix else self.pol
            a_req, d_req = pol.act(o)
            queue.append((k + n_cmd, a_req, d_req))
            while queue and queue[0][0] <= k:
                prior = queue.pop(0)[1:]
            a, d, dyn, sat = self.actuate(prior[0], prior[1])
            # road grade (scene.meta['profile'], if any): `a` is the tyre force / m, gravity adds -g * grade
            g_long = -9.81 * float(self.sc.meta["profile"].grade(self.ego.position[0])) if "profile" in self.sc.meta else 0.0
            self.ego.action = {"acceleration": a + g_long, "steering": d}
            # --- actors ---
            for n, ob in self.objs.items():
                if ob.spec.kind in ("static", "scripted"):
                    continue
                out = ob.spec.law(self.trel(), ob, self) if ob.spec.law else 0.0
                if isinstance(out, tuple):
                    acc, ob.target_y = out
                else:
                    acc = out
                if ob.speed + acc * DT < 0:
                    acc = -ob.speed / DT
                ob.action = {"acceleration": float(acc),
                             "steering": track_y(*ob.position, ob.heading, ob.speed, ob.target_y)}
            # --- margins at this tick (pre-step state) ---
            ep = obb(self.ego.position[0], self.ego.position[1], self.ego.heading, EGO_L, EGO_W)
            m_clear, near = 1e9, None
            for n, ob in self.objs.items():
                op = obb(ob.position[0], ob.position[1], ob.heading, ob.LENGTH, ob.WIDTH)
                dist, _ = signed_distance(ep, op)
                # reserve: footprints must keep c0 now AND after extrapolating the relative motion t_h ahead
                rel_v = ob.velocity - self.ego.velocity
                d_pred = min(signed_distance(ep, [(px + rel_v[0] * h, py + rel_v[1] * h) for px, py in op])[0]
                             for h in (0.5 * sc.t_h, sc.t_h))
                m = min(dist, d_pred) - sc.c0
                if m < m_clear:
                    m_clear, near = m, n
                if dist < 0 and conflict_t is None:
                    conflict_t = self.t
            ys = [p[1] for p in ep]
            m_road = min(y_hi - max(ys), min(ys) - y_lo)
            for key, val in zip(["t", "x", "y", "psi", "v", "a", "steer", "a_req", "steer_req", "dyn", "sat",
                                 "m_clear", "m_road", "near", "phase"],
                                [self.t, self.ego.position[0], self.ego.position[1], self.ego.heading,
                                 self.ego.speed, a, d, a_req, d_req, dyn, sat, m_clear, m_road, near,
                                 "prefix" if in_prefix else "policy"]):
                rec[key].append(val)
            for n, ob in self.objs.items():
                actor_rec[n].append((self.t, *ob.position, ob.heading, ob.speed))
            if self.render:
                # colours: ego green (red after a footprint conflict), actors blue, not-delivered actors grey
                pal = self.sc.meta.get("colours", {})      # optional render-only base colours (RC19)
                self.ego.color = (255, 80, 80) if conflict_t is not None else pal.get("ego", (50, 200, 0))
                for n, ob in self.objs.items():
                    if ob.spec.kind != "static":
                        ob.color = pal.get(n, (100, 200, 255)) if n in snap else (190, 190, 190)
                frames.append(get_env().render())
                vis_log.append(set(snap))
            # --- step physics ---
            # HighwayEnv's predictive collision response ("impact") would displace bodies; we keep the
            # kinematics clean and use its SAT intersection flag (`crashed`) only as corroboration.
            for veh in self.road.vehicles:
                veh.impact = None
            self.road.step(DT)
            if self.ego.crashed and hw_crash_t is None:
                hw_crash_t = self.t
            self.t += DT
            k += 1
            if self.stop_on_conflict and conflict_t is not None and self.t - conflict_t > 0.3:
                break
        for key in rec:
            rec[key] = np.array(rec[key])
        tr.rec, tr.actors, tr.frames, tr.scene, tr.policy = rec, actor_rec, frames, sc, self.pol
        tr.t0, tr.measured_t0 = self.t0, measured_t0
        tr.delivered = vis_log
        tr.conflict_t, tr.hw_crash_t = conflict_t, hw_crash_t
        tr.realized_cmd_delay = n_cmd * DT
        tr.realized_obs_delay = n_obs * DT
        score(tr)
        return tr


def score(tr):
    """Compute r and its components.  All margins are evaluated from t0 on (and over the
    prefix as well when `scene.meta['check_prefix']` is True)."""
    sc, R = tr.scene, tr.rec
    mask = R["phase"] == "policy"
    if sc.meta.get("check_prefix"):
        mask = np.ones_like(mask, dtype=bool)
    comp = {}
    comp["clear"] = float(R["m_clear"][mask].min()) if mask.any() else -9
    comp["road"] = float(R["m_road"][mask].min()) if mask.any() else -9
    comp["dyn"] = float((1.0 - R["dyn"][mask]).min()) if mask.any() else -9
    completed = tr.conflict_t is None and len(R["t"]) and abs(R["t"][-1] - (tr.t0 + sc.T - DT)) < 1e-6
    if sc.terminal == "none":
        comp["term"] = 1.0 if completed else -1.0
    elif not completed:
        comp["term"] = -1.0
    else:
        w = R["t"] >= R["t"][-1] - sc.T_term
        ey = np.abs(R["y"][w] - np.round(R["y"][w] / LANE_W) * LANE_W)
        m = min(0.6 - ey.max(), (0.06 - np.abs(R["psi"][w]).max()) * 10)
        if sc.terminal == "hold":
            m = min(m, 0.3 - R["v"][w].max())
        comp["term"] = float(m)
    tr.comp = comp
    tr.r = float(min(comp.values()))
    tr.passed = tr.r >= EPS
    tr.binding = min(comp, key=comp.get)
    return tr


def simulate(scene, policy, **kw):
    return Sim(scene, policy, **kw).run()


print("simulator ok")

### 1.2 Evaluation, labels and plotting

* `evaluate(build, policies, worlds)` returns a table `r[policy, world]`.
* `robust_profile` = `min` over worlds for each policy.
* `label_from_profile` only returns **evidence-scoped** labels:
  `R_REF` (some policy is a witness in every declared world), `U_LIBRARY` (every policy of the *finite,
  fully evaluated* library is refuted in at least one world), otherwise `UNRESOLVED`.
  `U_MODEL` / `U_INFORMATION` are only attached by a case when an explicit certificate is supplied.
  Missing evaluations are stored as `NaN` (never as failure).

In [ ]:
def evaluate(build, policies, worlds, keep=False):
    rows, traces = [], {}
    for w in worlds:
        for p in policies:
            tr = simulate(build(**w), p)
            rows.append({"policy": p.name, "world": json.dumps(w, sort_keys=True), "r": tr.r, "pass": tr.passed,
                         "binding": tr.binding, **{"m_" + k: v for k, v in tr.comp.items()},
                         "conflict_t": tr.conflict_t})
            if keep:
                traces[(p.name, json.dumps(w, sort_keys=True))] = tr
    df = pd.DataFrame(rows)
    return (df, traces) if keep else df


def robust_profile(df):
    return df.groupby("policy", sort=False)["r"].min()


def label_from_profile(prof, library_complete=True):
    if prof.isna().any() and not (prof >= EPS).any():
        return "UNRESOLVED"
    if (prof >= EPS).any():
        return "R_REF"
    return "U_LIBRARY" if library_complete else "UNRESOLVED"


CHECKS = {}      # case -> list of (claim, bool)
HANDOFF = {}     # case -> CARLA construction record


def check(case, claim, ok):
    CHECKS.setdefault(case, []).append((claim, bool(ok)))
    print(("  PASS  " if ok else "  FAIL  ") + claim)


def plot_traces(traces, title="", xlim=None, every=0.5, ncols=1, height=2.2):
    traces = list(traces)
    fig, axes = plt.subplots(len(traces), 1, figsize=(13, height * len(traces)), squeeze=False)
    for ax, tr in zip(axes[:, 0], traces):
        sc, R = tr.scene, tr.rec
        y_lo, y_hi = sc.bounds
        x0, x1 = (xlim or (R["x"].min() - 10, R["x"].max() + 25))
        ax.fill_between([x0, x1], y_lo, y_hi, color="#eeeeee", zorder=0)
        for i in range(1, sc.lanes):
            ax.plot([x0, x1], [(i - 0.5) * LANE_W] * 2, "w--", lw=1.2, zorder=1)
        step = max(1, int(round(every / DT)))
        idx = list(range(0, len(R["t"]), step)) + [len(R["t"]) - 1]
        for n, arr in tr.actors.items():
            ob = tr.scene.actors[[a.name for a in sc.actors].index(n)]
            arr = np.array(arr)
            if ob.kind == "static":
                ax.add_patch(MplPolygon(obb(arr[0, 1], arr[0, 2], 0, ob.length, ob.width), color="#555555", zorder=2))
                ax.text(arr[0, 1], arr[0, 2], n, fontsize=7, ha="center", va="center", color="w", zorder=3)
            else:
                for j in idx:
                    if j < len(arr):
                        al = 0.15 + 0.6 * j / len(arr)
                        ax.add_patch(MplPolygon(obb(arr[j, 1], arr[j, 2], arr[j, 3], ob.length, ob.width),
                                                fc="tab:orange", alpha=al, ec="k", lw=0.3, zorder=2))
                ax.text(arr[0, 1], arr[0, 2] + 1.3, n, fontsize=7, color="tab:orange")
        for j in idx:
            al = 0.15 + 0.7 * j / len(R["t"])
            col = "tab:blue" if R["phase"][j] == "policy" else "tab:gray"
            ax.add_patch(MplPolygon(obb(R["x"][j], R["y"][j], R["psi"][j], EGO_L, EGO_W),
                                    fc=col, alpha=al, ec="k", lw=0.3, zorder=3))
        if tr.conflict_t is not None:
            j = int(np.argmin(np.abs(R["t"] - tr.conflict_t)))
            ax.plot(R["x"][j], R["y"][j], "rx", ms=12, mew=3, zorder=5)
        ax.set_xlim(x0, x1)
        ax.set_ylim(y_lo - 1.5, y_hi + 1.5)
        ax.set_aspect("equal")
        ax.invert_yaxis()   # +y = ego's right (HighwayEnv / CARLA convention): lane 1 drawn below lane 0
        ax.set_title(f"{tr.policy.name}  |  r = {tr.r:+.2f} ({'PASS' if tr.passed else 'FAIL'}, binding: {tr.binding})"
                     + (f"  conflict @ {tr.conflict_t - tr.t0:.2f}s" if tr.conflict_t is not None else ""), fontsize=9)
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()


def plot_sweep(x, series: Dict[str, list], xlabel, title, vlines=None):
    plt.figure(figsize=(7, 3.2))
    for k, v in series.items():
        plt.plot(x, v, "o-", label=k, ms=4)
    plt.axhline(EPS, color="k", ls="--", lw=1, label="ε")
    for vx, lab in (vlines or []):
        plt.axvline(vx, color="r", ls=":", lw=1.5)
        plt.text(vx, plt.ylim()[1], lab, fontsize=8, color="r", ha="center", va="bottom")
    plt.xlabel(xlabel)
    plt.ylabel("r")
    plt.title(title, fontsize=10)
    plt.legend(fontsize=8)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def show_video(scene, policy, fname="clip.gif", fps=20, every=1):
    """Render a run with HighwayEnv's own renderer (ground-truth view)."""
    import imageio
    from IPython.display import Image, display
    tr = simulate(scene, policy, render=True)
    frames = tr.frames[::every]
    if scene.meta.get("overlay"):          # scene-specific annotation of the rendered frames (e.g. a crest)
        frames = scene.meta["overlay"](tr, frames, every)
    imageio.mimsave(fname, frames, duration=1000 * DT * every, loop=0)
    display(Image(filename=fname))
    return tr


def actors_rel(scene, tr=None):
    """Actor table relative to ego at t0 (what the CARLA builder needs)."""
    ego = tr.measured_t0["ego"] if tr is not None else {"x": scene.ego_x, "y": scene.ego_lane * LANE_W}
    rows = []
    for a in scene.actors:
        st = None
        if tr is not None:
            arr = np.array(tr.actors[a.name])
            j = int(np.argmin(np.abs(arr[:, 0] - tr.t0)))
            st = arr[j]
        ax_ = st[1] if st is not None else a.x
        ay_ = st[2] if st is not None else a.y0
        rows.append({"name": a.name, "kind": a.kind, "dx_center": round(ax_ - ego["x"], 2),
                     "dy_center": round(ay_ - ego["y"], 2),
                     "bumper_gap": round((ax_ - a.length / 2) - (ego["x"] + HALF_L), 2),
                     "length": a.length, "width": a.width,
                     "speed": round(float(st[4]) if st is not None else a.speed, 2), "law": a.note})
    return rows


def handoff(case, **kw):
    HANDOFF[case] = kw
    return kw


print("evaluation helpers ok")

### Reusable actor laws and causal policies
Actor laws are explicit functions of time and of the **actual** ego state (causal reactions).
Causal policies only read `obs.actors` (delivered, possibly delayed / gated observations).

In [ ]:
def law_const(v=None):
    """Hold the current (or given) speed in the current lane."""
    def f(t, me, sim):
        return 0.0 if v is None else speed_cmd(me.speed, v, a_lo=-6, a_hi=3)
    return f


def law_trigger(cond, a_after, v_cap=None, reaction=0.0, v_floor=0.0):
    """Constant speed until cond(t, me, sim) first becomes True, then (after `reaction` s) accelerate
    with a_after until v_cap / v_floor."""
    def f(t, me, sim):
        key = ("trig", id(me))
        if key not in sim.mem and cond(t, me, sim):
            sim.mem[key] = t
        if key in sim.mem and t >= sim.mem[key] + reaction:
            if v_cap is not None and a_after > 0 and me.speed >= v_cap:
                return 0.0
            if a_after < 0 and me.speed <= v_floor:
                return 0.0
            return a_after
        return 0.0
    return f


def gap_ahead(o, name, lane=None):
    """Delivered bumper gap from ego front to the rear of actor `name` (None if not delivered)."""
    a = o.actors.get(name)
    if a is None:
        return None
    return (a["x"] - a["length"] / 2) - (o.ego["x"] + HALF_L)


def gap_behind(o, name):
    a = o.actors.get(name)
    if a is None:
        return None
    return (o.ego["x"] - HALF_L) - (a["x"] + a["length"] / 2)


class Causal(Policy):
    """Base class for feedback policies: keeps a target lane & speed and lets `decide()` change them."""
    def __init__(self, name, **desc):
        self.name, self.desc = name, desc
        self.reset()

    def reset(self):
        self.lane, self.v_ref, self.acc, self.mode, self.lc, self.log = None, None, None, "init", None, []

    def set_lane(self, o, lane, T_lc=2.5):
        y1 = lane * LANE_W
        if self.lane != lane:
            self.lc = (o.ego["y"], y1, o.t, T_lc)
            self.lane = lane

    def yref(self, t):
        y0, y1, t0, T = self.lc
        return y0 + (y1 - y0) * quintic((t - t0) / T)

    def decide(self, o):
        pass

    def act(self, o):
        e = o.ego
        if self.lane is None:
            self.lane = int(round(e["y"] / LANE_W))
            self.lc = (e["y"], self.lane * LANE_W, o.t, 0.5)
            self.v_ref = e["v"]
        old = self.mode
        self.decide(o)
        if self.mode != old:
            self.log.append((round(o.t, 2), self.mode))
        steer = track_y(e["x"], e["y"], e["psi"], e["v"], self.yref(o.t + 0.4), Tp=0.8)
        a = self.acc if self.acc is not None else speed_cmd(e["v"], self.v_ref, a_lo=-4.0)
        return a, steer


print("laws and causal base ok")

### 1.3 Highway scenario toolkit
Shared by the real-world scenarios.

* **`crash_path`** — scripted kinematics of a vehicle involved in a crash at t₀: straight driving before, then a
  longitudinal slide to rest plus a lateral shift and yaw along a quintic. Parameters are stored on the function (`.params`) for the hand-off.
* **`law_idm`** — surrounding traffic as an Intelligent Driver Model on the leader state perceived `react` s ago.
  The leader is whatever body (ego included) laterally overlaps the actor's footprint, so a **cut-in is noticed only after the
  ego physically enters the lane**. Optional *rubbernecking* brake after the crash.
* **`hazard_seen` / `OnHazard`** — causal trigger on delivered observations (an actor decelerating harder than 4 m/s² or
  moving into the ego lane). **Every candidate is wrapped**, so all policies share the same detection latency.
* **`const_path` / `lane_change_path` / `crossing_path`** — further scripted actors: constant-velocity (also oncoming, heading π),
  a scripted lane change (cut-out), and an animal/pedestrian crossing that stops in the lane.
* **`hazard_seen`** also fires for an object ahead in the ego lane whose closing speed needs > 3 m/s² of braking (stopped or oncoming).
* **`EscapeAdjacent`** — the complete escape policy: evasive lateral move (cap `A_lat`) with braking limited to what the
  friction circle leaves (keeping a 10 % authority reserve), then lane tracking and IDM following of the delivered leader in the target lane.
  Options: a free target `y` (hard shoulder), `wait_slot` (brake until the target lane offers a slot: brake-steer), `stop` (stop and hold on arrival).
* **Road elevation** — `CrestProfile` (crest vertical curve: `z(x)`, `grade(x)`, terrain line-of-sight test, OpenDRIVE `<elevation>` records)
  and `line_of_sight(...)`, a `Scene.visible` rule that delivers an actor only once the sight line from the driver's eye clears the road.
  The simulator patch passes the simulator to such rules and applies gravity (−g·grade) to the ego when `scene.meta['profile']` is set.
* **`SwerveReturn`** — swerve by `dy` (opposing lane, or a partial in-lane offset), hold the line until the obstacle is behind, then return home.
* **`path_keyframes` / `ego_keyframes`** — replay tables for CARLA (poses relative to the ego at measured t₀, every 0.1 s).
* **`plot_region_map`** — categorical sweep map (each cell is also lettered, so it reads without colour).

In [ ]:
# ---------- scripted crash kinematics ----------
def crash_path(x_c, y0, v_pre, v_post, a_slide, y_rest, psi_rest, T_slide):
    """Pose of a vehicle involved in a crash at t = 0 (t relative to t0).
    t < 0 : straight in its lane at v_pre (x = x_c at the impact instant).
    t >= 0: longitudinal slide from v_post to rest at a_slide; lateral shift y0 -> y_rest and
            yaw 0 -> psi_rest along a quintic over T_slide.  Returns (x, y, psi, vx, vy)."""
    t_stop = v_post / a_slide

    def f(t):
        if t < 0:
            return x_c + v_pre * t, y0, 0.0, v_pre, 0.0
        s = min(t, t_stop)
        x = x_c + v_post * s - 0.5 * a_slide * s * s
        vx = v_post - a_slide * s
        u = min(t / T_slide, 1.0)
        y = y0 + (y_rest - y0) * quintic(u)
        vy = (y_rest - y0) * 30 * u * u * (1 - u) ** 2 / T_slide
        return x, y, psi_rest * quintic(u), vx, vy
    f.params = dict(x_c=x_c, y0=y0, v_pre=v_pre, v_post=v_post, a_slide=a_slide, y_rest=y_rest,
                    psi_rest=psi_rest, T_slide=T_slide, t_stop=t_stop)
    return f


def const_path(x0, y0, v, psi=0.0, t_brake=None, b=0.0):
    """Constant-velocity path along heading psi (psi = pi: oncoming traffic); optionally brakes at b from
    t_brake to rest. (x0, y0) = position at t = 0."""
    c, s = math.cos(psi), math.sin(psi)

    def f(t):
        if t_brake is None or t <= t_brake:
            d, vv = v * t, v
        else:
            tb = min(t - t_brake, v / b)
            d, vv = v * t_brake + v * tb - 0.5 * b * tb * tb, v - b * tb
        return x0 + c * d, y0 + s * d, psi, c * vv, s * vv
    f.params = dict(x0=x0, y0=y0, v=v, psi=psi, t_brake=t_brake, b=b)
    return f


def lane_change_path(x0, y0, v, y1, t_start, T_lc):
    """Constant speed v; lateral quintic y0 -> y1 over T_lc starting at t_start. (x0 at t = 0)."""
    def f(t):
        u = min(max((t - t_start) / T_lc, 0.0), 1.0)
        vy = (y1 - y0) * 30 * u * u * (1 - u) ** 2 / T_lc
        return x0 + v * t, y0 + (y1 - y0) * quintic(u), math.atan2(vy, v), v, vy
    f.params = dict(x0=x0, y0=y0, v=v, y1=y1, t_start=t_start, T_lc=T_lc)
    return f


def merge_accel_path(x0, y0, v0, y1, t_start, T_lc, a=0.0, t_acc=0.0, v_max=None):
    """Scripted merge: lateral quintic y0 -> y1 over T_lc from t_start, while accelerating at `a` from t_acc up to v_max
    (x0 = position at t = 0, speed v0 before t_acc)."""
    v_max = v0 if v_max is None else v_max
    t_sat = t_acc + (v_max - v0) / a if a > 0 else t_acc

    def f(t):
        ta = min(max(t - t_acc, 0.0), max(t_sat - t_acc, 0.0))
        x = x0 + v0 * t + 0.5 * a * ta * ta + a * ta * max(t - t_sat, 0.0) if a > 0 else x0 + v0 * t
        vx = v0 + a * ta
        u = min(max((t - t_start) / T_lc, 0.0), 1.0)
        vy = (y1 - y0) * 30 * u * u * (1 - u) ** 2 / T_lc
        return x, y0 + (y1 - y0) * quintic(u), math.atan2(vy, vx), vx, vy
    f.params = dict(x0=x0, y0=y0, v0=v0, y1=y1, t_start=t_start, T_lc=T_lc, a=a, t_acc=t_acc, v_max=v_max)
    return f


def crossing_path(x, y0, y1, t_start, T_cross, psi=-math.pi / 2):
    """Pedestrian/animal: stands at y0, then moves laterally to y1 along a quintic over T_cross and stops there.
    Heading psi is the body axis (−pi/2: facing toward −y, so the body length lies across the lane)."""
    def f(t):
        u = min(max((t - t_start) / T_cross, 0.0), 1.0)
        return x, y0 + (y1 - y0) * quintic(u), psi, 0.0, (y1 - y0) * 30 * u * u * (1 - u) ** 2 / T_cross
    f.params = dict(x=x, y0=y0, y1=y1, t_start=t_start, T_cross=T_cross, psi=psi)
    return f


# ---------- surrounding traffic: IDM with perception delay and an optional rubbernecking brake ----------
def leader_of(me, sim):
    """Nearest body ahead whose footprint laterally overlaps mine (ego included: a cut-in is
    noticed only once the ego physically enters the lane). Returns (bumper gap, leader) or (None, None)."""
    best = (None, None)
    for ob in [sim.ego] + list(sim.objs.values()):
        if ob is me:
            continue
        dx = ob.position[0] - me.position[0]
        if dx > 0 and abs(ob.position[1] - me.position[1]) < (ob.WIDTH + me.WIDTH) / 2 + 0.3:
            s = dx - (ob.LENGTH + me.LENGTH) / 2
            if best[0] is None or s < best[0]:
                best = (s, ob)
    return best


def idm_accel(v, v0, s, dv, T=1.5, s0=2.0, a=1.5, b=2.0):
    s_star = s0 + max(0.0, v * T + v * dv / (2 * math.sqrt(a * b)))
    return a * (1 - (v / max(v0, 0.1)) ** 4 - (s_star / max(s, 0.1)) ** 2)


def law_idm(v0, T=1.5, s0=2.0, a=1.5, b=2.0, react=0.0, b_max=6.0, rubberneck=None):
    """IDM car-following on the leader state perceived `react` s ago.
    rubberneck=(t_start, decel, v_floor): additionally brake at `decel` from t_start (rel. t0) down to v_floor."""
    def f(t, me, sim):
        buf = sim.mem.setdefault(("idm", id(me)), [])
        s, ld = leader_of(me, sim)
        buf.append((1e3, 0.0) if ld is None else (s, me.speed - ld.velocity[0]))
        s_p, dv_p = buf[max(0, len(buf) - 1 - int(round(react / DT)))]
        acc = idm_accel(me.speed, v0, s_p, dv_p, T, s0, a, b)
        if rubberneck is not None and t >= rubberneck[0] and me.speed > rubberneck[2]:
            acc = min(acc, -rubberneck[1])
        return float(np.clip(acc, -b_max, a))
    return f


# ---------- ego: hazard detection and candidate policies ----------
def extent(a):
    """Axis-aligned (x, y) extent of a delivered actor's (possibly rotated) footprint."""
    c, s = abs(math.cos(a["psi"])), abs(math.sin(a["psi"]))
    return a["length"] * c + a["width"] * s, a["length"] * s + a["width"] * c


def hazard_seen(o, lane_y, a_thr=-4.0, vy_thr=0.5, a_req=3.0):
    """Causal detector on *delivered* observations: some actor decelerates harder than a_thr, moves
    laterally toward the ego lane faster than vy_thr while within 1.5 lanes of it, or sits ahead in the
    ego lane with a closing speed that needs more than a_req m/s^2 of braking (stopped or oncoming objects)."""
    if len(o.history) < 2:
        return False
    e = o.ego
    (t1, s1), (t2, s2) = o.history[-2], o.history[-1]
    for b in s2.values():
        ex, ey = extent(b)
        s = b["x"] - ex / 2 - (e["x"] + HALF_L)
        closing = e["v"] - b["v"] * math.cos(b["psi"])
        if abs(b["y"] - lane_y) < (ey + EGO_W) / 2 and s > 0 and closing > 0 and closing ** 2 / (2 * s) > a_req:
            return True
    for n, b in s2.items():
        a = s1.get(n)
        if a is None:
            continue
        if (b["v"] - a["v"]) / (t2 - t1) < a_thr:
            return True
        vy = (b["y"] - a["y"]) / (t2 - t1)
        if vy * np.sign(lane_y - b["y"]) > vy_thr and abs(b["y"] - lane_y) < 1.5 * LANE_W:
            return True
    return False


class OnHazard(Policy):
    """Hold speed and lane until `hazard_seen` fires on delivered observations, then run `inner` with its
    clock starting at the detection. Every candidate is wrapped, so all share the same detection latency."""
    def __init__(self, inner, name=None):
        self.inner, self.name = inner, name or inner.name
        self.desc = {"trigger": "hazard_seen(a<-4 m/s^2 or lateral intrusion >0.5 m/s)", **inner.descriptor()}
        self.reset()

    def reset(self):
        self.t_det, self.lane_y = None, None
        self.inner.reset()
        self.hold = cruise()
        self.hold.reset()

    def act(self, o):
        if self.lane_y is None:
            self.lane_y = round(o.ego["y"] / LANE_W) * LANE_W
        if self.t_det is None and hazard_seen(o, self.lane_y):
            self.t_det = o.t
        if self.t_det is None:
            return self.hold.act(o)
        return self.inner.act(Obs(o.t - self.t_det, o.ego, o.actors, o.history, o.lanes))


def evade(lane=1, A=6.0, t=0.0, name=None, acc=None):
    return Plan(name or f"bypass(A_lat={A})", [{"t": 0}, {"t": t, "lane": lane, "A_lat": A, "acc": acc}], A_lat=A, t_start=t)


class EscapeAdjacent(Policy):
    """Complete escape policy: evasive lateral move to `lane` (or to lateral position `y`, e.g. a hard shoulder) with
    lateral-accel cap A_lat while braking at most b_evasive, then lane tracking and IDM car-following on the *delivered*
    leader in the target corridor (forward-moving actors only). Braking is always capped by what the declared friction
    circle (mu_g, 10 % reserve) leaves after the lateral demand.
    wait_slot: brake at b_max in the current lane until the delivered target-lane traffic leaves a slot of s_slot m
               (plus the closing during a ~1.5 s lateral move), then commit (brake-steer into a gap behind a vehicle).
    stop:      once settled in the target corridor, brake to a standstill and hold (e.g. on the shoulder).
    v_set:     IDM desired speed (default: the speed at detection), e.g. to match faster traffic in the target lane."""
    def __init__(self, lane=0, A_lat=6.0, b_evasive=3.0, T=1.0, s0=3.0, a=2.0, b=3.0, mu_g=7.0, y=None,
                 wait_slot=False, s_slot=1.5, stop=False, v_set=None, name=None):
        self.lane, self.A_lat, self.b_ev, self.mu = lane, A_lat, b_evasive, 0.9 * mu_g
        self.y_t, self.wait_slot, self.s_slot, self.stop = (lane * LANE_W if y is None else y), wait_slot, s_slot, stop
        self.v_set = v_set
        self.idm = dict(T=T, s0=s0, a=a, b=b)
        self.name = name or (f"escape_lane{lane}(A_lat={A_lat})" if y is None else f"escape_to_y={y}(A_lat={A_lat})")
        self.desc = dict(lane=lane, y_target=self.y_t, A_lat=A_lat, b_evasive=b_evasive, wait_slot=wait_slot,
                         s_slot=s_slot, stop=stop, v_set=v_set, **self.idm)
        self.reset()

    def reset(self):
        self.v0, self.settled, self.committed, self.y_home = None, False, False, None

    def slot_ok(self, o):
        e = o.ego
        for a in o.actors.values():
            if abs(a["y"] - self.y_t) > 2.3 or math.cos(a["psi"]) <= 0:
                continue
            ex = extent(a)[0]
            ahead = (a["x"] - ex / 2) - (e["x"] + HALF_L) - max(0.0, e["v"] - a["v"]) * 1.5   # closing during the move
            behind = (e["x"] - HALF_L) - (a["x"] + ex / 2) - max(0.0, a["v"] - e["v"]) * 1.0
            if max(ahead, behind) < self.s_slot:
                return False
        return True

    def act(self, o):
        e = o.ego
        y_t = self.y_t
        if self.v0 is None:
            self.v0, self.y_home = (e["v"] if self.v_set is None else self.v_set), round(e["y"] / LANE_W) * LANE_W
        if self.wait_slot and not self.committed:
            self.committed = self.slot_ok(o)
            if not self.committed:
                return -6.0, track_y(e["x"], e["y"], e["psi"], e["v"], self.y_home)
        if abs(y_t - e["y"]) > 1.5:
            steer = evade_y(e["y"], e["psi"], e["v"], y_t, A_lat=self.A_lat)
        else:
            steer = track_y(e["x"], e["y"], e["psi"], e["v"], y_t)
            self.settled = self.settled or (abs(y_t - e["y"]) < 0.5 and abs(e["psi"]) < 0.03)
        s, dv = 1e3, 0.0
        for a in o.actors.values():
            if abs(a["y"] - y_t) < 2.3 and a["x"] > e["x"] and math.cos(a["psi"]) > 0:
                gap = a["x"] - e["x"] - (extent(a)[0] + EGO_L) / 2
                if gap < s:
                    s, dv = gap, e["v"] - a["v"] * math.cos(a["psi"])
        acc = idm_accel(e["v"], self.v0, s, dv, **self.idm)
        if self.stop and self.settled:
            acc = min(acc, -6.0)
        grip = math.sqrt(max(self.mu ** 2 - lat_accel(e["v"], steer) ** 2, 0.0))   # friction circle left after steering
        b_lim = min(6.0 if self.settled else self.b_ev, grip)
        return float(np.clip(acc, -b_lim, min(self.idm["a"], grip))), steer


class SwerveReturn(Policy):
    """Swerve by `dy` (a full adjacent/opposing lane, or a partial offset inside the lane + shoulder), hold that
    line until every slow obstacle delivered in the home lane is `margin` m behind the ego rear, then return home
    (lateral-accel cap A_ret, default A_lat) and track. Longitudinal: constant `acc` (0 = keep speed) capped by the
    friction circle (10 % reserve)."""
    def __init__(self, dy, A_lat=6.0, margin=3.0, acc=0.0, mu_g=7.0, A_ret=None, name=None):
        self.dy, self.A_lat, self.margin, self.acc, self.mu = dy, A_lat, margin, acc, 0.9 * mu_g
        self.A_ret = A_lat if A_ret is None else A_ret
        self.name = name or f"swerve_return(dy={dy:+.1f})"
        self.desc = dict(dy=dy, A_lat=A_lat, A_ret=self.A_ret, margin=margin, acc=acc)
        self.reset()

    def reset(self):
        self.home, self.out, self.back = None, False, False

    def act(self, o):
        e = o.ego
        if self.home is None:
            self.home = round(e["y"] / LANE_W) * LANE_W
        y_out = self.home + self.dy
        self.out = self.out or abs(e["y"] - y_out) < 0.5
        if self.out and not self.back:
            obst = [a for a in o.actors.values()
                    if abs(a["y"] - self.home) < 2.3 and a["v"] * math.cos(a["psi"]) < 0.5 * e["v"]]
            self.back = all(e["x"] - HALF_L - (a["x"] + extent(a)[0] / 2) > self.margin for a in obst)
        y_t = self.home if self.back else y_out
        if abs(y_t - e["y"]) > 1.5:
            steer = evade_y(e["y"], e["psi"], e["v"], y_t, A_lat=self.A_ret if self.back else self.A_lat)
        else:
            steer = track_y(e["x"], e["y"], e["psi"], e["v"], y_t)
        b_lim = math.sqrt(max(self.mu ** 2 - lat_accel(e["v"], steer) ** 2, 0.0))
        return float(np.clip(self.acc, -b_lim, 2.0)), steer


# ---------- hand-off helpers ----------
def path_keyframes(scene, tr, every=0.1):
    """Scripted actor poses relative to the ego at measured t0, every `every` s (a CARLA replay table)."""
    e = tr.measured_t0["ego"]
    out = {}
    for a in scene.actors:
        if a.kind == "scripted":
            out[a.name] = []
            for t in np.round(np.arange(-scene.prefix_T, scene.T + 1e-9, every), 3):
                x, y, psi, vx, vy = a.law(t)
                out[a.name].append(dict(t=float(t), dx=round(x - e["x"], 3), dy=round(y - e["y"], 3),
                                        yaw=round(psi, 4), vx=round(vx, 3), vy=round(vy, 3)))
    return out


def ego_keyframes(tr, every=0.1):
    """Executed ego trajectory from t0 on, relative to the ego at measured t0 (reference for CARLA replay checks)."""
    R, e = tr.rec, tr.measured_t0["ego"]
    return [dict(t=round(R["t"][j] - tr.t0, 3), dx=round(R["x"][j] - e["x"], 3), dy=round(R["y"][j] - e["y"], 3),
                 yaw=round(R["psi"][j], 4), v=round(R["v"][j], 3), a=round(R["a"][j], 3), steer=round(R["steer"][j], 4))
            for j in range(0, len(R["t"]), max(1, int(round(every / DT)))) if R["phase"][j] == "policy"]


def plot_region_map(xs, ys, grid, xlabel, ylabel, title, labels, colors=("#d9d9d9", "tab:blue", "tab:orange"),
                    marks=("\u2013", "B", "E")):
    """Categorical map: grid[i, j] = index into `labels` for (ys[i], xs[j]); a None label is left out of the legend.
    Every cell also carries a letter, so the regions stay readable without colour."""
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    ax.pcolormesh(xs, ys, grid, cmap=ListedColormap(colors[:len(labels)]), vmin=-0.5, vmax=len(labels) - 0.5,
                  shading="nearest", edgecolors="w", linewidth=2)
    for i, y in enumerate(ys):
        for j, x in enumerate(xs):
            ax.text(x, y, marks[grid[i, j]], ha="center", va="center", fontsize=8,
                    color="k" if grid[i, j] == 0 else "w")
    ax.legend(handles=[Patch(color=c, label=f"{m}  {l}") for c, m, l in zip(colors, marks, labels) if l is not None],
              fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()


print("highway scenario toolkit ok")


# ---------- road elevation: crest vertical curve and terrain line of sight ----------
class CrestProfile:
    """Crest vertical curve: grade g1 before, g2 after, a symmetric parabola of length L centred on x_pvi
    (z = 0 at the PVI on the tangents). z(x) [m], grade(x) [-], and a terrain line-of-sight test."""
    def __init__(self, x_pvi, g1=0.05, g2=-0.05, L=25.0):
        self.x_pvi, self.g1, self.g2, self.L = float(x_pvi), float(g1), float(g2), float(L)
        self.bvc, self.evc = self.x_pvi - self.L / 2, self.x_pvi + self.L / 2

    def z(self, x):
        x = np.asarray(x, dtype=float)
        u = np.clip(x - self.bvc, 0.0, self.L)
        z_curve = -self.g1 * self.L / 2 + self.g1 * u + (self.g2 - self.g1) / (2 * self.L) * u * u
        return np.where(x < self.bvc, self.g1 * (x - self.x_pvi),
                        np.where(x > self.evc, self.g2 * (x - self.x_pvi), z_curve))

    def grade(self, x):
        u = np.clip(np.asarray(x, dtype=float) - self.bvc, 0.0, self.L)
        return self.g1 + (self.g2 - self.g1) * u / self.L

    def sees(self, x_eye, x_t, h_eye=1.08, h_t=0.6, n=80):
        """True when the straight line from the eye (h_eye above the road at x_eye) to the target point (h_t above
        the road at x_t) stays above the road surface in between."""
        if x_t <= x_eye:
            return True
        xs = np.linspace(x_eye, x_t, n)[1:-1]
        z_e, z_t = float(self.z(x_eye)) + h_eye, float(self.z(x_t)) + h_t
        return bool(np.all(z_e + (z_t - z_e) * (xs - x_eye) / (x_t - x_eye) > self.z(xs)))

    def sight_distance(self, x_t, h_eye=1.08, h_t=0.6):
        """Distance eye -> target at the first eye position (approaching from behind) that sees the target."""
        lo, hi = x_t - 400.0, x_t                                  # lo: hidden, hi: visible
        assert not self.sees(lo, x_t, h_eye, h_t), "target visible from 400 m: no crest occlusion"
        for _ in range(60):
            mid = 0.5 * (lo + hi)
            lo, hi = (lo, mid) if self.sees(mid, x_t, h_eye, h_t) else (mid, hi)
        return x_t - hi

    def opendrive(self, x0, x1):
        """OpenDRIVE <elevation> records (s, a, b, c, d), s measured from x0, covering [x0, x1]."""
        recs = []
        for xa, xb, b, c in [(x0, self.bvc, self.g1, 0.0), (self.bvc, self.evc, self.g1, (self.g2 - self.g1) / (2 * self.L)),
                             (self.evc, x1, self.g2, 0.0)]:
            xa, xb = max(xa, x0), min(xb, x1)
            if xb > xa:
                recs.append(dict(s=round(xa - x0, 4), a=round(float(self.z(xa)), 5), b=b, c=round(c, 7), d=0.0))
        return recs


def line_of_sight(profile, h_eye=1.08, h_target=0.6, eye_dx=-0.3, gated=lambda n: True):
    """Scene.visible rule: an actor is delivered to the ego only when the sight line from the driver's eye
    (eye_dx m ahead of the ego centre, h_eye above the road) to the actor's rear face at h_target clears the
    road surface. Terrain occlusion only (vehicles do not occlude each other)."""
    def vis(t, n, sim):
        if not gated(n):
            return True
        o = sim.objs[n]
        return profile.sees(sim.ego.position[0] + eye_dx, o.position[0] - o.LENGTH / 2, h_eye, h_target)
    vis.needs_sim = True
    return vis


---
# 2. Scenarios

Each scenario follows the same pattern as the case cards in the companion notebook:
**`build_*()`** → **policy library** → **profile over declared worlds** → **sweeps** → **`check(...)`** → **`handoff(...)`**.

## RC03 — Braking fails but lateral escape remains  (`R_REF`)
**Card (recap).** The longitudinal stopping requirement is not met, but a supported adjacent corridor permits evasive motion.
Witness: a calibrated steer or brake–steer continuation followed by stable tracking (a *complete* policy).
*Accept only when* braking is refuted and the bypass is supported across all declared worlds; check footprint, steering rate,
lateral acceleration and **downstream traffic**. *Paired contrast:* close the escape corridor without changing the front
obstacle; do not prelabel the result `U_MODEL`. **Required profile contrast:** `r(brake_hold) < ε` and `r(executable_bypass) ≥ ε`.

### RC03-S1 — Highway secondary collision: spun car slides into the ego lane

**Story.** The ego drives at 27 m/s (≈ 97 km/h) in the **middle lane** of a straight 3-lane highway. Ahead in the **right lane**,
a car doing 26 m/s rear-ends a slower car (20 m/s). The struck car (`victim`) spins (−0.6 rad) and slides into the ego lane, coming
to rest across it; the `striker` stops in the right lane. The ego cannot brake to a stop in time. The **left lane** carries traffic,
but there is a gap the ego can move into.

| Element | Prototype (pilot values) |
|---|---|
| Road | 3 lanes × 4 m, lane 0 = left, 1 = ego (middle), 2 = right; no shoulder (strict road boundary) |
| t₀ | impact instant; 1 s cruise before it. `d0` = ego front → victim rear at t₀ = **25 m** |
| `victim` (scripted) | 20 → 22 m/s at impact, slides to rest at 8 m/s², y: lane 2 → 0.4 m right of the ego-lane centre over 1.5 s, yaw −0.6 rad |
| `striker` (scripted) | 26 → 18 m/s at impact, stops at 8 m/s² in lane 2 (small yaw) |
| `left_lead`, `left_follow` | IDM, 24 m/s, 0.8 s perception delay; 40 m bumper gap around the ego. The lead may rubberneck-brake from t₀+1 s |
| Declared worlds | lead brake ∈ {0, 3, 5} m/s² × follower headway T ∈ {1.0, 1.5} s (6 worlds) |
| Ego envelope | default `Envelope` (b_max 6, μg 7 m/s², steering rate 1 rad/s, 0.1 s command delay) |

**Library** (all wrapped in `OnHazard`): brake hold · bypass to the right (blocked by the striker) · comfort lane change left
(T_lc = 3 s) · open-loop evasive bypass left that ignores the left-lane traffic · **`EscapeAdjacent`** with A_lat = 6 and 5 m/s².

The pilot speed is a deliberate *highway variant* above the card's 8–14 m/s range; the region map in the sweep shows where the
card's contrast holds as a function of speed.

In [ ]:
def build_rc03_s1(d0=25.0, v=27.0, v_left=24.0, left_gap=40.0, gap_center=-2.5, lead_brake=0.0,
                  follow_T=1.5, follow_react=0.8, left_spacing=None, obs_delay=0.0, T=8.0, prefix_T=1.0):
    """3-lane highway, ego in the middle lane (lane 1) at v. At t0 a striker rear-ends a slower car in the
    right lane (lane 2); the struck car spins and slides into the ego lane, the striker stops in lane 2.
    The left lane (lane 0) carries IDM traffic with a bumper gap `left_gap` centred `gap_center` m from the ego
    centre at t0; the lead may rubberneck-brake after the crash. `left_spacing` (paired contrast) replaces the
    gap by a dense platoon with that centre-to-centre spacing."""
    x_v = HALF_L + d0 + 2.5                      # victim centre at the impact instant (ego centre at x=0 at t0)
    victim = crash_path(x_v, 2 * LANE_W, v_pre=20.0, v_post=22.0, a_slide=8.0, y_rest=1.1 * LANE_W,
                        psi_rest=-0.6, T_slide=1.5)
    striker = crash_path(x_v - 5.0, 2 * LANE_W, v_pre=26.0, v_post=18.0, a_slide=8.0, y_rest=2 * LANE_W + 0.4,
                         psi_rest=0.15, T_slide=1.0)
    lead_dx = gap_center + left_gap / 2 + 2.5
    follow_dx = gap_center - left_gap / 2 - 2.5
    acts = [Actor("victim", x=0, lane=2, kind="scripted", law=victim,
                  note="struck car: 20 m/s in lane 2, pushed to 22 m/s, slides at 8 m/s^2, spins -0.6 rad into lane 1 over 1.5 s"),
            Actor("striker", x=0, lane=2, kind="scripted", law=striker,
                  note="striking car: 26 m/s in lane 2, 18 m/s after impact, stops at 8 m/s^2 in lane 2"),
            Actor("left_lead", x=lead_dx - v_left * prefix_T, lane=0, speed=v_left, kind="vehicle",
                  law=law_idm(v_left, T=1.5, react=0.8, rubberneck=(1.0, lead_brake, 12.0) if lead_brake > 0 else None),
                  note=f"IDM v0={v_left}; rubberneck brake {lead_brake} m/s^2 from t0+1.0 s down to 12 m/s"),
            Actor("left_follow", x=follow_dx - v_left * prefix_T, lane=0, speed=v_left, kind="vehicle",
                  law=law_idm(v_left, T=follow_T, react=follow_react),
                  note=f"IDM v0={v_left}, T={follow_T} s, perception delay {follow_react} s (reacts to ego cut-in)")]
    if left_spacing is not None:
        acts = acts[:2] + [Actor(f"left_{i}", x=dx - v_left * prefix_T, lane=0, speed=v_left, kind="vehicle",
                                 law=law_idm(v_left, T=1.5, react=0.8),
                                 note=f"dense platoon, {left_spacing} m centre spacing, IDM v0={v_left}")
                           for i, dx in enumerate(np.arange(-3, 4) * left_spacing)]
    return Scene("RC03-S1", lanes=3, ego_lane=1, ego_speed=v, ego_x=-v * prefix_T, actors=acts, T=T, T_term=2.0,
                 env=Envelope(obs_delay=obs_delay), prefix_policy=cruise(), prefix_T=prefix_T,
                 world=dict(d0=d0, v=v, lead_brake=lead_brake, follow_T=follow_T, left_gap=left_gap, gap_center=gap_center, left_spacing=left_spacing,
                           obs_delay=obs_delay))


RC03S1_WORLDS = [dict(lead_brake=lb, follow_T=ft) for lb in (0.0, 3.0, 5.0) for ft in (1.0, 1.5)]
W_NOM = dict(lead_brake=3.0, follow_T=1.5)
RC03S1_LIB = [OnHazard(brake_hold(6.0)),
              OnHazard(evade(lane=2, A=6.0, name="bypass_right(A_lat=6)")),
              OnHazard(Plan("lane_change_left(T_lc=3)", [{"t": 0, "lane": 0, "T_lc": 3.0}])),
              OnHazard(evade(lane=0, A=6.0, name="bypass_left_open_loop(A_lat=6)")),
              OnHazard(EscapeAdjacent(0, A_lat=6.0)),
              OnHazard(EscapeAdjacent(0, A_lat=5.0))]
WITNESS = "escape_lane0(A_lat=6.0)"


def k03(policy, w=W_NOM):
    return (policy, json.dumps(w, sort_keys=True))


df03s1, tr03s1 = evaluate(build_rc03_s1, RC03S1_LIB, RC03S1_WORLDS, keep=True)
prof03s1 = robust_profile(df03s1)
display(df03s1.pivot_table(index="policy", columns="world", values="r", sort=False).round(2))
display(pd.DataFrame({"robust r": prof03s1.round(2),
                      "binding (worst world)": df03s1.loc[df03s1.groupby("policy", sort=False)["r"].idxmin()]
                      .set_index("policy")["binding"]}))
plot_traces([tr03s1[k03(n)] for n in ["brake_hold(b=6.0)", "bypass_left_open_loop(A_lat=6)", WITNESS]],
            "RC03-S1 nominal world (lead rubberneck-brakes 3 m/s^2, follower T=1.5 s): v=27 m/s, d0=25 m",
            xlim=(-45, 140))


#### Sweeps (what the CARLA builder re-tunes)
* **(a) speed × d0 region map** — where braking alone passes (B), where only the escape passes (E), and where neither does (–).
  The RC03 contrast exists in the **E** band; aim the CARLA scene into it after re-measuring the stopping distance.
* **(b) d0 at the pilot speed** — the single-variable sweep requested by the card (escape curve is the minimum over the declared worlds).
* **(c) left-lane gap length × position** — which left-lane configurations admit the escape. When the gap lies well behind (or ahead of)
  the ego, the ego simply merges ahead of the "lead" (or behind the "follower"); the gap only matters when it brackets the escape path.
* **(d) perception latency** — `obs_delay` on every delivered observation, robust over the declared worlds.

In [ ]:
BRK, WIT = OnHazard(brake_hold(6.0)), OnHazard(EscapeAdjacent(0, A_lat=6.0))

# (a) ego speed x distance to the struck car at impact: which policy keeps r >= eps (nominal world)
v_grid, d_grid = np.arange(20.0, 33.0, 2.5), np.arange(5.0, 61.0, 5.0)
region = np.zeros((len(v_grid), len(d_grid)), dtype=int)
for i, v_ in enumerate(v_grid):
    for j, d_ in enumerate(d_grid):
        if simulate(build_rc03_s1(d0=d_, v=v_, **W_NOM), BRK).passed:
            region[i, j] = 1
        elif simulate(build_rc03_s1(d0=d_, v=v_, **W_NOM), WIT).passed:
            region[i, j] = 2
plot_region_map(d_grid, v_grid, region, "d0: ego front to struck-car rear at impact [m]", "ego speed [m/s]",
                "RC03-S1: where braking fails but the left escape passes (nominal world)",
                ["no witness in library", "brake_hold passes", "only the escape passes"])

# (b) single-variable sweep at the pilot speed (CARLA: re-measure and re-run this one first)
d_line = np.arange(5.0, 61.0, 2.5)
r_brk = [simulate(build_rc03_s1(d0=d_, **W_NOM), BRK).r for d_ in d_line]
r_wit = [min(simulate(build_rc03_s1(d0=d_, **w), WIT).r for w in RC03S1_WORLDS) for d_ in d_line]
plot_sweep(d_line, {"brake_hold(b=6)": r_brk, f"{WITNESS} (min over worlds)": r_wit},
           "d0 [m]", "RC03-S1 d0 sweep at v=27 m/s")
d0_brake = float(d_line[np.argmax(np.array(r_brk) >= EPS)])
d0_window = [float(d) for d, b, w in zip(d_line, r_brk, r_wit) if b < EPS and w >= EPS]
print(f"brake_hold first passes at d0 = {d0_brake} m; brake-fails/escape-passes window: {d0_window[0]}..{d0_window[-1]} m")

# (c) left-lane gap length x gap position relative to the ego (nominal world)
gap_grid, ctr_grid = np.arange(6.0, 43.0, 4.0), np.arange(-20.0, 21.0, 5.0)
gmap = np.array([[2 if simulate(build_rc03_s1(left_gap=g_, gap_center=c_, **W_NOM), WIT).passed else 0
                  for g_ in gap_grid] for c_ in ctr_grid])
plot_region_map(gap_grid, ctr_grid, gmap, "left-lane bumper gap at t0 [m]", "gap centre - ego centre at t0 [m]",
                f"RC03-S1: left-lane gaps that admit {WITNESS}", ["escape fails", None, "escape passes"])
min_gap = {float(c_): (float(gap_grid[np.argmax(row == 2)]) if (row == 2).any() else None) for c_, row in zip(ctr_grid, gmap)}
print("smallest passing gap per gap position:", min_gap)

# (d) perception latency (delivered-observation delay), robust over the declared worlds
lat_grid = np.round(np.arange(0.0, 1.01, 0.1), 2)
r_lat = [min(simulate(build_rc03_s1(obs_delay=od, **w), WIT).r for w in RC03S1_WORLDS) for od in lat_grid]
plot_sweep(lat_grid, {f"{WITNESS} (min over worlds)": r_lat}, "perception latency obs_delay [s]",
           "RC03-S1: detection latency budget of the escape")
lat_max = float(max(od for od, r in zip(lat_grid, r_lat) if r >= EPS))
print(f"escape stays a robust witness up to {lat_max} s perception latency")


#### Checks, paired contrast and hand-off
* the card's contrast (`brake_hold` refuted, escape supported in **every** declared world) plus the scenario-specific refutations;
* **paired contrast:** the left lane becomes a dense platoon (18 m centre spacing, ≈ 0.5 s headway), the crash is unchanged →
  labelled from the finite library only (`U_LIBRARY`, **not** `U_MODEL`);
* a **stress world outside the declared set** (lead brakes 5 m/s², follower tailgates at T = 0.8 s with 1.5 s reaction) is reported
  separately: the label is scoped to the declared worlds.
* The hand-off record adds `scripted_trajectories` (victim/striker replay table) and `witness_trace` (executed ego trajectory).

In [ ]:
rmax = df03s1.groupby("policy", sort=False)["r"].max()   # best world per policy
check("RC03-S1", "r(brake_hold) < eps in every declared world", rmax["brake_hold(b=6.0)"] < EPS)
check("RC03-S1", "r(executable escape into the left gap) >= eps in every declared world", prof03s1[WITNESS] >= EPS)
check("RC03-S1", "right-lane bypass refuted (striker stops in lane 2)", rmax["bypass_right(A_lat=6)"] < EPS)
check("RC03-S1", "comfort lane change (T_lc=3 s) refuted: the escape needs a calibrated evasive maneuver",
      rmax["lane_change_left(T_lc=3)"] < EPS)
check("RC03-S1", "open-loop left bypass refuted: downstream traffic must be managed after the escape",
      prof03s1["bypass_left_open_loop(A_lat=6)"] < EPS)
tw = tr03s1[k03(WITNESS)]
pol = tw.rec["phase"] == "policy"
check("RC03-S1", "witness executes without actuator saturation (steering rate / friction circle)",
      not tw.rec["sat"][pol].any() and tw.comp["dyn"] >= 0)
check("RC03-S1", "a d0 window exists where braking fails but the escape passes (v=27 m/s)", len(d0_window) > 0)

df03s1c = evaluate(build_rc03_s1, RC03S1_LIB, [dict(left_spacing=18.0, **w) for w in RC03S1_WORLDS])
lab03s1c = label_from_profile(robust_profile(df03s1c), library_complete=True)
print("paired contrast (dense left platoon, 18 m spacing, same crash) label from this finite library:", lab03s1c,
      "(NOT U_MODEL without a proof)")
check("RC03-S1", "closing the left gap (dense platoon) removes the witness -> U_LIBRARY, not U_MODEL",
      lab03s1c == "U_LIBRARY")

# outside the declared world set: hard-braking lead AND a tailgating, slow-reacting follower
STRESS = dict(lead_brake=5.0, follow_T=0.8, follow_react=1.5)
tr_stress = simulate(build_rc03_s1(**STRESS), WIT)
print(f"stress world {STRESS}: r({WITNESS}) = {tr_stress.r:+.2f} (binding: {tr_stress.binding}) "
      "-> declaring this world would make the label UNRESOLVED/U_LIBRARY; it is reported, not folded in")

sc_w = tw.scene
handoff("RC03-S1", parent="RC03", label=label_from_profile(prof03s1),
        site="straight 3-lane one-way highway, lane width 4 m; lane 0 = leftmost (y = -4 m from ego lane), "
             "ego in lane 1 (middle), lane 2 = rightmost",
        story="a faster car rear-ends a slower car in the right lane at t0; the struck car spins and slides into "
              "the ego lane; braking alone cannot stop the ego short of it, the left lane has a gap",
        evidence="executed witness in HighwayEnv reference dynamics over 6 declared worlds; not CARLA-validated",
        t0_rule="t0 = impact instant (striker front meets victim rear); the ego cruises before t0 and every policy "
                "activates on hazard_seen() over delivered observations (shared detection latency)",
        ego={"v": 27.0, "lane": 1, "envelope": asdict(sc_w.env)},
        actors=actors_rel(sc_w, tw),
        scripted_trajectories=path_keyframes(sc_w, tw),
        crash_params={a.name: a.law.params for a in sc_w.actors if a.kind == "scripted"},
        traffic_laws={a.name: a.note for a in sc_w.actors if a.kind == "vehicle"},
        worlds=RC03S1_WORLDS, stress_world={**STRESS, "r_witness": tr_stress.r},
        witness=WITNESS, witness_descriptor=dict(RC03S1_LIB[4].descriptor()), witness_trace=ego_keyframes(tw),
        refuted=[p for p in prof03s1.index if prof03s1[p] < EPS],
        sweep={"var": ["d0", "v", "left_gap", "gap_center", "obs_delay"], "pilot": {"d0": 25.0, "v": 27.0, "left_gap": 40.0},
               "d0_brake_passes_from": d0_brake, "d0_window_escape_only": [d0_window[0], d0_window[-1]],
               "region_map": {"v": v_grid.tolist(), "d0": d_grid.tolist(), "code": region.tolist(),
                              "codes": {0: "none", 1: "brake passes", 2: "escape only"}},
               "smallest_passing_gap_by_center": min_gap, "max_perception_latency_s": lat_max},
        checks="footprint (incl. spun victim OBB), steering rate, friction circle, left-lane lead and follower",
        paired_contrast={"change": "left lane becomes a dense platoon (18 m centre spacing), crash unchanged",
                         "label": lab03s1c},
        carla_notes=["Town04/Town06 straight segment with >= 3 same-direction lanes; ego in the middle lane",
                     "synchronous mode, fixed_delta_seconds = 0.05",
                     "victim/striker: follow scripted_trajectories with set_simulate_physics(False) + set_transform each "
                     "tick and set_target_velocity(vx, vy) so other agents/sensors see the slide velocity",
                     "left-lane traffic: client-side IDM with the listed parameters (not Traffic Manager defaults)",
                     "re-measure b_max and the 27 m/s lateral maneuver of the CARLA ego blueprint, then re-run the d0 sweep",
                     "pilot speed is a highway variant (27 m/s) above the card's 8-14 m/s pilot range"])


### RC03-S2 … S8 — shared runner
The remaining RC03 scenarios use `rc03_scenario(...)`, which runs exactly the evidence pattern of RC03-S1:
profile `r[policy, world]` over the declared worlds → traces in the nominal world (brake, a scenario-specific counterexample, the
witness) → a single-variable sweep (escape curve = min over the declared worlds) → the card's checks plus the scenario's own
refutations → the paired contrast → the CARLA hand-off record.

In [ ]:
BRAKE = "brake_hold(b=6.0)"


def rc03_scenario(sid, build, lib, worlds, nominal, witness, refute, contrast, sweep, xlim, title, record):
    """Run one RC03 scenario end to end with the same evidence pattern as RC03-S1:
    profile over the declared worlds -> traces in the nominal world -> single-variable sweep -> card checks ->
    paired contrast -> CARLA hand-off.
    refute:   {policy name: reason} that must be refuted (robust r < eps) - the scenario's own counterexamples.
    contrast: (description, build kwargs that close the escape corridor, everything else unchanged).
    sweep:    dict(var, values, xlabel) swept in the nominal world (escape curve: min over the declared worlds).
    record:   scenario-specific hand-off fields (story, site, t0_rule, carla_notes, ...)."""
    df, trs = evaluate(build, lib, worlds, keep=True)
    prof = robust_profile(df)
    best = df.groupby("policy", sort=False)["r"].max()
    display(df.pivot_table(index="policy", columns="world", values="r", sort=False).round(2))
    display(pd.DataFrame({"robust r": prof.round(2), "binding (worst world)":
                          df.loc[df.groupby("policy", sort=False)["r"].idxmin()].set_index("policy")["binding"]}))
    key = lambda p: (p, json.dumps(nominal, sort_keys=True))
    plot_traces([trs[key(n)] for n in [BRAKE, *refute][:2] + [witness]], f"{sid} nominal world {nominal}", xlim=xlim)

    var, xs = sweep["var"], np.asarray(sweep["values"], dtype=float)
    lib_by_name = {p.name: p for p in lib}
    r_brk = [simulate(build(**{**nominal, var: x}), lib_by_name[BRAKE]).r for x in xs]
    r_wit = [min(simulate(build(**{**w, var: x}), lib_by_name[witness]).r for w in worlds) for x in xs]
    plot_sweep(xs, {BRAKE: r_brk, f"{witness} (min over worlds)": r_wit}, sweep["xlabel"], f"{sid}: {title}")
    window = [float(x) for x, b, w in zip(xs, r_brk, r_wit) if b < EPS and w >= EPS]
    print(f"{var} values where braking fails but the escape passes: "
          + (f"{window[0]} .. {window[-1]}" if window else "none"))

    check(sid, "r(brake_hold) < eps in every declared world", best[BRAKE] < EPS)
    check(sid, f"r({witness}) >= eps in every declared world", prof[witness] >= EPS)
    for p, why in refute.items():
        check(sid, f"{p} refuted: {why}", prof[p] < EPS)
    tw = trs[key(witness)]
    pol = tw.rec["phase"] == "policy"
    check(sid, "witness executes without actuator saturation (steering rate / friction circle)",
          not tw.rec["sat"][pol].any() and tw.comp["dyn"] >= 0)
    check(sid, f"a {var} window exists where braking fails but the escape passes", len(window) > 0)
    dfc = evaluate(build, lib, [{**w, **contrast[1]} for w in worlds])
    labc = label_from_profile(robust_profile(dfc), library_complete=True)
    print(f"paired contrast ({contrast[0]}) label from this finite library: {labc} (NOT U_MODEL without a proof)")
    check(sid, "closing the escape corridor removes the witness -> U_LIBRARY, not U_MODEL", labc == "U_LIBRARY")

    sc = tw.scene
    handoff(sid, parent="RC03", label=label_from_profile(prof), **record,
            evidence=f"executed witness in HighwayEnv reference dynamics over {len(worlds)} declared worlds; not CARLA-validated",
            ego={"v": sc.ego_speed, "lane": sc.ego_lane, "envelope": asdict(sc.env)},
            road={"lanes": sc.lanes, "lane_width": LANE_W, "drivable_y": list(sc.bounds)},
            actors=actors_rel(sc, tw), scripted_trajectories=path_keyframes(sc, tw),
            scripted_params={a.name: a.law.params for a in sc.actors if a.kind == "scripted"},
            actor_laws={a.name: a.note for a in sc.actors},
            worlds=worlds, nominal_world=nominal, witness=witness, witness_descriptor=lib_by_name[witness].descriptor(),
            witness_trace=ego_keyframes(tw), refuted=[p for p in prof.index if prof[p] < EPS],
            sweep={"var": var, "values": xs.tolist(), "r_brake_nominal": r_brk, "r_witness_min_worlds": r_wit,
                   "window_brake_fails_escape_passes": [window[0], window[-1]] if window else None},
            paired_contrast={"change": contrast[0], "kwargs": contrast[1], "label": labc})
    return df, trs


print("RC03 scenario runner ok")


### RC03-S2 — Cut-out reveal: the lead swerves late around a stalled car

**Story.** The ego follows a car at a 1 s gap (25 m/s) in the middle lane of a 3-lane highway. A car has stalled in that lane.
The lead driver sees it late and **swerves into the left lane** at the last moment (1.2 s before reaching it), squeezing in front of
a car that is alongside the ego. Only now does the ego see the stalled car, about 0.6 s into the lead's swerve. It is too close to stop.
Following the lead to the left would hit the car alongside; the **right lane** is open, with a slower car well ahead.

**Why braking fails:** *occlusion* — the hazard is hidden behind the lead until the lead leaves.
**Escape:** evasive change to the *opposite* side from the one the lead used, then follow the slower right-lane car.
**Declared worlds:** lead swerve duration ∈ {1.0, 1.4} s × right-lane car speed ∈ {20, 24} m/s.
**Paired contrast:** the right lane is coned off for roadworks beside the stalled car.
**Sweep:** how late the lead cuts out (`cut_ttc`).

In [ ]:
def build_rc03_s2(v=25.0, headway=1.0, cut_ttc=1.2, T_cut=1.2, v_right=22.0, right_dx=35.0, right_closed=False, T=8.0):
    """3-lane highway, ego in lane 1 following a lead car at `headway` s. A stalled car stands in lane 1; the lead
    sees it, swerves into lane 0 when it is cut_ttc s away (over T_cut s). The stalled car is hidden behind the lead
    until the lead has moved one body width aside. Lane 0 has a car alongside the ego; lane 2 has a slower car ahead."""
    gap = v * headway
    x_lead = HALF_L + gap + 2.5                               # lead centre at t0 (start of its swerve)
    x_st = x_lead + 2.5 + v * cut_ttc + 2.5                   # stalled car centre
    lead = lane_change_path(x_lead, LANE_W, v, 0.0, 0.0, T_cut)
    t_rev = next(t for t in np.arange(0, T_cut, DT) if LANE_W - lead(t)[1] > 2.0)   # lateral clearance of the line of sight
    acts = [Actor("stalled", x=x_st, lane=1, kind="static", note="stalled car, lane 1, hidden behind the lead"),
            Actor("lead", x=0, lane=1, kind="scripted", law=lead,
                  note=f"lead at {v} m/s swerves to lane 0 over {T_cut} s when {cut_ttc} s from the stalled car"),
            Actor("left_car", x=-1.0 - v * 1.0, lane=0, speed=v, kind="vehicle", law=law_idm(v, T=1.2, react=0.8),
                  note=f"IDM {v} m/s, alongside the ego in lane 0 (makes room for the lead)"),
    ] + ([] if right_closed else [
            Actor("right_car", x=right_dx - v_right * 1.0, lane=2, speed=v_right, kind="vehicle",
                  law=law_idm(v_right, T=1.5, react=0.8), note=f"IDM {v_right} m/s in lane 2, {right_dx} m ahead of the ego centre")])
    if right_closed:
        acts.append(Actor("roadworks", x=x_st - 25.0, lane=2, length=70.0, width=3.0, kind="static",
                          note="paired contrast: lane 2 coned off (roadworks) from 60 m before to 10 m past the stalled car"))
    return Scene("RC03-S2", lanes=3, ego_lane=1, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 visible=lambda t, n: n != "stalled" or t >= t_rev, prefix_policy=cruise(), prefix_T=1.0,
                 world=dict(v=v, headway=headway, cut_ttc=cut_ttc, T_cut=T_cut, v_right=v_right, t_reveal=t_rev),
                 meta=dict(t_reveal=t_rev))


WORLDS_S2 = [dict(T_cut=tc, v_right=vr) for tc in (1.0, 1.4) for vr in (20.0, 24.0)]
LIB_S2 = [OnHazard(brake_hold(6.0)), OnHazard(EscapeAdjacent(2, A_lat=6.0)),
          OnHazard(EscapeAdjacent(0, A_lat=6.0, name="follow_the_lead_left(A_lat=6)")),
          OnHazard(evade(lane=2, A=6.0, name="bypass_right_open_loop(A_lat=6)")),
          OnHazard(Plan("lane_change_right(T_lc=3)", [{"t": 0, "lane": 2, "T_lc": 3.0}]))]
print("stalled car revealed at t0 +", build_rc03_s2().meta["t_reveal"], "s (nominal swerve)")
df_s2, tr_s2 = rc03_scenario(
    "RC03-S2", build_rc03_s2, LIB_S2, WORLDS_S2, nominal=dict(T_cut=1.0, v_right=20.0), witness="escape_lane2(A_lat=6.0)",
    refute={"follow_the_lead_left(A_lat=6)": "the lead's escape lane is occupied by the car alongside",
            "bypass_right_open_loop(A_lat=6)": "ignores the slower right-lane car (downstream traffic)",
            "lane_change_right(T_lc=3)": "comfort lane change is too slow after a late reveal"},
    contrast=("lane 2 coned off for roadworks beside the stalled car; cut-out unchanged", dict(right_closed=True)),
    sweep=dict(var="cut_ttc", values=np.arange(0.6, 2.61, 0.2), xlabel="lead swerves this many s before reaching the stalled car"),
    xlim=(-35, 140), title="how late the lead cuts out",
    record=dict(site="straight 3-lane one-way highway; ego in lane 1 (middle)",
                story="the car ahead swerves left at the last moment and uncovers a stalled car; the left lane is taken by "
                      "the car the lead squeezed in front of, so the ego must escape to the right",
                t0_rule="t0 = start of the lead's swerve; the stalled car is delivered only after the lead has moved 2 m aside "
                        "(scene.meta['t_reveal']); policies trigger on hazard_seen()",
                carla_notes=["occlusion is geometric in CARLA: verify the stalled car is not detected before t_reveal with the "
                             "chosen sensor set, otherwise gate its detections explicitly",
                             "lead: scripted lane change (set_transform or a tracked path) that does not react to the ego"]))


### RC03-S3 — Traffic-jam tail just past an overpass crest: three full lanes, escape to the lane with the shortest queue

**Story.** The ego drives at 25 m/s in the **middle lane of a busy 3-lane motorway** that rises over an **overpass**: a +5 % climb and a
−5 % descent joined by a 40 m crest curve. On the far side the traffic has come to a **complete stop**, with the ego-lane queue
tail 20 m past the crest. The crest hides it: the driver's sight line (eye 1.08 m → tail lights 0.6 m) first clears the road
surface when the ego is **52 m** from the tail, less than the ego's stopping distance (which the downhill grade makes longer). The
**left lane is open beside the ego, but its queue is just as long** (it ends 0–10 m further on), so a lane change left is possible yet
still leaves too little distance to stop. The **right lane carries the fewest cars**, so its queue ends about 50 m further on. Only
*after* it sees the queue does the ego make an evasive lane change right, which buys the distance it needs, and then brakes to a
**stop behind the right-lane queue**.

**Elevation model.** The road has a real vertical profile (`CrestProfile`): queue vehicles are delivered to the ego only when the
terrain line of sight clears the crest (`line_of_sight`), so t0 and the sight distance *follow from the geometry*, and gravity acts
on the ego along the grade. The HighwayEnv clips stay top-down but mark the crest and grey out queue cars still hidden behind it;
the side-view figure below shows the profile and the sight line.
The CARLA record carries the profile as OpenDRIVE `<elevation>` records plus the height of every actor.

**Why braking fails:** *sight distance* — a vertical curve hides a stationary queue.
**Escape:** a lane change into the lane with more room ahead, then a full stop there (`terminal='hold'`). The lateral move
turns *stopping distance* into *available distance*; it is not used to drive past the jam.
**Declared worlds:** right-lane queue tail is a car or a 12 m truck × left-lane queue ends 0, 5 or 10 m beyond the ego-lane tail.
**Paired contrast:** all three queues end level (no extra room in the right lane).
**Braking during the escape:** the escape and its left-lane mirror brake as hard as the friction circle allows during the lateral move
(`b_evasive=6`) and at the full 6 m/s² once in the lane.
**Sweep:** extra room in the right-lane queue (`right_room`).

In [ ]:
EYE_DX, H_EYE, H_TARGET = -0.3, 1.08, 0.6      # driver's eye (from ego centre, above road) / target = tail lights


def crest_overlay(prof, scaling=6.0, centre=(0.35, 0.5), size=(1000, 240)):
    """Annotate HighwayEnv frames (top-down, camera on the ego): shade the crest curve, mark its top, label the grades,
    and grey out static queue vehicles the ego cannot see yet (terrain line of sight, as delivered to the policy)."""
    from PIL import Image, ImageDraw, ImageFont
    font = ImageFont.load_default(size=13)

    def f(tr, frames, every):
        xs, ys = tr.rec["x"][::every], tr.rec["y"][::every]
        if len(xs) != len(frames):
            return frames
        out = []
        for fr, ex, ey in zip(frames, xs, ys):
            im = Image.fromarray(fr[..., :3]).convert("RGBA")
            ov = Image.new("RGBA", im.size, (0, 0, 0, 0))
            d = ImageDraw.Draw(ov)
            px = lambda x: centre[0] * size[0] + (x - ex) * scaling
            py = lambda y: centre[1] * size[1] + (y - ey) * scaling
            top, bot = py(-LANE_W / 2), py((tr.scene.lanes - 0.5) * LANE_W)
            d.rectangle([px(prof.bvc), top, px(prof.evc), bot], fill=(255, 255, 255, 45))
            d.line([px(prof.x_pvi), top - 6, px(prof.x_pvi), bot + 6], fill=(255, 255, 255, 230), width=2)
            d.text((px(prof.x_pvi), top - 10), "CREST", fill=(255, 255, 255, 255), font=font, anchor="ms")
            d.text((px(prof.bvc) - 6, bot + 16), f"uphill {prof.g1:+.0%}", fill=(230, 230, 230, 255), font=font, anchor="rs")
            d.text((px(prof.evc) + 6, bot + 16), f"downhill {prof.g2:+.0%}", fill=(230, 230, 230, 255), font=font, anchor="ls")
            for a in tr.scene.actors:
                if a.kind == "static" and not prof.sees(ex + EYE_DX, a.x - a.length / 2, H_EYE, H_TARGET):
                    d.rectangle([px(a.x - a.length / 2), py(a.y0 - a.width / 2), px(a.x + a.length / 2),
                                 py(a.y0 + a.width / 2)], fill=(150, 150, 150, 255), outline=(60, 60, 60, 255))
            d.text((8, size[1] - 8), "grey = hidden behind the crest", fill=(230, 230, 230, 255), font=font, anchor="ls")
            out.append(np.asarray(Image.alpha_composite(im, ov).convert("RGB")))
        return out
    return f


def build_rc03_s3(v=25.0, g1=0.05, g2=-0.05, L_vc=40.0, tail_past_crest=20.0, right_room=50.0, left_room=5.0,
                  tail="car", T=11.0):
    """3-lane motorway in dense traffic over an overpass crest (grades g1 -> g2 joined by an L_vc m vertical curve);
    ego in the middle lane (lane 1). A traffic jam is stopped on the far side, its ego-lane tail `tail_past_crest` m past
    the crest. Queue vehicles are delivered to the ego only when terrain line of sight (eye H_EYE -> tail lights H_TARGET)
    clears the crest, so t0 = the first tick the ego-lane tail is visible, and the sight distance follows from the
    geometry. The left-lane queue is just as long (tail `left_room` m further on); the right lane carries the fewest
    cars (tail `right_room` m further on). Escape: change right, stop there. Gravity acts on the ego along the grade."""
    sight = CrestProfile(0.0, g1, g2, L_vc).sight_distance(tail_past_crest, H_EYE, H_TARGET) - 1e-3
    x_tail = EYE_DX + sight                                     # rear face of the ego-lane tail (ego centre x=0 at t0)
    prof = CrestProfile(x_tail - tail_past_crest, g1, g2, L_vc)
    L, W = (12.0, 2.55) if tail == "truck" else (4.8, 2.0)
    acts = [Actor("q_mid_tail", x=x_tail + 2.4, lane=1, length=4.8, kind="static", note="stopped car, ego-lane queue tail")]
    acts += [Actor(f"q_mid{i}", x=x_tail + 2.4 + 7.0 * (i + 1), lane=1, length=4.8, kind="static", note="queue, lane 1")
             for i in range(4)]
    acts += [Actor(f"q_left{i}", x=x_tail + left_room + 2.4 + 7.0 * i, lane=0, length=4.8, kind="static",
                   note="queue, lane 0" + (f" (tail {left_room} m beyond the ego-lane tail)" if i == 0 else ""))
             for i in range(5)]
    x_r = x_tail + right_room                                   # rear face of the right-lane tail
    acts += [Actor("q_right_tail", x=x_r + L / 2, lane=2, length=L, width=W, kind="static",
                   note=f"stopped {tail}, right-lane queue tail, {right_room} m beyond the ego-lane tail")]
    acts += [Actor(f"q_right{i}", x=x_r + L + 2.0 + 2.4 + 7.0 * i, lane=2, length=4.8, kind="static", note="queue, lane 2")
             for i in range(2)]
    return Scene("RC03-S3", lanes=3, ego_lane=1, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0, terminal="hold",
                 visible=line_of_sight(prof, H_EYE, H_TARGET, EYE_DX, gated=lambda n: n.startswith("q")),
                 prefix_policy=cruise(v), prefix_T=1.0,
                 world=dict(v=v, g1=g1, g2=g2, L_vc=L_vc, tail_past_crest=tail_past_crest, right_room=right_room,
                            left_room=left_room, tail=tail),
                 meta=dict(profile=prof, sight_distance=sight, gap_at_t0=x_tail - HALF_L, overlay=crest_overlay(prof)))


def plot_crest_view(sc, t_before=1.0):
    """Side view (vertical scale exaggerated): road profile, the queue vehicles, and the driver's sight line to the
    ego-lane queue tail t_before s before t0 (blocked by the crest) and at t0 (just clears it)."""
    prof, x_t = sc.meta["profile"], next(a.x - a.length / 2 for a in sc.actors if a.name == "q_mid_tail")
    xs = np.linspace(-45, x_t + 75, 600)
    fig, ax = plt.subplots(figsize=(12, 3.2))
    ax.fill_between(xs, prof.z(xs) - 3, prof.z(xs), color="0.8")
    ax.plot(xs, prof.z(xs), color="0.3", lw=1.5)
    for a in sc.actors:
        h = 3.5 if a.length > 10 else 1.45
        x0 = a.x - a.length / 2
        ax.add_patch(plt.Rectangle((x0, float(prof.z(a.x))), a.length, h, color="0.55" if a.lane != 1 else "0.3",
                                   alpha=0.35 if a.lane != 1 else 0.9, lw=0))
    z_t = float(prof.z(x_t)) + H_TARGET
    for x_c, col, lab in [(-sc.ego_speed * t_before, "C3", f"t0-{t_before:g} s: crest hides the queue"),
                          (0.0, "C2", f"t0: first sight, {sc.meta['gap_at_t0']:.1f} m bumper gap")]:
        xe = x_c + EYE_DX
        ze = float(prof.z(xe)) + H_EYE
        ax.add_patch(plt.Rectangle((x_c - HALF_L, float(prof.z(x_c))), EGO_L, 1.45, color=col, alpha=0.8, lw=0))
        ax.plot([xe, x_t], [ze, z_t], color=col, ls="--", lw=1.2, label=lab)
    ax.axvline(prof.x_pvi, color="0.6", lw=0.8, ls=":")
    ax.text(prof.x_pvi, float(prof.z(prof.x_pvi)) + 2.2, f"crest  {prof.g1:+.0%} / {prof.g2:+.0%}, L = {prof.L:g} m",
            ha="center", fontsize=9, color="0.3")
    ax.set_xlim(xs[0], xs[-1]); ax.set_ylim(float(prof.z(xs).min()) - 1, float(prof.z(prof.x_pvi)) + 4)
    ax.set_xlabel("x along the road [m] (ego centre at t0 = 0)"); ax.set_ylabel("z [m] (exaggerated)")
    ax.set_title(f"{sc.name}: crest geometry and sight line (eye {H_EYE} m, target = tail lights {H_TARGET} m)")
    ax.legend(loc="lower left", fontsize=9, frameon=False)
    plt.tight_layout(); plt.show()


def crest_elevation_record(sc, x0=-150.0, x1=300.0, h_target=H_TARGET):
    """Elevation block for the CARLA hand-off (x relative to the ego centre at t0)."""
    prof = sc.meta["profile"]
    x_t0 = lambda a: a.law(0.0)[0] if a.kind == "scripted" else a.x + (a.speed or 0.0) * sc.prefix_T
    rec = {"profile": "crest vertical curve: tangents g1 -> g2 joined by a symmetric parabola",
           "g1": prof.g1, "g2": prof.g2, "L_vc": prof.L, "x_pvi": round(prof.x_pvi, 3),
           "x_bvc": round(prof.bvc, 3), "x_evc": round(prof.evc, 3), "z_reference": "z = 0 at the PVI on the tangents",
           "h_eye": H_EYE, "eye_dx_from_ego_centre": EYE_DX, "h_target": h_target}
    if "sight_distance" in sc.meta:
        rec.update(sight_distance_eye_to_target=round(sc.meta["sight_distance"], 2),
                   bumper_gap_at_first_sight=round(sc.meta["gap_at_t0"], 2))
    rec.update(opendrive_elevation={"s_origin_x": x0, "records": prof.opendrive(x0, x1)},
               z_samples=[[float(x), round(float(prof.z(x)), 3)] for x in np.arange(x0, x1 + 1e-9, 10.0)],
               actor_z_t0={a.name: round(float(prof.z(x_t0(a))), 3) for a in sc.actors},
               ego_z_t0=round(float(prof.z(0.0)), 3))
    return rec


SC3_NOM = build_rc03_s3()
print(f"crest {SC3_NOM.meta['profile'].g1:+.0%}/{SC3_NOM.meta['profile'].g2:+.0%}, L_vc={SC3_NOM.meta['profile'].L:g} m: "
      f"queue tail {SC3_NOM.world['tail_past_crest']:g} m past the crest is first seen at "
      f"{SC3_NOM.meta['sight_distance']:.1f} m (eye to tail lights) = {SC3_NOM.meta['gap_at_t0']:.1f} m bumper gap")
plot_crest_view(SC3_NOM)

WORLDS_S3 = [dict(tail=k, left_room=lr) for k in ("car", "truck") for lr in (0.0, 5.0, 10.0)]
LIB_S3 = [OnHazard(brake_hold(6.0)), OnHazard(EscapeAdjacent(2, A_lat=6.0, b_evasive=6.0, stop=True, name="escape_right_and_stop(A_lat=6)")),
          OnHazard(EscapeAdjacent(0, A_lat=6.0, b_evasive=6.0, stop=True, name="escape_left_and_stop(A_lat=6)")),
          OnHazard(Plan("lane_change_right(T_lc=3)", [{"t": 0, "lane": 2, "T_lc": 3.0, "acc": -3.0}]))]
df_s3, tr_s3 = rc03_scenario(
    "RC03-S3", build_rc03_s3, LIB_S3, WORLDS_S3, nominal=dict(tail="car", left_room=5.0),
    witness="escape_right_and_stop(A_lat=6)",
    refute={"escape_left_and_stop(A_lat=6)": "the left-lane queue is as long as the ego lane's: the move is free but too little distance is left to stop",
            "lane_change_right(T_lc=3)": "a comfort-rate lane change brakes too late and too gently"},
    contrast=("the right-lane queue ends level with the ego-lane queue (all three lanes equally full)", dict(right_room=0.0)),
    sweep=dict(var="right_room", values=np.arange(0.0, 70.1, 5.0), xlabel="extra room in the right-lane queue [m]"),
    xlim=(-35, 150), title="extra room in the right lane",
    record=dict(site="straight 3-lane motorway in dense traffic rising over an overpass: +5 % / -5 % grades joined by a "
                     "40 m crest curve; ego in lane 1 (middle lane); no shoulder",
                elevation=crest_elevation_record(SC3_NOM),
                story="a stopped traffic jam lies just past an overpass crest; the crest hides it until the ego is about 52 m from its tail; the "
                      "ego-lane tail is too close to stop for; the left lane is open but queued just as far back, so "
                      "changing left does not help; the right lane carries the fewest cars, so its queue ends further on: the ego changes right and stops there",
                t0_rule="t0 = first tick the terrain line of sight (eye 1.08 m -> tail lights 0.6 m) to the ego-lane queue tail clears "
                        "the crest; policies trigger on hazard_seen()",
                carla_notes=["build the road with the OpenDRIVE <elevation> records in elevation.opendrive_elevation (s = x - s_origin_x); "
                             "a straight 3-lane road, so planView is one line",
                             "spawn the queue vehicles at elevation.actor_z (+ wheel offset); spawn the ego so that it is "
                             "at x = 0 with the speed in `ego` at t0",
                             "camera / lidar detection is occluded by the crest by itself; with ground-truth detection apply "
                             "the same eye/target line-of-sight test",
                             "decelerations in the witness are tyre forces: on the -5 % far side gravity removes ~0.5 m/s^2 "
                             "of net deceleration (modelled here, and present in CARLA physics)",
                             "terminal contract is 'hold': stopped in the right lane behind its queue tail for the last 2 s"]))
_p3 = SC3_NOM.meta["profile"]
_xt3 = next(a.x - a.length / 2 for a in SC3_NOM.actors if a.name == "q_mid_tail")
check("RC03-S3", "the crest hides the ego-lane queue tail 1 s before t0 and reveals it at t0 (terrain line of sight)",
      not _p3.sees(-SC3_NOM.ego_speed + EYE_DX, _xt3, H_EYE, H_TARGET) and _p3.sees(EYE_DX, _xt3, H_EYE, H_TARGET))
_t3 = tr_s3[("escape_right_and_stop(A_lat=6)", json.dumps(dict(tail="car", left_room=5.0), sort_keys=True))]
print(f"witness detects the queue at t0 + {_t3.policy.t_det:.2f} s")
check("RC03-S3", "the ego reacts only after the queue comes into view (detection within 0.2 s of t0)",
      0.0 <= _t3.policy.t_det <= 0.2)


### RC03-S4 — Moose on a rural two-way road: swerve through the opposing lane and back

**Story.** At 22 m/s (≈ 80 km/h) on a rural two-lane road, a **moose steps out of the right verge** about 38 m ahead and freezes in the
ego lane. Braking needs about 43 m. The right verge is not drivable. The **opposing lane** is free for now; an oncoming car is
170–200 m away. The ego swerves into the opposing lane, passes the moose, and **returns to its own lane** before the oncoming car arrives.

**Why braking fails:** a *sudden crossing* by an animal at short range.
**Escape:** a temporary excursion into a lane that belongs to oncoming traffic: a *swerve-and-return*, timed by the oncoming gap.
Staying in the opposing lane fails.
**Declared worlds:** moose freezes at the lane centre or 0.6 m toward the centre line × oncoming car at 170 or 200 m.
**Paired contrast:** the oncoming car is only 110 m away.
**Sweep:** oncoming-car distance (the size of the gap in oncoming traffic).

In [ ]:
def build_rc03_s4(v=22.0, d0=38.0, moose_y=LANE_W, onc_dist=200.0, onc_brake=0.0, onc_v=22.0, T=8.0):
    """Rural two-lane two-way road (lane 1 = ego direction, lane 0 = opposing). A moose steps out from the right
    verge at t0 and freezes in the ego lane. An oncoming car approaches in lane 0; the escape is a swerve through the
    opposing lane and back before it arrives."""
    x_m = HALF_L + d0 + 0.45
    acts = [Actor("moose", x=0, y=LANE_W * 1.5 + 1.5, length=2.6, width=0.9, kind="scripted",
                  law=crossing_path(x_m, LANE_W * 1.5 + 1.5, moose_y, 0.0, 1.0),
                  note=f"moose (2.6 m body across the lane) steps from the right verge at t0, freezes at y={moose_y - LANE_W:+.1f} m from the ego-lane centre after 1.0 s"),
            Actor("oncoming", x=0, lane=0, kind="scripted",
                  law=const_path(HALF_L + onc_dist, 0.0, onc_v, psi=math.pi, t_brake=1.0 if onc_brake else None, b=onc_brake),
                  note=f"oncoming car {onc_v} m/s, {onc_dist} m ahead at t0" + (f", brakes {onc_brake} m/s^2 from t0+1 s" if onc_brake else ""))]
    return Scene("RC03-S4", lanes=2, ego_lane=1, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 prefix_policy=cruise(), prefix_T=1.0,
                 world=dict(v=v, d0=d0, moose_y=moose_y, onc_dist=onc_dist, onc_brake=onc_brake))


WORLDS_S4 = [dict(moose_y=my, onc_dist=od) for my in (LANE_W, LANE_W - 0.6) for od in (170.0, 200.0)]
LIB_S4 = [OnHazard(brake_hold(6.0)), OnHazard(SwerveReturn(-3.6)),
          OnHazard(EscapeAdjacent(0, A_lat=6.0, name="swerve_and_stay_in_opposing_lane(A_lat=6)")),
          OnHazard(evade(lane=2, A=6.0, name="swerve_right_onto_verge(A_lat=6)")),
          OnHazard(Plan("lane_change_left(T_lc=3)", [{"t": 0, "lane": 0, "T_lc": 3.0}]))]
df_s4, tr_s4 = rc03_scenario(
    "RC03-S4", build_rc03_s4, LIB_S4, WORLDS_S4, nominal=dict(moose_y=LANE_W, onc_dist=200.0), witness="swerve_return(dy=-3.6)",
    refute={"swerve_and_stay_in_opposing_lane(A_lat=6)": "staying in the opposing lane meets the oncoming car",
            "swerve_right_onto_verge(A_lat=6)": "the verge is not drivable (road boundary)",
            "lane_change_left(T_lc=3)": "comfort-rate move does not clear the moose"},
    contrast=("the oncoming car is 110 m away at t0, arriving during the swerve; moose unchanged", dict(onc_dist=110.0)),
    sweep=dict(var="onc_dist", values=np.arange(80.0, 261.0, 20.0), xlabel="oncoming car distance at t0 [m]"),
    xlim=(-35, 140), title="gap in oncoming traffic",
    record=dict(site="straight rural two-lane two-way road (lane 1 = ego direction, lane 0 = opposing), no shoulder",
                story="a moose steps out of the right verge and freezes in the ego lane; braking cannot stop in time; the "
                      "ego swerves through the opposing lane in a gap before an oncoming car and returns",
                t0_rule="t0 = the moose starts moving onto the road; policies trigger on hazard_seen() (lateral intrusion)",
                carla_notes=["animal: a static-mesh prop of 2.6 x 0.9 m moved along scripted_trajectories with "
                             "set_transform; confirm it is in the collision/semantic set the ego perceives",
                             "oncoming car: scripted constant velocity (heading pi), does not react to the ego"]))


### RC03-S5 — Wrong-way driver while overtaking a truck: brake, then slot in behind it

**Story.** At night the ego (30 m/s) is in the **left lane of a 2-lane motorway**, closing on a truck in the right lane that it is
about to overtake. A **wrong-way driver** appears in the ego's lane at the edge of headlight range (200 m) and does not react.
Braking in lane cannot avoid an oncoming car. Steering right at once hits the truck, and the median barrier closes the left side.
The ego **brakes hard first**, drops behind the truck, and **then steers into the slot** between the truck and the car following it.

**Why braking fails:** the obstacle is *moving toward* the ego, so stopping does not create separation.
**Escape:** a *brake-steer* — the lateral corridor opens only after a braking prefix. This is the card's "brake–steer continuation".
**Declared worlds:** wrong-way speed ∈ {25, 30, 35} m/s × truck speed ∈ {23, 25} m/s (with a 21 m/s truck *and* a 35 m/s wrong-way car the witness fails; that world is left out of the declared set).
**Paired contrast:** the right lane holds a rigid 4-truck convoy with 6 m gaps.
**Sweep:** distance at which the wrong-way car is first seen.

In [ ]:
def build_rc03_s5(v=30.0, d_ww=200.0, v_ww=30.0, v_truck=23.0, truck_gap=5.0, follow_gap=30.0, platoon=0, T=8.0):
    """2-lane motorway carriageway at night. The ego (lane 0, fast lane) is closing on a truck in lane 1 to overtake
    it. A wrong-way driver appears in lane 0 at d_ww m (headlight range, t0) and does not react. Braking in lane 0
    cannot avoid an oncoming car; the escape is a brake-steer into lane 1 behind the truck.
    platoon (paired contrast): number of further trucks in a rigid convoy behind the first (6 m gaps)."""
    x_truck = HALF_L + truck_gap + 6.0                          # truck centre: its rear is truck_gap ahead of the ego front
    acts = [Actor("wrong_way", x=0, lane=0, kind="scripted", law=const_path(HALF_L + d_ww + 2.5, 0.0, v_ww, psi=math.pi),
                  note=f"wrong-way car {v_ww} m/s in lane 0, visible from {d_ww} m, never reacts"),
            Actor("truck", x=x_truck - v_truck * 1.0, lane=1, length=12.0, width=2.55, speed=v_truck, kind="vehicle",
                  law=law_idm(v_truck, T=1.5, react=1.0), note=f"truck 12 m, IDM {v_truck} m/s, rear {truck_gap} m ahead of the ego front"),
            *[Actor(f"truck{i + 2}", x=0, lane=1, length=12.0, width=2.55, kind="scripted",
                    law=const_path(x_truck - 18.0 * (i + 1), LANE_W, v_truck),
                    note=f"paired contrast: convoy truck at a constant {v_truck} m/s, 6 m gap") for i in range(platoon)],
            Actor("truck_follower", x=x_truck - 18.0 * platoon - 6.0 - follow_gap - 2.5 - v_truck * 1.0, lane=1, speed=v_truck, kind="vehicle",
                  law=law_idm(v_truck, T=1.2, react=0.8), note=f"car {follow_gap} m behind the truck, IDM {v_truck} m/s")]
    return Scene("RC03-S5", lanes=2, ego_lane=0, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 visible=lambda t, n: n != "wrong_way" or t >= 0.0, prefix_policy=cruise(), prefix_T=1.0,
                 world=dict(v=v, d_ww=d_ww, v_ww=v_ww, v_truck=v_truck, truck_gap=truck_gap, follow_gap=follow_gap,
                           platoon=platoon))


WORLDS_S5 = [dict(v_ww=vw, v_truck=vt) for vw in (25.0, 30.0, 35.0) for vt in (23.0, 25.0)]
LIB_S5 = [OnHazard(brake_hold(6.0)), OnHazard(EscapeAdjacent(1, A_lat=5.0, wait_slot=True)),
          OnHazard(EscapeAdjacent(1, A_lat=5.0, name="escape_right_immediately(A_lat=5)")),
          OnHazard(evade(lane=-1, A=6.0, name="swerve_left_into_median(A_lat=6)")),
          OnHazard(Plan("lane_change_right(T_lc=3)", [{"t": 0, "lane": 1, "T_lc": 3.0}]))]
df_s5, tr_s5 = rc03_scenario(
    "RC03-S5", build_rc03_s5, LIB_S5, WORLDS_S5, nominal=dict(v_ww=30.0, v_truck=23.0), witness="escape_lane1(A_lat=5.0)",
    refute={"escape_right_immediately(A_lat=5)": "steering right at once hits the truck being overtaken",
            "swerve_left_into_median(A_lat=6)": "the median barrier bounds the carriageway",
            "lane_change_right(T_lc=3)": "a comfort lane change does not wait for the slot behind the truck"},
    contrast=("lane 1 holds a rigid 4-truck convoy with 6 m gaps; wrong-way driver unchanged", dict(platoon=3)),
    sweep=dict(var="d_ww", values=np.arange(100.0, 281.0, 20.0), xlabel="wrong-way car first visible at [m]"),
    xlim=(-35, 200), title="detection range of the wrong-way driver",
    record=dict(site="straight 2-lane motorway carriageway with median barrier; ego in lane 0 (fast lane) overtaking a truck",
                story="at night a wrong-way driver appears in the ego's lane; braking cannot avoid an oncoming car; the "
                      "truck the ego was about to overtake blocks the right lane, so the ego brakes hard to drop behind it "
                      "and then steers into the slot",
                t0_rule="t0 = first tick the wrong-way car is visible (headlight range d_ww); policies trigger on hazard_seen()",
                carla_notes=["wrong-way car: scripted constant velocity, heading pi, in the ego's lane; no reaction",
                             "truck: a CARLA truck blueprint (re-measure its bounding box) driven by client-side IDM",
                             "outside the declared worlds, v_truck = 21 m/s with v_ww = 35 m/s refutes this witness: report, "
                             "do not fold in"]))


### RC03-S6 — Lost cargo with a car in the blind spot: partial shift onto the shoulder

**Story.** The ego follows a pickup at 28 m/s with a 1 s gap in the right lane of a 2-lane motorway (paved 2.5 m shoulder). A **crate
falls off the pickup**, tumbles and stops in the **left part of the ego lane**. The ego cannot stop in time. A car is sitting in the
ego's **blind spot** in the left lane, so a lane change left (full or partial) hits it. A full lane change is not needed: a **2.3 m shift
to the right**, partly onto the shoulder, clears the crate. The ego then returns to its lane.

**Why braking fails:** *short following distance* to a vehicle that sheds a load (the debris stops far faster than the pickup).
**Escape:** a *partial* lateral offset inside the lane plus shoulder, not a lane change; a smaller in-lane shift (1.2 m) is too little.
**Declared worlds:** crate rests 0.6 or 0.2 m left of the lane centre × blind-spot car 3 m behind or 2 m ahead of the ego centre.
**Paired contrast:** guard rail at the lane edge (no paved shoulder).
**Sweep:** paved shoulder width.

In [ ]:
def build_rc03_s6(v=28.0, headway=1.0, debris_y=-0.6, a_debris=15.0, shoulder=2.5, blind_dx=-3.0, T=8.0):
    """2-lane motorway, ego in lane 1 (right) following a pickup at `headway` s. At t0 a crate falls off the pickup,
    tumbles and stops in the left part of the ego lane. A car sits in the ego's blind spot in lane 0, so a full lane
    change left is closed; the escape is a partial shift right onto the paved shoulder edge, then back."""
    x_p = HALF_L + v * headway + 2.75                           # pickup centre at t0
    acts = [Actor("pickup", x=0, lane=1, length=5.5, kind="scripted", law=const_path(x_p, LANE_W, v),
                  note=f"pickup {v} m/s, keeps going"),
            Actor("crate", x=0, lane=1, length=1.5, width=1.0, kind="scripted",
                  law=crash_path(x_p - 1.6, LANE_W, v, v, a_debris, LANE_W + debris_y, 0.9, 0.8),
                  note=f"crate 1.5x1.0 m falls from the bed at t0, tumbles to rest at {a_debris} m/s^2, {debris_y:+.1f} m from the lane centre"),
            Actor("blind_spot_car", x=blind_dx - v * 1.0, lane=0, speed=v, kind="vehicle", law=law_idm(v, T=1.2, react=0.8),
                  note=f"car in lane 0 at {v} m/s, {blind_dx:+.1f} m from the ego centre (blind spot)")]
    return Scene("RC03-S6", lanes=2, ego_lane=1, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 road_bounds=(-LANE_W / 2, 1.5 * LANE_W + shoulder), prefix_policy=cruise(), prefix_T=1.0,
                 world=dict(v=v, headway=headway, debris_y=debris_y, a_debris=a_debris, shoulder=shoulder, blind_dx=blind_dx))


WORLDS_S6 = [dict(debris_y=dy, blind_dx=bd) for dy in (-0.6, -0.2) for bd in (-3.0, 2.0)]
LIB_S6 = [OnHazard(brake_hold(6.0)), OnHazard(SwerveReturn(2.3)),
          OnHazard(EscapeAdjacent(0, A_lat=6.0, name="lane_change_left_evasive(A_lat=6)")),
          OnHazard(SwerveReturn(-2.3)), OnHazard(SwerveReturn(1.2))]
df_s6, tr_s6 = rc03_scenario(
    "RC03-S6", build_rc03_s6, LIB_S6, WORLDS_S6, nominal=dict(debris_y=-0.6, blind_dx=-3.0), witness="swerve_return(dy=+2.3)",
    refute={"lane_change_left_evasive(A_lat=6)": "the blind-spot car occupies the left lane",
            "swerve_return(dy=-2.3)": "a partial shift left also meets the blind-spot car",
            "swerve_return(dy=+1.2)": "a shift that stays inside the lane does not clear the crate"},
    contrast=("guard rail at the lane edge (no paved shoulder); crate and blind-spot car unchanged", dict(shoulder=0.0)),
    sweep=dict(var="shoulder", values=np.arange(0.0, 3.51, 0.5), xlabel="paved shoulder width [m]"),
    xlim=(-35, 140), title="paved shoulder width",
    record=dict(site="straight 2-lane motorway, 2.5 m paved shoulder on the right; ego in lane 1 (right lane)",
                story="a crate falls off the pickup the ego is following and stops in the left part of the ego lane; a car "
                      "in the ego's blind spot closes the left lane; a partial shift right onto the shoulder edge clears "
                      "the crate, then the ego returns to its lane",
                t0_rule="t0 = the crate leaves the pickup bed; policies trigger on hazard_seen() (crate deceleration)",
                carla_notes=["crate: a static-mesh prop (1.5 x 1.0 m) moved along scripted_trajectories; before t0 it "
                             "rides on the pickup, so attach it or spawn it at t0",
                             "the witness never leaves the carriageway: check the shoulder is drivable in the chosen map"]))


### RC03-S7 — Lost cargo with an open left lane: full evasive lane change left

**Story.** Same start as S6: the ego follows a pickup at 28 m/s with a 1 s gap in the right lane of a 2-lane motorway, and a
**crate falls off the pickup** and comes to rest in the ego lane. This time there is **no shoulder** — a guard rail runs along the
right lane edge — but the **left lane has room**: the nearest left-lane cars are about 20 m ahead of and behind the ego (a 45 m
gap). The ego makes a **full evasive lane change left** into that gap and carries on behind the left-lane car; the left-lane
follower reacts to the cut-in.

**Why braking fails:** *short following distance* to a vehicle that sheds a load.
**Escape:** a complete lane change left (the mirror image of S6, where a blind-spot car closed the left side and the shoulder was open).
**Declared worlds:** crate rests 0.3 m right or 0.3 m left of the lane centre × left-lane follower headway 1.0 or 1.5 s.
**Paired contrast:** the left lane is bumper-to-bumper (8 m gap).
**Sweep:** size of the gap in the left lane.

In [ ]:
def build_rc03_s7(v=28.0, headway=1.0, debris_y=0.3, a_debris=15.0, left_gap=45.0, gap_center=-3.0, follow_T=1.5,
                  follow_react=0.8, blind_dx=None, T=8.0):
    """2-lane motorway with a guard rail at the right lane edge (no shoulder); ego in lane 1 (right) following a pickup at
    `headway` s. At t0 a crate falls off the pickup and stops in the ego lane. The left lane carries IDM traffic with a
    bumper gap `left_gap` centred `gap_center` m from the ego centre, so a full lane change left is open.
    `blind_dx` optionally adds a car in the ego's blind spot (the RC03-S6 situation)."""
    x_p = HALF_L + v * headway + 2.75                           # pickup centre at t0
    v_l = v - 2.0
    acts = [Actor("pickup", x=0, lane=1, length=5.5, kind="scripted", law=const_path(x_p, LANE_W, v),
                  note=f"pickup {v} m/s, keeps going"),
            Actor("crate", x=0, lane=1, length=1.5, width=1.0, kind="scripted",
                  law=crash_path(x_p - 1.6, LANE_W, v, v, a_debris, LANE_W + debris_y, 0.9, 0.8),
                  note=f"crate 1.5x1.0 m falls from the bed at t0, tumbles to rest at {a_debris} m/s^2, {debris_y:+.1f} m from the lane centre"),
            Actor("left_lead", x=gap_center + left_gap / 2 + 2.4 - v_l * 1.0, lane=0, speed=v_l, kind="vehicle",
                  law=law_idm(v_l, T=1.5, react=0.8), note=f"IDM v0={v_l}, front of the left-lane gap"),
            Actor("left_follow", x=gap_center - left_gap / 2 - 2.4 - v_l * 1.0, lane=0, speed=v_l, kind="vehicle",
                  law=law_idm(v_l, T=follow_T, react=follow_react),
                  note=f"IDM v0={v_l}, T={follow_T} s, perception delay {follow_react} s (reacts to the ego cut-in)")]
    if blind_dx is not None:
        acts.append(Actor("blind_spot_car", x=blind_dx - v * 1.0, lane=0, speed=v, kind="vehicle",
                          law=law_idm(v, T=1.2, react=0.8), note=f"paired contrast: car in the ego's blind spot, {blind_dx:+.1f} m"))
    return Scene("RC03-S7", lanes=2, ego_lane=1, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 prefix_policy=cruise(), prefix_T=1.0,
                 world=dict(v=v, headway=headway, debris_y=debris_y, a_debris=a_debris, left_gap=left_gap,
                            gap_center=gap_center, follow_T=follow_T, follow_react=follow_react, blind_dx=blind_dx))


WORLDS_S7 = [dict(debris_y=dy, follow_T=ft) for dy in (0.3, -0.3) for ft in (1.0, 1.5)]
LIB_S7 = [OnHazard(brake_hold(6.0)), OnHazard(EscapeAdjacent(0, A_lat=6.0, name="lane_change_left_evasive(A_lat=6)")),
          OnHazard(SwerveReturn(2.3)), OnHazard(SwerveReturn(1.2)),
          OnHazard(Plan("lane_change_left(T_lc=3)", [{"t": 0, "lane": 0, "T_lc": 3.0}]))]
df_s7, tr_s7 = rc03_scenario(
    "RC03-S7", build_rc03_s7, LIB_S7, WORLDS_S7, nominal=dict(debris_y=0.3, follow_T=1.5),
    witness="lane_change_left_evasive(A_lat=6)",
    refute={"swerve_return(dy=+2.3)": "there is no shoulder: the guard rail closes the right side",
            "swerve_return(dy=+1.2)": "a shift that stays inside the lane does not clear the crate",
            "lane_change_left(T_lc=3)": "a comfort-rate lane change is too slow to clear the crate"},
    contrast=("the left lane is bumper-to-bumper: the gap beside the ego shrinks to 8 m; crate and guard rail unchanged", dict(left_gap=8.0)),
    sweep=dict(var="left_gap", values=np.arange(10.0, 60.1, 5.0), xlabel="bumper gap in the left lane [m]"),
    xlim=(-45, 140), title="gap in the left lane",
    record=dict(site="straight 2-lane motorway, guard rail at the right lane edge (no shoulder); ego in lane 1 (right lane)",
                story="a crate falls off the pickup the ego is following and stops in the ego lane; the guard rail rules "
                      "out the right side, but the left lane has a gap beside the ego, so the ego makes a full evasive "
                      "lane change left and continues behind the left-lane car",
                t0_rule="t0 = the crate leaves the pickup bed; policies trigger on hazard_seen() (crate deceleration)",
                carla_notes=["crate: a static-mesh prop (1.5 x 1.0 m) moved along scripted_trajectories; before t0 it "
                             "rides on the pickup, so attach it or spawn it at t0",
                             "left-lane cars use actor_laws (IDM, perception delay); the follower reacts to the cut-in"]))


### RC03-S8 — City intersection: a side-street car pulls past its stop sign into the ego lane

**Story.** The ego drives at **60 km/h (16.7 m/s) in the right (kerb) lane of a two-lane city arterial**. At the next
intersection the arterial has no stop sign, but the **side street from the right does**. The car waiting there finishes its
stop and pulls out — then stops again when its driver sees the ego, with its **front 2–2.8 m into the ego lane** (about half its
body across the lane), only 25 m ahead. At 60 km/h the ego needs ~30 m to stop, so **braking is too late**. The **left lane is
open** (a car is well ahead), so the ego makes an evasive lane change left.

**Why braking fails:** a *crossing-traffic intrusion* at short range — the side-street car enters the ego lane ~1.5 s ahead.
**Escape:** a full evasive lane change left; staying in the lane (a 1.5 m nudge) does not clear a car that far in, and a
comfort-rate lane change is too slow. The clips draw the side street and its stop line (HighwayEnv has only the arterial).
**Declared worlds:** the car's front stops 2.0 or 2.8 m into the lane × it pulls out over 1.2 or 1.8 s.
**Paired contrast:** a car is beside the ego in the left lane.
**Sweep:** distance to the side-street car when it pulls out.

In [ ]:
STREET_W = 8.0                                              # width of the side street (two 4 m lanes)


def intersection_overlay(x_c, street_w=STREET_W, scaling=6.0, centre=(0.35, 0.5), size=(1000, 240)):
    """Annotate HighwayEnv frames with a cross street at x_c: the street surface beyond both road edges, the crossing
    direction's stop line on the right-hand side, and labels."""
    from PIL import Image, ImageDraw, ImageFont
    font = ImageFont.load_default(size=13)

    def f(tr, frames, every):
        xs, ys = tr.rec["x"][::every], tr.rec["y"][::every]
        if len(xs) != len(frames):
            return frames
        out = []
        for fr, ex, ey in zip(frames, xs, ys):
            im = Image.fromarray(fr[..., :3]).convert("RGBA")
            ov = Image.new("RGBA", im.size, (0, 0, 0, 0))
            d = ImageDraw.Draw(ov)
            px = lambda x: centre[0] * size[0] + (x - ex) * scaling
            py = lambda y: centre[1] * size[1] + (y - ey) * scaling
            top, bot = py(-LANE_W / 2), py((tr.scene.lanes - 0.5) * LANE_W)
            x0, x1 = px(x_c - street_w / 2), px(x_c + street_w / 2)
            d.rectangle([x0, 0, x1, top], fill=(255, 255, 255, 40))
            d.rectangle([x0, bot, x1, size[1]], fill=(255, 255, 255, 40))
            d.line([px(x_c), 0, px(x_c), top], fill=(220, 200, 60, 255), width=1)
            d.line([px(x_c), bot, px(x_c), size[1]], fill=(220, 200, 60, 255), width=1)
            ys_stop = bot + 2.0 * scaling                       # stop line 2 m from the road edge, inbound lane only
            d.line([px(x_c), ys_stop, x1, ys_stop], fill=(255, 255, 255, 255), width=3)
            d.text((x1 + 6, ys_stop + 4), "STOP", fill=(255, 90, 90, 255), font=font, anchor="lt")
            d.text((x1 + 6, size[1] - 10), "side street (stop sign)", fill=(230, 230, 230, 255), font=font, anchor="ls")
            out.append(np.asarray(Image.alpha_composite(im, ov).convert("RGB")))
        return out
    return f


def build_city_stop(sid="RC03-S8", v=16.7, d_x=25.0, protrude=2.4, T_creep=1.5, left_alongside=False,
                    rear_v=None, rear_gap=50.0, rear_react=1.0, T=8.0, terminal="track"):
    """City arterial, two lanes in the ego's direction (lane 0 left, lane 1 right), ego in the right lane at v (60 km/h).
    A side street joins from the right at an intersection where only the side street has a stop sign. At t0 the car
    waiting at its stop line pulls out and stops again with its front `protrude` m into the ego lane, d_x m ahead
    (bumper gap to its near side). The left lane is open (a car well ahead). `left_alongside` (RC03 contrast): a car
    beside the ego in the left lane. `rear_v` (RC07): a speeding car comes up the left lane from rear_gap m behind."""
    x_c = HALF_L + d_x + 1.0                                   # crossing car centre along the arterial (its width is along x);
                                                               # it sits in the side street's inbound lane, right of its centre line
    y_edge = 1.5 * LANE_W                                      # right edge of the ego lane (kerb line)
    acts = [Actor("crossing_car", x=0, lane=1, length=4.8, width=2.0, kind="scripted",
                  law=crossing_path(x_c, y_edge + 2.0 + 2.4, y_edge + 2.4 - protrude, 0.0, T_creep),
                  note=f"side-street car at its stop line pulls out at t0 and stops after {T_creep} s with its front "
                       f"{protrude} m into the ego lane (half its body across the lane line of the kerb lane)"),
            Actor("left_lead", x=45.0 - v * 1.0, lane=0, speed=v - 1.0, kind="vehicle", law=law_idm(v - 1.0, T=1.5, react=0.8),
                  note="car well ahead in the left lane")]
    if left_alongside:
        acts.append(Actor("left_alongside", x=-1.0 - v * 1.0, lane=0, speed=v, kind="vehicle", law=law_idm(v, T=1.2, react=0.8),
                          note="paired contrast: a car beside the ego in the left lane"))
    if rear_v is not None:
        acts.append(Actor("speeding_car", x=-HALF_L - rear_gap - 2.4 - rear_v * 1.0, lane=0, speed=rear_v, length=4.8,
                          kind="vehicle", law=law_trigger(ego_ahead_in_lane(20.0), -5.0, reaction=rear_react),
                          note=f"left lane, {rear_v} m/s, {rear_gap} m behind at t0; notices a car ahead in its lane only "
                               f"within 20 m, then brakes 5 m/s^2 after {rear_react} s"))
    return Scene(sid, lanes=2, ego_lane=1, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0, terminal=terminal,
                 prefix_policy=cruise(), prefix_T=1.0,
                 world=dict(v=v, d_x=d_x, protrude=protrude, T_creep=T_creep, left_alongside=left_alongside,
                            rear_v=rear_v, rear_gap=rear_gap, rear_react=rear_react),
                 meta=dict(overlay=intersection_overlay(x_c - LANE_W / 2), x_intersection=x_c - LANE_W / 2))


build_rc03_s8 = lambda **kw: build_city_stop("RC03-S8", **kw)
WORLDS_S8 = [dict(protrude=p, T_creep=tc) for p in (2.0, 2.8) for tc in (1.2, 1.8)]
LIB_S8 = [OnHazard(brake_hold(6.0)), OnHazard(EscapeAdjacent(0, A_lat=6.0, name="escape_left(A_lat=6)")),
          OnHazard(Plan("lane_change_left(T_lc=3)", [{"t": 0, "lane": 0, "T_lc": 3.0}])),
          OnHazard(SwerveReturn(-1.5, A_lat=4.0, name="nudge_left_in_lane(dy=-1.5)"))]
df_s8, tr_s8 = rc03_scenario(
    "RC03-S8", build_rc03_s8, LIB_S8, WORLDS_S8, nominal=dict(protrude=2.8, T_creep=1.2), witness="escape_left(A_lat=6)",
    refute={"lane_change_left(T_lc=3)": "a comfort-rate lane change does not clear the car's front in time",
            "nudge_left_in_lane(dy=-1.5)": "staying in the lane does not clear a car protruding 2-2.8 m"},
    contrast=("a car is beside the ego in the left lane; crossing car unchanged", dict(left_alongside=True)),
    sweep=dict(var="d_x", values=np.arange(8.0, 40.1, 2.0), xlabel="distance to the crossing car when it pulls out [m]"),
    xlim=(-25, 90), title="distance when the side-street car pulls out",
    record=dict(site="city arterial, 2 lanes in the ego's direction, 60 km/h; intersection with a side street from the "
                     "right that has a stop sign (the arterial has none); ego in lane 1 (right / kerb lane)",
                story="a side-street car finishes its stop, pulls out and stops again with half its body in the ego's lane "
                      "just ahead; braking is too late; the left lane is open, so the ego changes lanes",
                t0_rule="t0 = the side-street car starts to pull out; policies trigger on hazard_seen() (lateral intrusion)",
                carla_notes=["needs a map intersection with a side street on the right and a stop sign for it only "
                             "(e.g. Town03 / Town10HD minor junctions); place the crossing car on the side-street lane",
                             "crossing car: replay scripted_trajectories (heading -pi/2 relative to the ego, i.e. "
                             "pointing across the arterial)"]))


### RC03 scenario gallery
The executed witness of every RC03 scenario in its nominal world (blue = ego, orange = actors, grey = static obstacles).
All eight share the card's contrast: `brake_hold` is refuted and a *complete* lateral escape is supported. They differ in why
braking fails and in what kind of lateral corridor is used.

In [ ]:
GALLERY = [("RC03-S1", tr03s1[k03(WITNESS)])] + [
    (sid, trs[(HANDOFF[sid]["witness"], json.dumps(HANDOFF[sid]["nominal_world"], sort_keys=True))])
    for sid, trs in [("RC03-S2", tr_s2), ("RC03-S3", tr_s3), ("RC03-S4", tr_s4), ("RC03-S5", tr_s5), ("RC03-S6", tr_s6), ("RC03-S7", tr_s7), ("RC03-S8", tr_s8)]]
for sid, tr in GALLERY:
    tr.policy.name = f"{sid}: {tr.policy.inner.name}"
plot_traces([tr for _, tr in GALLERY], "RC03 scenarios - executed witnesses (nominal worlds)", xlim=(-35, 160), height=1.9)
for sid, tr in GALLERY:
    tr.policy.name = tr.policy.inner.name


---
## RC07 — Primary avoidance with continued secondary-threat management  (`R_REF` only with a complete post-manoeuvre policy)
**Card (recap).** Two policies avoid the front obstacle, but one hands back or stops in the path of a secondary actor that
arrives *after* the first avoidance; a continued, controlled response remains feasible. Witness: complete the primary avoidance
and continue in the supported corridor until separation from the secondary actor and tracking satisfy the terminal contract.
*Accept only when* short-horizon success is compared with full-horizon margins; refute the premature-handback policy without
falsely declaring the scene unrecoverable. *Paired contrast:* close the continuation corridor or advance the secondary actor;
the negative side needs complete library evaluation or a stronger proof. **Required profile contrast:** primary-only test
passes; full handback policy `< ε`; supported continuation `≥ ε`.

**How the scenarios implement it.**
* *Primary-only test* = every candidate evaluated for the first **3 s** after t0 with no terminal contract (`terminal='none'`).
* *Full horizon* = 12 s with the tracking terminal contract (`terminal='track'`, last 2 s).
* Every candidate is wrapped in `OnHazard`, so they share the same detection; they differ only in what happens **after**
  the primary avoidance: a *handback* (stay where the swerve left the ego, or make a protective stop) vs a *continuation*.
* Each scenario also checks that the handback's secondary conflict happens **after** the 3 s primary horizon — the case's point
  is that a short scoring window would wrongly accept it.

### RC07 — shared runner and helpers
`rc07_scenario(...)` runs the card's evidence pattern for one scenario: primary-only profile (3 s) and full-horizon profile
over the declared worlds → traces of the handback(s) and the witness in the nominal world → a sweep of the secondary actor
(handback curve in the nominal world, witness curve = min over the declared worlds) → the card's checks → the paired contrast
→ the CARLA hand-off record (which also stores both profiles and the handback's secondary-conflict time).
Helpers: `ego_ahead_in_lane(dist)` — trigger for a late-noticing driver (reacts only once the ego is in its lane within `dist` m);
`t_closest(tr)` — time of the conflict (or of the smallest clearance) after t0. `SwerveReturn` gained `A_ret`, a separate
(gentler) lateral-acceleration cap for the return, which is not an emergency move.

In [ ]:
def ego_ahead_in_lane(dist, half_w=2.0):
    """Trigger for a late-noticing driver: the ego is in this actor's lane, ahead of it along its heading, within dist m."""
    def cond(t, me, sim):
        e = sim.ego
        dx = (e.position[0] - me.position[0]) * math.cos(me.heading)
        return abs(e.position[1] - me.position[1]) < half_w and 0.0 < dx < dist
    return cond


def t_closest(tr):
    """Time after t0 of the conflict (or, if none, of the smallest clearance margin) in a trace."""
    if tr.conflict_t is not None:
        return float(tr.conflict_t - tr.t0)
    R = tr.rec
    pol = R["phase"] == "policy"
    i = int(np.argmin(np.where(pol, R["m_clear"], np.inf)))
    return float(R["t"][i] - tr.t0)


def rc07_scenario(sid, build, lib, worlds, nominal, witness, handback, contrast, sweep, xlim, title, record, T_short=3.0):
    """Run one RC07 scenario with the card's evidence pattern:
    primary-only (short-horizon, T_short s) profile vs full-horizon profile over the declared worlds -> traces ->
    single-variable sweep of the secondary threat -> card checks -> paired contrast -> CARLA hand-off.
    handback: {policy name: reason} - premature handback / stop policies that must pass the short horizon but fail the
              full horizon, with their first conflict after T_short.
    contrast: (description, build kwargs that close the continuation corridor or advance the secondary actor)."""
    short = evaluate(lambda **w: build(**w, T=T_short, terminal="none"), lib, worlds)
    full, trs = evaluate(build, lib, worlds, keep=True)
    ps, pf = robust_profile(short), robust_profile(full)
    key = lambda p: (p, json.dumps(nominal, sort_keys=True))
    worst = full.loc[full.groupby("policy", sort=False)["r"].idxmin()].set_index("policy")
    display(pd.DataFrame({f"primary-only r (first {T_short:g} s)": ps.round(2), "full-horizon r": pf.round(2),
                          "binding (worst world)": worst["binding"],
                          "closest approach, nominal [s after t0]": {p.name: round(t_closest(trs[key(p.name)]), 2) for p in lib}}))
    display(full.pivot_table(index="policy", columns="world", values="r", sort=False).round(2))
    plot_traces([trs[key(n)] for n in [*handback, witness]], f"{sid} nominal world {nominal} (full horizon)", xlim=xlim)

    var, xs = sweep["var"], np.asarray(sweep["values"], dtype=float)
    by_name = {p.name: p for p in lib}
    hb0 = next(iter(handback))
    r_hb = [simulate(build(**{**nominal, var: x}), by_name[hb0]).r for x in xs]
    r_wit = [min(simulate(build(**{**w, var: x}), by_name[witness]).r for w in worlds) for x in xs]
    plot_sweep(xs, {hb0: r_hb, f"{witness} (min over worlds)": r_wit}, sweep["xlabel"], f"{sid}: {title}")
    window = [float(x) for x, b, w in zip(xs, r_hb, r_wit) if b < EPS and w >= EPS]
    print(f"{var} values where the handback fails but the continuation passes: "
          + (f"{window[0]} .. {window[-1]}" if window else "none"))

    check(sid, f"primary-only ({T_short:g} s) test passes for every candidate", bool((ps >= EPS).all()))
    for p, why in handback.items():
        check(sid, f"full-horizon {p} < eps: {why}", pf[p] < EPS)
        check(sid, f"{p}: the secondary conflict comes after the {T_short:g} s primary horizon",
              t_closest(trs[key(p)]) > T_short)
    check(sid, f"supported continuation {witness} >= eps in every declared world", pf[witness] >= EPS)
    tw = trs[key(witness)]
    pol = tw.rec["phase"] == "policy"
    check(sid, "witness executes without actuator saturation (steering rate / friction circle)",
          not tw.rec["sat"][pol].any() and tw.comp["dyn"] >= 0)
    check(sid, f"a {var} window exists where the handback fails but the continuation passes", len(window) > 0)
    dfc = evaluate(build, lib, [{**w, **contrast[1]} for w in worlds])
    labc = label_from_profile(robust_profile(dfc), library_complete=True)
    print(f"paired contrast ({contrast[0]}) label from this finite library: {labc} (NOT U_MODEL without a proof)")
    check(sid, "closing the continuation (or advancing the secondary actor) removes the witness -> U_LIBRARY", labc == "U_LIBRARY")

    sc = tw.scene
    handoff(sid, parent="RC07", label=label_from_profile(pf), **record,
            evidence=f"executed witness in HighwayEnv reference dynamics over {len(worlds)} declared worlds; not CARLA-validated",
            scoring=f"primary-only horizon {T_short:g} s (terminal 'none') vs full horizon {sc.T:g} s (terminal '{sc.terminal}')",
            ego={"v": sc.ego_speed, "lane": sc.ego_lane, "envelope": asdict(sc.env)},
            road={"lanes": sc.lanes, "lane_width": LANE_W, "drivable_y": list(sc.bounds)},
            actors=actors_rel(sc, tw), scripted_trajectories=path_keyframes(sc, tw),
            scripted_params={a.name: a.law.params for a in sc.actors if a.kind == "scripted"},
            actor_laws={a.name: a.note for a in sc.actors},
            worlds=worlds, nominal_world=nominal, witness=witness, witness_descriptor=by_name[witness].descriptor(),
            witness_trace=ego_keyframes(tw), handback_refuted=list(handback),
            profiles={"primary_only": ps.round(3).to_dict(), "full_horizon": pf.round(3).to_dict()},
            secondary_conflict_t={p: round(t_closest(trs[key(p)]), 2) for p in handback},
            sweep={"var": var, "values": xs.tolist(), "r_handback_nominal": r_hb, "r_witness_min_worlds": r_wit,
                   "window_handback_fails_continuation_passes": [window[0], window[-1]] if window else None},
            paired_contrast={"change": contrast[0], "kwargs": contrast[1], "label": labc})
    return full, trs


print("RC07 scenario runner ok")


### RC07-S1 — Moose on the highway, then a speeding car in the escape lane
**Story.** The ego drives at 25 m/s (90 km/h) in the **right lane of a two-lane highway**. A **moose leaps from the right verge**
about 45 m ahead and stands in the ego lane. The ego swerves into the left lane, which is free *for now*. But a **speeding car**
(36–40 m/s, 130–145 km/h) is coming up the left lane from 80 m behind; its driver only notices a car in the lane ahead within
25 m and then brakes at 5 m/s² after 0.8–1.2 s. Once past the moose, the ego must **change back to the right lane** before the
speeding car arrives.

**Handback refuted:** staying in the left lane (lane keeping takes over) and a protective stop in the left lane — both avoid
the moose in the first 3 s, and both are hit by the speeding car later.
**Witness:** pass the moose, then return right (gentle 3 m/s² return) and track.
**Declared worlds:** speeding-car speed ∈ {36, 40} m/s × driver reaction ∈ {0.8, 1.2} s.
**Paired contrast:** the speeding car is only 30 m behind at t0 (it arrives while the ego is still beside the moose).
**Sweep:** the speeding car's gap behind the ego at t0 (its arrival time).

In [ ]:
def build_rc07_s1(v=25.0, d_moose=45.0, rear_v=38.0, rear_gap=80.0, rear_notice=25.0, rear_react=1.0, rear_b=5.0,
                  calf=False, T=12.0, terminal="track"):
    """2-lane one-direction highway, ego in the right lane (lane 1). At t0 a moose leaps from the right verge and stops in
    the ego lane; the ego swerves into the left lane (lane 0). A speeding car comes up the left lane from rear_gap m behind;
    its driver notices a car ahead in the lane only within rear_notice m, then brakes rear_b m/s^2 after rear_react s.
    `calf`: optionally a calf stands in the right lane 25 m beyond the moose."""
    x_m = HALF_L + d_moose + 0.45
    x_r = -HALF_L - rear_gap - 2.4                              # speeding car centre at t0
    acts = [Actor("moose", x=0, y=LANE_W * 1.5 + 1.5, length=2.6, width=0.9, kind="scripted",
                  law=crossing_path(x_m, LANE_W * 1.5 + 1.5, LANE_W, 0.0, 1.0),
                  note="moose (2.6 m body across the lane) leaps from the right verge at t0 and stands in the ego lane"),
            Actor("speeding_car", x=x_r - rear_v * 1.0, lane=0, speed=rear_v, length=4.8, kind="vehicle",
                  law=law_trigger(ego_ahead_in_lane(rear_notice), -rear_b, reaction=rear_react),
                  note=f"left lane, {rear_v} m/s, {rear_gap} m behind the ego at t0; notices a car ahead in its lane only "
                       f"within {rear_notice} m, then brakes {rear_b} m/s^2 after {rear_react} s")]
    if calf:
        acts.append(Actor("calf", x=x_m + 25.0, lane=1, length=1.8, width=0.7, kind="static",
                          note="paired contrast: a moose calf stands in the right lane 25 m beyond its mother"))
    return Scene("RC07-S1", lanes=2, ego_lane=1, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 terminal=terminal, prefix_policy=cruise(), prefix_T=1.0,
                 world=dict(v=v, d_moose=d_moose, rear_v=rear_v, rear_gap=rear_gap, rear_notice=rear_notice,
                            rear_react=rear_react, rear_b=rear_b, calf=calf))


WORLDS_7S1 = [dict(rear_v=rv, rear_react=rr) for rv in (36.0, 40.0) for rr in (0.8, 1.2)]
LIB_7S1 = [OnHazard(EscapeAdjacent(0, A_lat=6.0, name="avoid_then_stay_left")),
           OnHazard(EscapeAdjacent(0, A_lat=6.0, stop=True, name="avoid_then_stop_left")),
           OnHazard(SwerveReturn(-LANE_W, A_ret=3.0, name="avoid_continue_return"))]
df_7s1, tr_7s1 = rc07_scenario(
    "RC07-S1", build_rc07_s1, LIB_7S1, WORLDS_7S1, nominal=dict(rear_v=40.0, rear_react=0.8),
    witness="avoid_continue_return",
    handback={"avoid_then_stay_left": "the speeding car arrives in the left lane and cannot stop in time",
              "avoid_then_stop_left": "a protective stop in the left lane is in the speeding car's path"},
    contrast=("the speeding car is only 30 m behind at t0: it arrives while the ego is still passing the moose", dict(rear_gap=30.0)),
    sweep=dict(var="rear_gap", values=np.arange(20.0, 220.1, 20.0), xlabel="speeding car's gap behind the ego at t0 [m]"),
    xlim=(-60, 220), title="arrival of the speeding car",
    record=dict(site="straight 2-lane one-direction highway with a verge; ego in lane 1 (right lane)",
                story="a moose leaps into the right lane; the ego swerves into the left lane, where a speeding car is coming "
                      "up fast; the ego passes the moose and returns to the right lane before the speeding car arrives",
                t0_rule="t0 = the moose starts its leap; policies trigger on hazard_seen() (lateral intrusion)",
                carla_notes=["moose: a 2.6 x 0.9 m prop moved along scripted_trajectories (CARLA has no moose blueprint; "
                             "a large static prop or a walker scaled up works)",
                             "speeding car: autopilot off; constant speed, then the trigger law in actor_laws"]))


### RC07-S2 — Ladder off a van, then the fast right lane: keep up with traffic
**Story.** The ego drives at 25 m/s (90 km/h) in the **middle lane of a 3-lane highway**, 1.5 s behind a van. The left lane is a
dense platoon at the same speed, but the **right lane is flowing clearly faster, at about 35 m/s (125 km/h)**. A **ladder falls off
the van's roof rack** and slides to rest across the ego lane; the ego swerves into a gap in the right lane. A right-lane car is
coming up 45 m behind at 34–36 m/s; its driver expects the lane to keep flowing and reacts only to a car within 20 m ahead.

**Handback refuted:** holding the original 25 m/s in the right lane — the ego becomes a slow car in a fast lane and is
rear-ended about 4 s later; a protective stop there is hit even sooner.
**Witness:** complete the swerve and **accelerate to the right lane's speed** (35 m/s, within the friction circle), so the
follower never closes in.
**Declared worlds:** follower speed ∈ {34, 36} m/s × driver reaction ∈ {1.0, 1.4} s.
**Paired contrast:** the follower is only 15 m behind at t0 (no time to get up to speed).
**Sweep:** the follower's gap behind the ego at t0.

In [ ]:
def build_rc07_s2(v=25.0, headway=1.5, v_right=35.0, f_gap=45.0, f_v=None, f_notice=20.0, f_react=1.2, f_b=5.0,
                  T=12.0, terminal="track"):
    """3-lane highway, ego in the middle lane (lane 1) at v behind a van; the left lane is a dense platoon at v, while the
    right lane flows clearly faster (v_right). At t0 a ladder falls off the van and slides to rest in the ego lane; the
    ego swerves into a gap in the right lane. A right-lane car (f_v, default v_right) is f_gap m behind the ego at t0;
    its driver expects the lane to keep flowing and reacts only to a car within f_notice m ahead (brakes f_b after f_react s)."""
    f_v = v_right if f_v is None else f_v
    x_van = HALF_L + v * headway + 2.75
    x_f = -HALF_L - f_gap - 2.4
    acts = [Actor("van", x=0, lane=1, length=5.5, kind="scripted", law=const_path(x_van, LANE_W, v),
                  note=f"van {v} m/s, keeps going after losing the ladder"),
            Actor("ladder", x=0, lane=1, length=3.6, width=0.5, kind="scripted",
                  law=crash_path(x_van - 1.0, LANE_W, v, v, 12.0, LANE_W + 0.3, 0.35, 1.0),
                  note="3.6 m aluminium ladder falls from the van roof at t0, slides to rest across the ego lane"),
            Actor("right_follower", x=x_f - f_v * 1.0, lane=2, length=4.8, speed=f_v, kind="vehicle",
                  law=law_trigger(ego_ahead_in_lane(f_notice), -f_b, reaction=f_react),
                  note=f"right-lane car at {f_v} m/s, {f_gap} m behind the ego at t0; reacts only to a car within "
                       f"{f_notice} m ahead, braking {f_b} m/s^2 after {f_react} s"),
            Actor("right_lead", x=45.0 - v_right * 1.0, lane=2, speed=v_right, kind="vehicle",
                  law=law_idm(v_right, T=1.5, react=0.8), note=f"right-lane car 45 m ahead, {v_right} m/s")]
    acts += [Actor(f"left_{i}", x=dx - v * 1.0, lane=0, speed=v, kind="vehicle", law=law_idm(v, T=1.0, react=0.8),
                   note="dense left-lane platoon") for i, dx in enumerate((-24.0, -12.0, 0.0, 12.0, 24.0))]
    acts += [Actor(f"right_{i}", x=dx - v_right * 1.0, lane=2, speed=v_right, kind="vehicle",
                   law=law_idm(v_right, T=1.5, react=0.8), note="right-lane traffic") for i, dx in enumerate((-110.0, 95.0))]
    return Scene("RC07-S2", lanes=3, ego_lane=1, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 terminal=terminal, prefix_policy=cruise(), prefix_T=1.0,
                 world=dict(v=v, headway=headway, v_right=v_right, f_gap=f_gap, f_v=f_v, f_notice=f_notice,
                            f_react=f_react, f_b=f_b))


WORLDS_7S2 = [dict(f_v=fv, f_react=fr) for fv in (34.0, 36.0) for fr in (1.0, 1.4)]
LIB_7S2 = [OnHazard(EscapeAdjacent(2, A_lat=5.0, name="avoid_then_hold_speed")),
           OnHazard(EscapeAdjacent(2, A_lat=5.0, b_evasive=6.0, stop=True, name="avoid_then_stop_right")),
           OnHazard(EscapeAdjacent(2, A_lat=5.0, a=3.0, v_set=35.0, name="avoid_then_match_traffic"))]
df_7s2, tr_7s2 = rc07_scenario(
    "RC07-S2", build_rc07_s2, LIB_7S2, WORLDS_7S2, nominal=dict(f_v=36.0, f_react=1.0),
    witness="avoid_then_match_traffic",
    handback={"avoid_then_hold_speed": "at its old 25 m/s the ego is a slow car in a 35 m/s lane and is rear-ended",
              "avoid_then_stop_right": "a protective stop in the fast lane is rear-ended sooner"},
    contrast=("the right-lane car is only 15 m behind at t0 (no time to get up to speed)", dict(f_gap=15.0)),
    sweep=dict(var="f_gap", values=np.arange(10.0, 90.1, 5.0), xlabel="right-lane car's gap behind the ego at t0 [m]"),
    xlim=(-70, 260), title="right-lane follower behind the gap",
    record=dict(site="straight 3-lane highway; the right lane flows ~10 m/s faster than the middle and left lanes; "
                     "ego in lane 1 (middle lane)",
                story="a ladder falls off a van ahead and lands in the ego lane; the left lane is a dense platoon, so the "
                      "ego swerves into a gap in the faster right lane; holding its old speed there (or stopping) makes it "
                      "an obstacle to the fast traffic behind, while accelerating to the right lane's speed keeps the "
                      "follower at a safe distance",
                t0_rule="t0 = the ladder leaves the van's roof rack; policies trigger on hazard_seen() (ladder deceleration)",
                carla_notes=["ladder: a 3.6 x 0.5 m prop moved along scripted_trajectories; spawn it on the van or at t0",
                             "set the right-lane traffic speed explicitly (traffic manager speed difference per vehicle)",
                             "right-lane follower: constant speed, then the trigger law in actor_laws (autopilot off)"]))


### RC07-S3 — Tyre tread in congested traffic, then a lane-splitting motorcycle: recentre *in time*
**Story.** Congested motorway, 20 m/s, a guard rail at the right lane edge and a dense queue in the left lane. The car ahead
**straddles a shed truck-tyre tread**, revealing it in the right half of the ego lane only 30 m ahead. The ego **shifts 1.1 m
left inside its own lane**, toward the lane line, and clears the tread. A **motorcycle is lane-splitting** along that lane line
at 28–32 m/s, only **48 m behind**. Any driver recentres after such a shift; the question is *when*.

**Handback refuted:** a normal, unhurried recentre — drifting back once the tread is ~30 m behind (lateral 1.5 m/s²): the ego
still straddles the motorcycle's line when a 32 m/s motorcycle arrives (~3.8 s after t0); a 28 m/s one would
miss it, so the leisurely recentre is refuted by the faster declared world.
**Witness:** recentre **promptly**, as soon as the tread is behind the ego (lateral 3 m/s²).
**Declared worlds:** motorcycle speed ∈ {28, 32} m/s × its line ∈ {0.2 m left of, on} the lane line.
**Paired contrast:** the motorcycle is only 15 m behind at t0 (it arrives while the ego is still shifted).
**Sweep:** motorcycle gap behind the ego at t0.

In [ ]:
def build_rc07_s3(v=20.0, d_tread=30.0, tread_y=0.9, moto_v=30.0, moto_gap=48.0, moto_y=2.0, moto_notice=15.0, T=12.0, terminal="track"):
    """Congested 2-lane motorway (guard rail on the right, no shoulder); ego in the right lane (lane 1) at v, a dense
    platoon alongside in the left lane. The car ahead straddles a shed truck-tyre tread lying in the right half of the
    ego lane, revealing it at d_tread m (t0). The ego shifts left inside its lane, toward the lane line. A motorcycle
    lane-splits along the lane line (y = moto_y) at moto_v, moto_gap m behind at t0; its rider brakes late."""
    x_t = HALF_L + d_tread + 0.6
    acts = [Actor("tread", x=x_t, y=LANE_W + tread_y, length=1.2, width=0.6, kind="static",
                  note=f"shed tyre tread 1.2 x 0.6 m, {tread_y:+.1f} m right of the ego-lane centre"),
            Actor("lead", x=0, lane=1, length=4.8, kind="scripted", law=const_path(x_t + 8.0, LANE_W, v),
                  note=f"car ahead {v} m/s; it has just straddled the tread"),
            Actor("motorcycle", x=-HALF_L - moto_gap - 1.1 - moto_v * 1.0, y=moto_y, length=2.2, width=0.8, speed=moto_v,
                  kind="vehicle", law=law_trigger(ego_ahead_in_lane(moto_notice, half_w=1.3), -5.0, reaction=0.8),
                  note=f"lane-splitting motorcycle at y={moto_y - LANE_W / 2:+.1f} m from the lane line, {moto_v} m/s, "
                       f"{moto_gap} m behind at t0; brakes 5 m/s^2 0.8 s after the ego blocks its line within 15 m")]
    acts += [Actor(f"left_{i}", x=dx - v * 1.0, lane=0, speed=v, kind="vehicle", law=law_idm(v, T=0.8, react=0.8),
                   note="dense left-lane platoon") for i, dx in enumerate(np.arange(-48.0, 61.0, 12.0))]
    return Scene("RC07-S3", lanes=2, ego_lane=1, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 terminal=terminal, prefix_policy=cruise(), prefix_T=1.0,
                 visible=lambda t, n: n != "tread" or t >= 0.0,
                 world=dict(v=v, d_tread=d_tread, tread_y=tread_y, moto_v=moto_v, moto_gap=moto_gap, moto_y=moto_y))


WORLDS_7S3 = [dict(moto_v=mv, moto_y=my) for mv in (28.0, 32.0) for my in (1.8, 2.0)]
LIB_7S3 = [OnHazard(SwerveReturn(-1.1, A_lat=3.0, A_ret=1.5, margin=30.0, name="avoid_then_recentre_leisurely")),
           OnHazard(SwerveReturn(-1.1, A_lat=3.0, A_ret=3.0, margin=1.0, name="avoid_then_recentre_promptly"))]
df_7s3, tr_7s3 = rc07_scenario(
    "RC07-S3", build_rc07_s3, LIB_7S3, WORLDS_7S3, nominal=dict(moto_v=32.0, moto_y=2.0),
    witness="avoid_then_recentre_promptly",
    handback={"avoid_then_recentre_leisurely": "a normal, unhurried drift back (30 m after the tread) still straddles the "
                                               "motorcycle's line when it arrives"},
    contrast=("the motorcycle is only 15 m behind at t0: it arrives while the ego is still shifted", dict(moto_gap=15.0)),
    sweep=dict(var="moto_gap", values=np.arange(10.0, 90.1, 5.0), xlabel="motorcycle gap behind the ego at t0 [m]"),
    xlim=(-60, 160), title="arrival of the lane-splitting motorcycle",
    record=dict(site="congested straight 2-lane motorway, guard rail at the right lane edge; ego in lane 1 (right lane)",
                story="the car ahead straddles a shed tyre tread, revealing it in the right half of the ego lane; the left "
                      "lane is a dense queue, so the ego shifts left inside its lane toward the lane line; a motorcycle "
                      "lane-splitting along that line is close behind, so the ego must recentre promptly once past the tread - a normal, "
                      "unhurried recentre is too late",
                t0_rule="t0 = the tread is revealed by the car ahead; policies trigger on hazard_seen() (object ahead in lane)",
                carla_notes=["tread: a flat static prop (1.2 x 0.6 m); it must be detectable by the ego's perception",
                             "motorcycle: a motorbike blueprint driven along y = moto_y with the trigger law in actor_laws; "
                             "CARLA's traffic manager does not lane-split, so script it"]))


### RC07-S4 — Fallen cyclist before a crest: don't stop to help in the oncoming lane
**Story.** On a **rural two-lane, two-way road** (20 m/s) the road climbs over a **crest** (+4 % / −4 %, 100 m vertical curve,
top about 55 m ahead). A **cyclist ahead falls** and slides across the ego lane 28 m ahead — too close to stop. The ego swerves
into the **opposing lane**. The instinct is to **stop to help** — but an **oncoming car** (20–25 m/s, 150 m away) is still
hidden behind the crest (car-to-car line of sight, eye height 1.08 m both ways). Its driver brakes hard one second after the
crest stops hiding the scene, which is not enough if the ego is still in its lane.

**Handback refuted:** stopping beside the cyclist in the opposing lane; lane keeping that latches onto the opposing lane.
(Stopping survives one world — a 20 m/s oncoming car that brakes in time — and fails the others, so it is refuted robustly.)
**Witness:** pass the cyclist and return to the own lane before the oncoming car crests.
**Declared worlds:** oncoming speed ∈ {20, 25} m/s × cyclist rests 0 or 0.5 m left of the lane centre.
**Paired contrast:** the oncoming car is already 100 m away at t0 (it appears while the ego is still passing).
**Sweep:** oncoming car distance at t0. The road profile is exported as in RC03-S3; the clips mark the crest.

In [ ]:
def build_rc07_s4(v=20.0, d_cyc=28.0, cyc_y=0.0, onc_v=22.0, onc_x0=150.0, g=0.04, L_vc=100.0, x_pvi=55.0,
                  T=12.0, terminal="track"):
    """Rural two-lane two-way road over a crest (+g / -g, L_vc m vertical curve, top at x_pvi); ego in lane 1 (own
    direction), lane 0 = opposing. At t0 a cyclist ahead falls and slides to rest across the ego lane, d_cyc m ahead.
    The ego swerves into the opposing lane. An oncoming car (onc_v, front at onc_x0 at t0) is hidden behind the crest
    until car-to-car line of sight clears; its driver brakes 6 m/s^2 one second after it can see the scene."""
    prof = CrestProfile(x_pvi, g, -g, L_vc)
    x_c = HALF_L + d_cyc + 0.9
    x_onc = onc_x0 + 2.4                                         # oncoming centre at t0 (front face at onc_x0)
    t_vis = next(t for t in np.arange(0.0, 30.0, DT) if prof.sees(x_c, x_onc - onc_v * t - 2.4, H_EYE, H_EYE))
    acts = [Actor("cyclist", x=0, lane=1, length=1.8, width=0.7, kind="scripted",
                  law=crash_path(x_c, LANE_W + 1.3, 6.0, 6.0, 9.0, LANE_W + cyc_y, 1.2, 0.6),
                  note=f"cyclist riding at 6 m/s near the right edge falls at t0 and slides to rest across the ego lane "
                       f"({cyc_y:+.1f} m from the lane centre)"),
            Actor("oncoming", x=0, lane=0, kind="scripted",
                  law=const_path(x_onc, 0.0, onc_v, psi=math.pi, t_brake=t_vis + 1.0, b=6.0),
                  note=f"oncoming car {onc_v} m/s, hidden by the crest until t0+{t_vis:.2f} s, brakes 6 m/s^2 from "
                       f"t0+{t_vis + 1.0:.2f} s")]
    return Scene("RC07-S4", lanes=2, ego_lane=1, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 terminal=terminal, prefix_policy=cruise(v), prefix_T=1.0,
                 visible=line_of_sight(prof, H_EYE, H_EYE, EYE_DX, gated=lambda n: n == "oncoming"),
                 world=dict(v=v, d_cyc=d_cyc, cyc_y=cyc_y, onc_v=onc_v, onc_x0=onc_x0, g=g, L_vc=L_vc, x_pvi=x_pvi),
                 meta=dict(profile=prof, overlay=crest_overlay(prof), t_onc_visible=t_vis))


WORLDS_7S4 = [dict(onc_v=ov, cyc_y=cy) for ov in (20.0, 25.0) for cy in (0.0, -0.5)]
LIB_7S4 = [OnHazard(EscapeAdjacent(0, A_lat=6.0, b_evasive=6.0, stop=True, name="avoid_then_stop_to_help")),
           OnHazard(EscapeAdjacent(0, A_lat=6.0, name="avoid_then_stay_in_opposing_lane")),
           OnHazard(SwerveReturn(-LANE_W, A_ret=3.0, name="avoid_continue_return"))]
df_7s4, tr_7s4 = rc07_scenario(
    "RC07-S4", build_rc07_s4, LIB_7S4, WORLDS_7S4, nominal=dict(onc_v=25.0, cyc_y=0.0),
    witness="avoid_continue_return",
    handback={"avoid_then_stop_to_help": "stopping beside the fallen cyclist leaves the ego in the opposing lane",
              "avoid_then_stay_in_opposing_lane": "lane keeping latches onto the opposing lane"},
    contrast=("the oncoming car is already near the crest (100 m away at t0): it appears while the ego is passing",
              dict(onc_x0=100.0)),
    sweep=dict(var="onc_x0", values=np.arange(90.0, 260.1, 15.0), xlabel="oncoming car distance at t0 [m]"),
    xlim=(-30, 200), title="arrival of the hidden oncoming car",
    record=dict(site="rural two-lane two-way road over a crest (+4 % / -4 %, 100 m vertical curve); ego in lane 1",
                story="a cyclist ahead falls and slides into the ego lane; the ego swerves into the opposing lane; the "
                      "instinct is to stop to help, but a car is coming over the crest; the ego returns to its own lane "
                      "past the cyclist before the oncoming car appears",
                t0_rule="t0 = the cyclist falls; policies trigger on hazard_seen(); the oncoming car is delivered only "
                        "once car-to-car terrain line of sight clears (eye 1.08 m both ways)",
                elevation=crest_elevation_record(build_rc07_s4(), h_target=H_EYE),
                carla_notes=["road elevation: see elevation (OpenDRIVE <elevation> records); the oncoming car must be hidden "
                             "by the crest itself, not spawned late",
                             "cyclist: a bicycle blueprint (or prop) moved along scripted_trajectories; it lies still after the fall",
                             "the oncoming driver's braking is scripted from its own line of sight (t_brake in scripted_params)"]))


### RC07-S5 — Tyre blowout in the left lane, then a squeezed on-ramp car accelerating into the ego
**Story.** The ego drives at 28 m/s in the **left lane** of a two-lane motorway, 1 s behind another car, next to an **on-ramp
acceleration lane**. The car ahead **blows a tyre** and skids to a stop (9 m/s²) in the left lane; the ego swerves into the right
lane. A **fast car (32–34 m/s) is tailgating up that lane**: 40 m behind at t0 and only **~25 m behind the ego** when the ego has
settled in — close, but with just enough room for one car. The **on-ramp car** (27 m/s, about level with the ego) must merge
before the ramp ends; 2 s after t0 it **cuts into that slot** between the ego and the fast car. The fast car closes to within a few
metres before its driver brakes, so the on-ramp car **floors it (2.5–3.5 m/s² toward 35 m/s)** to avoid being rear-ended —
straight at the ego, whom its driver (watching the mirror) has not registered.

**Handback refuted:** staying in the right lane at 28 m/s (the accelerating on-ramp car runs into the ego ~4–5 s after t0).
**Witness:** pass the stopped car and return to the left lane (gentle 3 m/s² return) before the squeeze develops.
**Declared worlds:** on-ramp car acceleration ∈ {2.5, 3.5} m/s² × fast-car speed ∈ {32, 34} m/s.
**Paired contrast:** a second car is stopped 45 m further on in the left lane (a pile-up), so there is no way back.
**Sweep:** the on-ramp car's position relative to the ego at t0.

In [ ]:
def anything_ahead_in_lane(dist, half_w=2.0):
    """Trigger: any vehicle (the ego or an actor) is in this actor's lane ahead of it within dist m (bumper gap)."""
    def cond(t, me, sim):
        for ob in [sim.ego] + list(sim.objs.values()):
            if ob is me:
                continue
            gap = (ob.position[0] - me.position[0]) * math.cos(me.heading) - (ob.LENGTH + me.LENGTH) / 2
            if abs(ob.position[1] - me.position[1]) < half_w and 0.0 < gap < dist:
                return True
        return False
    return cond


def build_rc07_s5(v=28.0, headway=1.0, m_v=27.0, m_x0=-9.5, m_t=2.0, m_a=3.0, m_vmax=35.0, rear_v=34.0, rear_gap=40.0,
                  rear_notice=12.0, rear_react=0.6, rear_b=6.0,
                  ramp_end=150.0, pileup=False, T=12.0, terminal="track"):
    """2-lane motorway (lane 0 = left, lane 1 = right) with an on-ramp acceleration lane (lane 2) ending ramp_end m ahead.
    Ego in the left lane following a car at `headway` s. At t0 the car ahead blows a tyre, skids at 9 m/s^2 and stops in
    the left lane; the ego swerves into the right lane. A fast car (rear_v) is coming up the right lane rear_gap m behind
    the ego (centre distance at t0), tailgating: it brakes only once something is within rear_notice m ahead or moving
    into its lane. The on-ramp car (m_v, centre m_x0 m from the ego at t0) must merge before the ramp ends: from m_t s
    it cuts into the slot between the ego and the fast car and, with the fast car closing on it, accelerates at m_a up
    to m_vmax - its driver watches the mirror, not the ego ahead.
    `pileup` (paired contrast): a second car is stopped 45 m further on in the left lane."""
    x_l = HALF_L + v * headway + 2.4
    x_rest = x_l + v * v / (2 * 9.0)
    acts = [Actor("blowout_car", x=0, lane=0, length=4.8, kind="scripted", law=const_path(x_l, 0.0, v, t_brake=0.0, b=9.0),
                  note=f"car ahead in the left lane, {v} m/s, blows a tyre at t0 and skids to a stop at 9 m/s^2"),
            Actor("merging_car", x=0, lane=2, length=4.8, kind="scripted",
                  law=merge_accel_path(m_x0, 2 * LANE_W, m_v, LANE_W, m_t, 2.5, a=m_a, t_acc=m_t, v_max=m_vmax),
                  note=f"on-ramp car {m_v} m/s, {m_x0:+.0f} m from the ego at t0; from t0+{m_t} s merges into the right "
                       f"lane over 2.5 s and accelerates at {m_a} m/s^2 to {m_vmax} m/s ahead of the fast car"),
            Actor("fast_car", x=-rear_gap - rear_v * 1.0, lane=1, length=4.8, speed=rear_v, kind="vehicle",
                  law=law_trigger(anything_ahead_in_lane(rear_notice, half_w=3.2), -rear_b, reaction=rear_react, v_floor=24.0),
                  note=f"tailgating fast car in the right lane, {rear_v} m/s, centre {rear_gap} m behind the ego at t0; "
                       f"brakes {rear_b} m/s^2 (down to 24 m/s) {rear_react} s after anything is within {rear_notice} m ahead "
                       f"or moving into its lane"),
            Actor("ramp_end", x=ramp_end + 30.0, lane=2, length=60.0, width=3.0, kind="static",
                  note=f"end of the acceleration lane (gore / barrier) {ramp_end} m ahead")]
    if pileup:
        acts.append(Actor("pileup_car", x=x_rest + 45.0, lane=0, length=4.8, kind="static",
                          note="paired contrast: a second car stopped 45 m further on in the left lane"))
    return Scene("RC07-S5", lanes=3, ego_lane=0, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 terminal=terminal, prefix_policy=cruise(), prefix_T=1.0,
                 world=dict(v=v, headway=headway, m_v=m_v, m_x0=m_x0, m_t=m_t, m_a=m_a, m_vmax=m_vmax, rear_v=rear_v,
                            rear_gap=rear_gap, rear_b=rear_b, ramp_end=ramp_end, pileup=pileup))


WORLDS_7S5 = [dict(m_a=ma, rear_v=rv) for ma in (2.5, 3.5) for rv in (32.0, 34.0)]
LIB_7S5 = [OnHazard(EscapeAdjacent(1, A_lat=5.0, name="avoid_then_stay_right")),
           OnHazard(SwerveReturn(LANE_W, A_lat=5.0, A_ret=3.0, name="avoid_continue_return"))]
df_7s5, tr_7s5 = rc07_scenario(
    "RC07-S5", build_rc07_s5, LIB_7S5, WORLDS_7S5, nominal=dict(m_a=2.5, rear_v=34.0),
    witness="avoid_continue_return",
    handback={"avoid_then_stay_right": "the on-ramp car cuts in behind the ego and accelerates away from the fast car - into the ego"},
    contrast=("a second car is stopped 45 m further on in the left lane (no way back before the merge)", dict(pileup=True)),
    sweep=dict(var="m_x0", values=np.arange(-24.0, 4.1, 2.0), xlabel="on-ramp car position relative to the ego at t0 [m]"),
    xlim=(-35, 220), title="position of the merging car",
    record=dict(site="straight 2-lane motorway with an on-ramp acceleration lane on the right; ego in lane 0 (left lane)",
                story="the car ahead in the left lane blows a tyre and skids to a stop; the ego swerves into the right lane; "
                      "a fast car is tailgating up that lane, ~25 m behind the ego once it has settled in; the on-ramp car - "
                      "which must merge before the ramp ends - cuts into that slot and accelerates hard to avoid being "
                      "rear-ended by the fast car, straight at the ego; "
                      "the ego passes the stopped car and returns to the left lane in time",
                t0_rule="t0 = the tyre blows (hard deceleration); policies trigger on hazard_seen()",
                carla_notes=["needs a map with an on-ramp acceleration lane (e.g. Town04 / Town06 highway entries)",
                             "merging car: replay scripted_trajectories (it does not react to the ego; it reacts to the "
                             "fast car, which is why it accelerates)",
                             "fast car: IDM per actor_laws, or traffic-manager speed set to rear_v",
                             "blowout car: replay its deceleration profile; a tyre-blowout physics model is not needed"]))


### RC07-S6 — City intersection, then a speeding car in the left lane
**Story.** The intersection of RC03-S8: at 60 km/h in the right lane of a city arterial, a side-street car pulls past its stop
sign and stops with half its body in the ego lane 25 m ahead; braking is too late, so the ego changes into the left lane. But a
**speeding car** (24–28 m/s, 85–100 km/h in a 60 zone) is coming up the left lane from 50 m behind; its driver notices a car
ahead in the lane only within 20 m. Once past the side-street car, the ego must **switch back to the right lane**.

**Handback refuted:** staying in the left lane, and a protective stop there after the near miss.
**Witness:** pass the side-street car, return right (gentle 3 m/s² return) and track.
**Declared worlds:** speeding-car speed ∈ {24, 28} m/s × driver reaction ∈ {0.8, 1.2} s.
**Paired contrast:** the speeding car is only 15 m behind at t0 (beside the ego when it needs the left lane).
**Sweep:** the speeding car's gap behind the ego at t0.

In [ ]:
build_rc07_s6 = lambda **kw: build_city_stop("RC07-S6", **{"rear_v": 26.0, "T": 12.0, **kw})
WORLDS_7S6 = [dict(rear_v=rv, rear_react=rr) for rv in (24.0, 28.0) for rr in (0.8, 1.2)]
LIB_7S6 = [OnHazard(EscapeAdjacent(0, A_lat=6.0, name="avoid_then_stay_left")),
           OnHazard(EscapeAdjacent(0, A_lat=6.0, stop=True, name="avoid_then_stop_left")),
           OnHazard(SwerveReturn(-LANE_W, A_ret=3.0, name="avoid_continue_return"))]
df_7s6, tr_7s6 = rc07_scenario(
    "RC07-S6", build_rc07_s6, LIB_7S6, WORLDS_7S6, nominal=dict(rear_v=28.0, rear_react=0.8),
    witness="avoid_continue_return",
    handback={"avoid_then_stay_left": "the speeding car comes up the left lane and cannot stop in time",
              "avoid_then_stop_left": "a protective stop in the left lane is in the speeding car's path"},
    contrast=("the speeding car is only 15 m behind at t0: it is beside the ego when the ego needs the left lane",
              dict(rear_gap=15.0)),
    sweep=dict(var="rear_gap", values=np.arange(10.0, 150.1, 10.0), xlabel="speeding car's gap behind the ego at t0 [m]"),
    xlim=(-60, 160), title="arrival of the speeding car",
    record=dict(site="city arterial, 2 lanes in the ego's direction, 60 km/h; intersection with a stop-controlled side "
                     "street from the right; ego in lane 1 (right / kerb lane)",
                story="a side-street car pulls out past its stop line and stops with half its body in the ego's lane; the "
                      "ego changes into the left lane, where a speeding car is coming up fast; once past the side-street "
                      "car, the ego returns to the right lane before the speeding car arrives",
                t0_rule="t0 = the side-street car starts to pull out; policies trigger on hazard_seen() (lateral intrusion)",
                carla_notes=["same intersection and crossing-car replay as RC03-S8",
                             "speeding car: constant speed, then the trigger law in actor_laws (autopilot off)"]))


### RC07-S7 — Wheel in the middle lane, a broken-down car in the right lane, and the car braking behind it
**Story.** The ego drives at 25 m/s in the **middle lane of a 3-lane highway**; the left lane is a dense platoon. Ahead in the
**right lane, car A has broken down** (85 m ahead). **Car B** is coming up the right lane behind the ego at 31 m/s and has
**planned a firm stop behind A** (≈5.3 m/s², using all the distance it has). A **wheel comes off the truck ahead** and rolls to
rest in the middle lane 22 m ahead, so the ego swerves right. It *could* still brake and stop behind A — but that is exactly
where B planned to stop, and B (reacting 1.0–1.4 s after the ego appears, braking at most 6–6.5 m/s²) cannot stop short enough.

**Handback refuted:** stay in the right lane and stop behind A — the ego avoids A, and B runs into the ego ~4 s after t0.
**Witness:** once past the wheel, **cut back into the middle lane** (a quick 6 m/s² return) and carry on past A.
**Declared worlds:** B's reaction ∈ {1.0, 1.4} s × B's maximum braking ∈ {6.0, 6.5} m/s².
**Paired contrast:** tyre fragments cover 35 m of the middle lane beyond the wheel (no way back before A).
**Sweep:** B's position relative to the ego at t0.

In [ ]:
def law_planned_stop(t_plan, b_plan, notice, react, b_max):
    """Driver who has planned a stop behind a stopped car: constant speed, then b_plan from t_plan (rel. t0); once the
    ego is in its lane ahead within `notice` m it brakes at b_max after `react` s (until standstill)."""
    seen = ego_ahead_in_lane(notice)

    def f(t, me, sim):
        if me.speed <= 0.05:
            return 0.0
        key = ("seen", id(me))
        if key not in sim.mem and seen(t, me, sim):
            sim.mem[key] = t
        if key in sim.mem and t >= sim.mem[key] + react:
            return -b_max
        return -b_plan if t >= t_plan else 0.0
    return f


def build_rc07_s7(v=25.0, d_obs=22.0, x_a=85.0, b_x0=-26.0, b_v0=31.0, b_t=0.5, b_react=1.2, b_max=6.0, debris2=False,
                  T=12.0, terminal="track"):
    """3-lane highway, ego in the middle lane (lane 1) at v; the left lane is a dense platoon. Car A has broken down and
    stands in the right lane (rear face x_a m ahead of the ego centre at t0). Car B comes up the right lane behind the ego
    (centre b_x0, speed b_v0 at t0) and, from t0+b_t, brakes at the constant rate that stops it 2 m behind A - a firm
    stop that uses all the distance available. At t0 a wheel comes off the truck ahead and rolls to rest in the middle
    lane d_obs m ahead. If the ego cuts into the right lane and stops behind A, it takes ~8 m of B's stopping distance;
    B brakes at b_max b_react s after it notices the ego.
    `debris2` (paired contrast): tyre fragments spread over 35 m of the middle lane beyond the wheel."""
    x_truck = HALF_L + v * 1.5 + 6.0
    x_w = HALF_L + d_obs + 0.4
    b_plan = b_v0 ** 2 / (2.0 * ((x_a - 2.0) - (b_x0 + 2.4) - b_v0 * b_t))
    xb_start = b_x0 - b_v0 * 1.0                                 # B centre at t0-1 (constant speed until t0+b_t)
    acts = [Actor("truck", x=0, lane=1, length=12.0, width=2.55, kind="scripted", law=const_path(x_truck, LANE_W, v),
                  note=f"truck ahead in the middle lane, {v} m/s, loses a wheel at t0"),
            Actor("wheel", x=0, lane=1, length=0.8, width=0.8, kind="scripted",
                  law=crash_path(x_w + 12.0, LANE_W, v, v - 3.0, 14.0, LANE_W - 0.2, 0.0, 0.8),
                  note=f"truck wheel, rolls to rest in the middle lane about {d_obs} m ahead"),
            Actor("car_A", x=x_a + 2.4, lane=2, length=4.8, kind="static", note=f"car A broken down in the right lane, {x_a} m ahead"),
            Actor("car_B", x=xb_start, lane=2, length=4.8, speed=b_v0, kind="vehicle",
                  law=law_planned_stop(b_t, b_plan, 25.0, b_react, b_max),
                  note=f"car B in the right lane, {b_v0} m/s, {b_x0:+.0f} m from the ego at t0; from t0+{b_t} s brakes "
                       f"{b_plan:.2f} m/s^2 to stop 2 m behind A; brakes {b_max} m/s^2 {b_react} s after it notices the "
                       f"ego in its lane within 25 m")]
    acts += [Actor(f"left_{i}", x=dx - v * 1.0, lane=0, speed=v, kind="vehicle", law=law_idm(v, T=1.0, react=0.8),
                   note="dense left-lane platoon") for i, dx in enumerate((-36.0, -24.0, -12.0, 0.0, 12.0, 24.0, 36.0))]
    if debris2:
        acts.append(Actor("debris2", x=x_w + 5.0 + 17.5, lane=1, length=35.0, width=2.0, kind="static",
                          note="paired contrast: tyre fragments spread over 35 m of the middle lane beyond the wheel"))
    return Scene("RC07-S7", lanes=3, ego_lane=1, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 terminal=terminal, prefix_policy=cruise(), prefix_T=1.0,
                 world=dict(v=v, d_obs=d_obs, x_a=x_a, b_x0=b_x0, b_v0=b_v0, b_t=b_t, b_react=b_react, b_max=b_max, debris2=debris2),
                 meta=dict(b_plan=b_plan))


WORLDS_7S7 = [dict(b_react=br, b_max=bm) for br in (1.0, 1.4) for bm in (6.0, 6.5)]
LIB_7S7 = [OnHazard(EscapeAdjacent(2, A_lat=6.0, b_evasive=6.0, name="avoid_then_stop_behind_A")),
           OnHazard(SwerveReturn(LANE_W, A_lat=6.0, A_ret=6.0, margin=1.0, name="avoid_then_cut_back_to_middle"))]
df_7s7, tr_7s7 = rc07_scenario(
    "RC07-S7", build_rc07_s7, LIB_7S7, WORLDS_7S7, nominal=dict(b_react=1.0, b_max=6.0),
    witness="avoid_then_cut_back_to_middle",
    handback={"avoid_then_stop_behind_A": "the ego can stop behind A, but it takes B's stopping space and B cannot "
                                          "stop short enough"},
    contrast=("tyre fragments cover 35 m of the middle lane beyond the wheel (no way back before A)", dict(debris2=True)),
    sweep=dict(var="b_x0", values=np.arange(-44.0, -13.9, 2.0), xlabel="car B position relative to the ego at t0 [m]"),
    xlim=(-50, 130), title="position of car B",
    record=dict(site="straight 3-lane highway; ego in lane 1 (middle lane); a broken-down car A in the right lane",
                story="a wheel comes off the truck ahead and rolls into the middle lane; the left lane is full, so the ego "
                      "swerves right - toward broken-down car A, which it could still stop behind; but car B, coming up "
                      "the right lane faster, has already planned a firm stop behind A, and the ego now occupies that "
                      "space; the ego must cut back into the middle lane once past the wheel",
                t0_rule="t0 = the wheel comes off the truck; policies trigger on hazard_seen() (wheel deceleration)",
                carla_notes=["car A: a parked vehicle with hazard lights in the right lane",
                             "car B: constant speed, then the planned-stop law in actor_laws (b_plan in its note); it "
                             "brakes harder only after it notices the ego",
                             "wheel: a 0.8 m prop moved along scripted_trajectories"]))


### RC07 scenario gallery
The executed witness of every RC07 scenario in its nominal world. Each one completes the primary avoidance and then keeps
managing the secondary threat (return to the lane, keep moving, or recentre) instead of handing back.

In [ ]:
GALLERY7 = [(sid, trs[(HANDOFF[sid]["witness"], json.dumps(HANDOFF[sid]["nominal_world"], sort_keys=True))])
            for sid, trs in [("RC07-S1", tr_7s1), ("RC07-S2", tr_7s2), ("RC07-S3", tr_7s3), ("RC07-S4", tr_7s4), ("RC07-S5", tr_7s5), ("RC07-S6", tr_7s6), ("RC07-S7", tr_7s7)]]
for sid, tr in GALLERY7:
    tr.policy.name = f"{sid}: {tr.policy.inner.name}"
plot_traces([tr for _, tr in GALLERY7], "RC07 scenarios - executed witnesses (nominal worlds)", xlim=(-35, 220), height=1.9)
for sid, tr in GALLERY7:
    tr.policy.name = tr.policy.inner.name


---
## RC11 — Late evidence prevents robust causal recovery  (`U_INFORMATION` at the scope of a finite information tree)
**Card (recap).** Each hidden world is individually solvable, but the worlds need different decisions *before* the policy can
tell them apart. *CARLA construction:* keep dynamics and worlds fixed and delay only the permitted observation stream; use an
exactly enumerated finite information tree. *Witness / refutation:* per-world solution witnesses plus a proof that every shared
causal policy in the named class fails in at least one world. *Accept only when* no common stop or other robust prefix succeeds;
never describe zero robust success as "a collision in every world". *Paired contrast:* reveal the same evidence earlier → RC10
(a shared prefix, then a branch on delivered evidence, succeeds). **Required profile contrast:** every world has a witness, but no
shared causal policy passes within the proved class.

**How the scenarios implement it.**
* Two hidden **modes** per scenario (two CARLA episodes that are identical except for the mode).
* **Information tree** (the proved class): shared prefix brake ∈ {0, 3, 6} m/s² × commit time ∈ {0, 0.3, …, 1.2} s × default
  option (each side, stop, …), plus *wait-for-evidence* for each prefix. Every member switches to the right option the moment
  the evidence arrives — it is causal: it acts only on **delivered** observations.
* **Per-world witness** = the best member of the tree in that world (the clairvoyant choice).
* **Timing**: the measured last branch-response time per world and prefix vs the time the evidence arrives.
* **Leakage test**: every shared policy produces the same ego history in both worlds until its evidence arrives.
* **Clips**: in one world, the *same* prefix and commit time with the wrong vs the right default, each held once picked
  (the policies in the tree, by contrast, still switch when evidence arrives — too late); the hidden obstacle is
  **erased from the frames** until it is delivered to the ego, so nothing in the clip gives the answer away before the pick.

Two kinds of late evidence appear: an **information gate** (S1 low beams, S2 fog, S3 glare — the obstacle is there but not
delivered) and **late physical divergence** (S4 spin, S5 drowsy driver, S6 child — the worlds are physically identical until the
actor commits). For the second kind the RC10 contrast is the same divergence happening earlier.

### RC11 — toolkit and runner
* `SharedBranch` — a shared causal policy: prefix braking, then the option chosen by `evidence(o)` on delivered observations, or
  the default at the commit time. `information_tree(...)` enumerates the reference class.
* Information gates: `range_gate(names, d)` (headlights), `range_gate_all(d)` (fog), time gates (glare) via `Scene.visible`.
* Frame overlays (HighwayEnv draws everything): `hide_undelivered` erases actors that are not yet delivered to the ego,
  `night_overlay`, `fog_overlay`, `tunnel_overlay`, `label_overlay`. The simulator now logs the delivered set per frame.
* `rc11_scenario(...)` runs: tree × worlds → profile and per-world witnesses → timing table → the two clip policies →
  sweep of the information parameter (best wait-for-evidence vs per-world clairvoyant escape) → the card's checks → the RC10
  contrast → the CARLA hand-off (per mode: actors, scripted trajectories, witness and its trace).

In [ ]:
# ---------- RC11: shared causal policies, information gates and frame overlays ----------
NEVER = lambda o: None                                         # evidence function that never resolves the hidden mode


class SharedBranch(Policy):
    """Shared causal policy for a hidden-mode scene. Shared prefix: brake b_s in the current lane. As soon as
    `evidence(o)` resolves the hidden mode from *delivered* observations, switch to answer[mode]; if nothing has been
    resolved by t_commit, commit to `default`. options: name -> lateral target y (None = stay in the current lane) or
    'stop'. Escape: evasive lateral move (cap A_lat) with braking b_esc (if b_s > 0) capped by the friction circle.
    hold=True keeps a committed default even if evidence arrives later (used for the clips: "picks a track")."""
    def __init__(self, evidence, answer, options, b_s=0.0, t_commit=None, default=None, A_lat=6.0, b_esc=2.0,
                 mu_g=7.0, hold=False, name=None):
        self.evidence, self.answer, self.options = evidence, answer, options
        self.b_s, self.t_commit, self.default, self.A_lat, self.b_esc, self.mu = b_s, t_commit, default, A_lat, b_esc, 0.9 * mu_g
        self.hold = hold
        self.name = name or (f"wait_for_evidence(b={b_s:g})" if t_commit is None else
                             f"shared(b={b_s:g},commit={t_commit:g},default={default})")
        self.desc = dict(b_s=b_s, t_commit=t_commit, default=default, A_lat=A_lat, b_esc=b_esc, hold=hold,
                         options={k: v for k, v in options.items()}, answer=answer)
        self.reset()

    def reset(self):
        self.target, self.t_dec, self.t_evidence, self.y_home, self.log = None, None, None, None, []

    def act(self, o):
        e = o.ego
        if self.y_home is None:
            self.y_home = round(e["y"] / LANE_W) * LANE_W
        m = self.evidence(o)
        new = self.target
        if m is not None:
            self.t_evidence = o.t if self.t_evidence is None else self.t_evidence
            if not (self.hold and self.target is not None):
                new = self.answer[m]
        elif self.target is None and self.t_commit is not None and o.t >= self.t_commit - 1e-9:
            new = self.default
        if new != self.target:
            self.target, self.t_dec = new, o.t
            self.log.append((round(o.t, 2), f"-> {new}"))
        if self.target is None:
            return -self.b_s, track_y(e["x"], e["y"], e["psi"], e["v"], self.y_home)
        if self.options[self.target] == "stop":
            return -6.0, track_y(e["x"], e["y"], e["psi"], e["v"], self.y_home)
        y1 = self.options[self.target]
        y1 = self.y_home if y1 is None else y1
        if abs(y1 - e["y"]) > 1.5:
            steer = evade_y(e["y"], e["psi"], e["v"], y1, A_lat=self.A_lat)
        else:
            steer = track_y(e["x"], e["y"], e["psi"], e["v"], y1)
        grip = math.sqrt(max(self.mu ** 2 - lat_accel(e["v"], steer) ** 2, 0.0))
        return -min(self.b_esc if self.b_s > 0 else 0.0, grip), steer


def information_tree(evidence, answer, options, b_levels=(0.0, 3.0, 6.0), commits=(0.0, 0.3, 0.6, 0.9, 1.2)):
    """Finite reference class of shared causal policies: prefix braking x commit time x default option, plus
    wait-for-evidence (no default) for each prefix. Every member switches to the right option once evidence arrives."""
    tree = [SharedBranch(evidence, answer, options, b_s=b, t_commit=tc, default=d)
            for b in b_levels for tc in commits for d in options]
    tree += [SharedBranch(evidence, answer, options, b_s=b) for b in b_levels]
    return tree


def range_gate(names, d_vis):
    """Scene.visible rule: actors in `names` are delivered only when their nearest face is within d_vis m ahead of
    the ego front (headlight / fog visibility range); all other actors always."""
    def vis(t, n, sim):
        if n not in names:
            return True
        o, e = sim.objs[n], sim.ego
        return (o.position[0] - o.LENGTH / 2) - (e.position[0] + HALF_L) <= d_vis
    vis.needs_sim = True
    return vis


def range_gate_all(d_vis):
    """Fog: every actor is delivered only within d_vis m of the ego front."""
    def vis(t, n, sim):
        o, e = sim.objs[n], sim.ego
        return (o.position[0] - o.LENGTH / 2) - (e.position[0] + HALF_L) <= d_vis
    vis.needs_sim = True
    return vis


# ---------- frame overlays (HighwayEnv camera: ego-centred, 6 px/m, ego at 35 % of the width) ----------
def frame_overlay(draw, scaling=6.0, centre=(0.35, 0.5), size=(1000, 240)):
    """Wrap draw(d, px, py, t, k, tr, layer) into a Scene.meta['overlay'] function. k = index into tr.rec / tr.actors."""
    from PIL import Image, ImageDraw

    def f(tr, frames, every):
        idx = list(range(0, len(tr.rec["t"]), every))
        if len(idx) != len(frames):
            return frames
        out = []
        for fr, k in zip(frames, idx):
            ex, ey, t = tr.rec["x"][k], tr.rec["y"][k], tr.rec["t"][k] - tr.t0
            im = Image.fromarray(fr[..., :3]).convert("RGBA")
            ov = Image.new("RGBA", im.size, (0, 0, 0, 0))
            d = ImageDraw.Draw(ov)
            px = lambda x: centre[0] * size[0] + (x - ex) * scaling
            py = lambda y: centre[1] * size[1] + (y - ey) * scaling
            draw(d, px, py, t, k, tr, ov)
            out.append(np.asarray(Image.alpha_composite(im, ov).convert("RGB")))
        return out
    return f


def compose_overlays(*fs):
    def f(tr, frames, every):
        for g in fs:
            frames = g(tr, frames, every)
        return frames
    return f


def hide_undelivered(names, road=(100, 100, 100, 255)):
    """Erase actors in `names` from frames in which they are not delivered to the ego (no visual evidence)."""
    lens = {}

    def draw(d, px, py, t, k, tr, ov):
        if not lens:
            lens.update({a.name: (a.length, a.width) for a in tr.scene.actors})
        deliv = tr.delivered[k] if k < len(tr.delivered) else set()
        for n in names:
            if n in deliv or n not in tr.actors:
                continue
            _, x, y, psi, _ = tr.actors[n][k]
            L, W = lens[n]
            ex_, ey_ = extent(dict(length=L, width=W, psi=psi))
            d.rectangle([px(x - ex_ / 2) - 2, py(y - ey_ / 2) - 2, px(x + ex_ / 2) + 2, py(y + ey_ / 2) + 2], fill=road)
    return frame_overlay(draw)


def night_overlay(d_vis, beam_w0=3.0, beam_w1=14.0):
    """Darken the frame except the ego's low-beam footprint (d_vis m ahead)."""
    def draw(d, px, py, t, k, tr, ov):
        ex, ey = tr.rec["x"][k], tr.rec["y"][k]
        d.rectangle([0, 0, ov.size[0], ov.size[1]], fill=(5, 10, 30, 165))
        x0, x1 = ex + HALF_L, ex + HALF_L + d_vis
        d.polygon([(px(x0), py(ey - beam_w0 / 2)), (px(x1), py(ey - beam_w1 / 2)), (px(x1), py(ey + beam_w1 / 2)),
                   (px(x0), py(ey + beam_w0 / 2))], fill=(0, 0, 0, 0))
        d.text((8, 8), "night, unlit road - ego low beams", fill=(230, 230, 230, 255))
    return frame_overlay(draw)


def fog_overlay(V):
    """Fog: haze that thickens with distance from the ego, opaque beyond the visibility range V."""
    def draw(d, px, py, t, k, tr, ov):
        x_front = tr.rec["x"][k] + HALF_L
        for xs in np.arange(-60.0, 200.0, 2.0):                 # 2 m strips, haze alpha grows with distance
            a = int(min(235, 235 * (abs(xs + 1.0) / V) ** 2))
            d.rectangle([px(x_front + xs), 0, px(x_front + xs + 2.0), ov.size[1]], fill=(225, 225, 225, a))
        d.text((8, 8), f"fog: visibility {V:g} m", fill=(60, 60, 60, 255))
    return frame_overlay(draw)


def tunnel_overlay(x_portal, t_glare):
    """Tunnel interior (x < x_portal) drawn dark with walls; beyond the portal a white glare while the camera is
    still adapting (t < t_glare)."""
    def draw(d, px, py, t, k, tr, ov):
        top, bot = py(-LANE_W / 2), py((tr.scene.lanes - 0.5) * LANE_W)
        d.rectangle([0, top, px(x_portal), bot], fill=(0, 0, 0, 90))
        d.rectangle([0, top - 8, px(x_portal), top], fill=(40, 40, 40, 255))
        d.rectangle([0, bot, px(x_portal), bot + 8], fill=(40, 40, 40, 255))
        d.line([px(x_portal), top - 14, px(x_portal), bot + 14], fill=(255, 255, 255, 255), width=3)
        d.text((px(x_portal) - 70, top - 26), "TUNNEL EXIT", fill=(255, 255, 255, 255))
        if t < t_glare:
            d.rectangle([px(x_portal), 0, ov.size[0], ov.size[1]], fill=(255, 255, 245, 250))
            d.text((px(x_portal) + 10, 8), "camera saturated by glare", fill=(120, 120, 120, 255))
    return frame_overlay(draw)


def mark_actor(name, r_px=9, colour=(255, 150, 0, 255)):
    """Draw a ring around a small actor (e.g. a pedestrian) so it is visible at HighwayEnv's scale."""
    def draw(d, px, py, t, k, tr, ov):
        if name in tr.actors and name in tr.delivered[k]:
            _, x, y, _, _ = tr.actors[name][k]
            d.ellipse([px(x) - r_px, py(y) - r_px, px(x) + r_px, py(y) + r_px], outline=colour, width=2)
    return frame_overlay(draw)


def label_overlay(lines):
    def draw(d, px, py, t, k, tr, ov):
        for i, s in enumerate(lines):
            d.text((8, 8 + 14 * i), s, fill=(235, 235, 235, 255))
    return frame_overlay(draw)


print("RC11 toolkit ok")


In [ ]:
def last_branch_time(build, mode, option, answer, options, b_s, t_grid=np.arange(0.0, 2.01, 0.05)):
    """Latest commit time at which a blind shared prefix (brake b_s) followed by `option` still passes in `mode`."""
    ok = [float(t) for t in t_grid
          if simulate(build(mode=mode), SharedBranch(NEVER, answer, options, b_s=b_s, t_commit=float(t), default=option)).r >= EPS]
    return max(ok) if ok else None


def rc11_scenario(sid, build, modes, answer, options, evidence, info_var, info_early, nominal_mode, gif, sweep, xlim,
                  record, b_levels=(0.0, 3.0, 6.0), commits=(0.0, 0.3, 0.6, 0.9, 1.2)):
    """Run one RC11 scenario with the card's evidence pattern.
    modes: hidden modes (worlds differ only in `mode`); answer: mode -> option that survives it; options: option -> y.
    evidence(o): resolves the mode from delivered observations (None while ambiguous).
    info_var / info_early: the information parameter and the earlier-evidence value used for the RC10 contrast.
    gif: dict(b_s, t_commit[, wrong, mode, tag]) for a pair of clips (wrong vs right default, same world, commit before
         any evidence, held once picked) - or a list of such dicts for several worlds."""
    worlds = [dict(mode=m) for m in modes]
    tree = information_tree(evidence, answer, options, b_levels, commits)
    df, trs = evaluate(build, tree, worlds, keep=True)
    prof = robust_profile(df)
    best_world = df.loc[df.groupby("world")["r"].idxmax()].set_index("world")
    print(f"information tree: {len(tree)} shared causal policies x {len(worlds)} worlds; "
          f"best robust r = {prof.max():+.2f} ({prof.idxmax()})")
    display(pd.DataFrame({"robust r (min over worlds)": prof.sort_values(ascending=False).head(6).round(2)}))
    display(best_world[["policy", "r"]].rename(columns={"policy": "per-world witness (best in the tree)", "r": "r"}).round(2))

    # timing: last branch-response time vs evidence arrival, per prefix
    rows = []
    for b in (0.0, 6.0):
        t_last = {m: last_branch_time(build, m, answer[m], answer, options, b) for m in modes}
        t_ev = {}
        for m in modes:
            pol = SharedBranch(evidence, answer, options, b_s=b)
            simulate(build(mode=m), pol)
            t_ev[m] = pol.t_evidence
        rows.append({"prefix brake [m/s^2]": b, **{f"last branch time, {m} [s]": t_last[m] for m in modes},
                     **{f"evidence arrives, {m} [s]": (None if t_ev[m] is None else round(t_ev[m], 2)) for m in modes}})
    timing = pd.DataFrame(rows)
    display(timing)

    key = lambda p, m: (p, json.dumps(dict(mode=m), sort_keys=True))
    clips = []
    for g in (gif if isinstance(gif, list) else [gif]):
        m = g.get("mode", nominal_mode)
        wrong = g.get("wrong") or next(o for o in options if o != answer[m] and options[o] != "stop")
        p_fail = SharedBranch(evidence, answer, options, b_s=g["b_s"], t_commit=g["t_commit"], default=wrong, hold=True,
                              name=f"commits to {wrong} (wrong for mode {m})")
        p_ok = SharedBranch(evidence, answer, options, b_s=g["b_s"], t_commit=g["t_commit"], default=answer[m], hold=True,
                            name=f"commits to {answer[m]} (right for mode {m})")
        tr_fail, tr_ok = simulate(build(mode=m), p_fail), simulate(build(mode=m), p_ok)
        clips.append(dict(mode=m, tag=g.get("tag", ""), t_commit=g["t_commit"], fail=p_fail, ok=p_ok,
                          tr_fail=tr_fail, tr_ok=tr_ok))
        plot_traces([tr_fail, tr_ok], f"{sid} mode {m}: the same shared prefix, two blind commitments", xlim=xlim)

    var, xs = sweep["var"], np.asarray(sweep["values"], dtype=float)
    waits = [SharedBranch(evidence, answer, options, b_s=b) for b in b_levels]
    r_wait = [max(min(simulate(build(mode=m, **{var: x}), w).r for m in modes) for w in waits) for x in xs]
    r_clair = [min(max(simulate(build(mode=m, **{var: x}),
                                SharedBranch(NEVER, answer, options, b_s=b, t_commit=0.0, default=answer[m])).r
                       for b in (0.0, 6.0)) for m in modes) for x in xs]
    plot_sweep(xs, {"best wait-for-evidence (shared, causal)": r_wait, "per-world clairvoyant escape (min over worlds)": r_clair},
               sweep["xlabel"], f"{sid}: {sweep['title']}")
    late = [float(x) for x, a, c in zip(xs, r_wait, r_clair) if a < EPS and c >= EPS]
    print(f"{var} values where every world is solvable but waiting for evidence fails: "
          + (f"{late[0]} .. {late[-1]}" if late else "none"))

    # checks (card RC11)
    check(sid, "every world has a witness in the tree (per-world clairvoyant success)", bool((best_world["r"] >= EPS).all()))
    check(sid, f"no shared causal policy in the enumerated class ({len(tree)}) passes every world", bool((prof < EPS).all()))
    stop_r = min(simulate(build(mode=m), SharedBranch(NEVER, answer, options, b_s=6.0)).r for m in modes)
    check(sid, "a common stop is not a robust prefix (else the case is not information-infeasible)", stop_r < EPS)
    ev_late = all(any(r[f"evidence arrives, {m} [s]"] is None or r[f"last branch time, {m} [s]"] is None
                      or r[f"evidence arrives, {m} [s]"] > r[f"last branch time, {m} [s]"] for m in modes) for r in rows)
    check(sid, "for every prefix, some world's evidence arrives after that world's last branch-response time", ev_late)
    same = True
    for p in tree:
        t_cut = min([t for t in [simulate_t_ev(build, p, m) for m in modes] if t is not None], default=None)
        ra, rb = trs[key(p.name, modes[0])].rec, trs[key(p.name, modes[1])].rec
        n = min(len(ra["t"]), len(rb["t"]))
        if t_cut is not None:
            n = min(n, int(np.searchsorted(ra["t"] - trs[key(p.name, modes[0])].t0, t_cut)))
        same &= bool(np.allclose(ra["x"][:n], rb["x"][:n]) and np.allclose(ra["y"][:n], rb["y"][:n]))
    check(sid, "every shared policy produces identical ego histories in all worlds until evidence arrives (no leakage)", same)
    early = [max(min(simulate(build(mode=m, **{var: info_early}), w).r for m in modes) for w in waits)][0]
    print(f"paired contrast (RC10): {var} = {info_early} -> best wait-for-evidence robust r = {early:+.2f}")
    check(sid, f"paired contrast: the same evidence earlier ({var} = {info_early}) recovers RC10 (R_REF)", early >= EPS)
    check(sid, "clips: in each clip world the wrong commitment fails and the right one survives, before any evidence",
          all(c["tr_fail"].r < EPS and c["tr_ok"].r >= EPS
              and (c["fail"].t_evidence is None or c["fail"].t_evidence > c["t_commit"]) for c in clips))

    u_info = bool((best_world["r"] >= EPS).all() and (prof < EPS).all() and stop_r < EPS)
    per_mode = {}
    for m in modes:
        tw = trs[key(best_world.loc[json.dumps(dict(mode=m), sort_keys=True), "policy"], m)]
        sc = tw.scene
        per_mode[m] = {"actors": actors_rel(sc, tw), "scripted_trajectories": path_keyframes(sc, tw),
                       "scripted_params": {a.name: a.law.params for a in sc.actors if a.kind == "scripted"},
                       "actor_laws": {a.name: a.note for a in sc.actors}, "world": sc.world,
                       "witness": tw.policy.name, "witness_descriptor": tw.policy.descriptor(),
                       "witness_trace": ego_keyframes(tw)}
    sc0 = clips[0]["tr_ok"].scene
    handoff(sid, parent="RC11", label=("U_INFORMATION" if u_info else "UNRESOLVED"), **record,
            scope=f"finite information tree of {len(tree)} shared causal policies (prefix brake {list(b_levels)} x "
                  f"commit {list(commits)} s x default {list(options)}, plus wait-for-evidence); discrete class only",
            evidence="HighwayEnv reference dynamics; not CARLA-validated; zero robust success != collision in every world",
            ego={"v": sc0.ego_speed, "lane": sc0.ego_lane, "envelope": asdict(sc0.env)},
            road={"lanes": sc0.lanes, "lane_width": LANE_W, "drivable_y": list(sc0.bounds)},
            modes=list(modes), options={k: v for k, v in options.items()}, answer=answer, per_mode=per_mode,
            timing=timing.to_dict("records"), best_shared_policy={"name": prof.idxmax(), "robust_r": round(float(prof.max()), 3)},
            sweep={"var": var, "values": xs.tolist(), "r_best_wait": r_wait, "r_clairvoyant_min": r_clair,
                   "information_infeasible_range": [late[0], late[-1]] if late else None},
            paired_contrast={"change": f"{var} = {info_early} (evidence earlier, same worlds)", "kwargs": {var: info_early},
                             "best_wait_robust_r": round(early, 3), "label": "R_REF" if early >= EPS else "UNRESOLVED"},
            clips=[{"world": dict(mode=c["mode"]), "fail": c["fail"].descriptor(), "survive": c["ok"].descriptor()}
                   for c in clips])
    return df, clips


def simulate_t_ev(build, pol, mode):
    """Time the policy first receives resolving evidence in `mode` (None if never)."""
    simulate(build(mode=mode), pol)
    return pol.t_evidence


print("RC11 scenario runner ok")


### RC11-S1 — Night crash: the second car is unlit
**Story.** At night on an unlit 3-lane motorway the ego drives at 25 m/s (90 km/h) in the middle lane. A two-car crash has just
happened: **car 1** stands in the middle lane with its **hazard lights on** (visible from far), 38 m ahead — too close to stop.
**Car 2**, its lights smashed, came to rest 8 m further on in **either the left or the right lane**. It shows up only inside the
ego's **low-beam range (25 m)** — about 0.9 s after t0, while the last moment to choose a side is 0.55–0.8 s.

**Modes:** L (car 2 in the left lane → go right) / R (→ go left). **Stopping** fails in both.
**Clips (mode L):** brake, then commit at 0.6 s — to the left (car 2 appears in the beams, too late) vs to the right.
**Paired contrast (RC10):** street lights / high beams: car 2 visible from 60 m → brake, see, branch.
**Sweep:** the distance at which the unlit car becomes visible.

In [ ]:
def build_rc11_s1(mode="L", v=25.0, D=38.0, gap2=8.0, d_vis=25.0, T=6.0):
    """Night, unlit 3-lane motorway, ego in the middle lane (lane 1) at v. A two-car crash: car 1 has stopped in the
    middle lane with its hazard lights on (visible from far), D m ahead. Car 2, its lights smashed, came to rest gap2 m
    further on in the left lane (mode 'L') or the right lane (mode 'R'); it is delivered to the ego only once it is
    within the low-beam range d_vis. Both modes are identical until then."""
    x1 = HALF_L + D + 2.4
    acts = [Actor("car1", x=x1, lane=1, length=4.8, kind="static", note=f"crashed car 1, hazard lights on, {D} m ahead"),
            Actor("car2", x=x1 + gap2, lane=0 if mode == "L" else 2, length=4.8, kind="static",
                  note=f"crashed car 2, unlit, {'left' if mode == 'L' else 'right'} lane, {gap2} m beyond car 1; "
                       f"visible only within {d_vis} m (low beams)")]
    return Scene("RC11-S1", lanes=3, ego_lane=1, ego_speed=v, ego_x=0.0, actors=acts, T=T, T_term=2.0,
                 visible=range_gate({"car2"}, d_vis),
                 world=dict(mode=mode, v=v, D=D, gap2=gap2, d_vis=d_vis),
                 meta=dict(overlay=compose_overlays(hide_undelivered(["car2"]), night_overlay(d_vis))))


def evidence_s1(o):
    c = o.actors.get("car2")
    return None if c is None else ("L" if c["y"] < LANE_W else "R")


OPTIONS_11S1 = {"left": 0.0, "right": 2 * LANE_W, "stop": "stop"}
ANSWER_11S1 = {"L": "right", "R": "left"}
df_11s1, clips_11s1 = rc11_scenario(
    "RC11-S1", build_rc11_s1, ("L", "R"), ANSWER_11S1, OPTIONS_11S1, evidence_s1,
    info_var="d_vis", info_early=60.0, nominal_mode="L", gif=dict(b_s=6.0, t_commit=0.6),
    sweep=dict(var="d_vis", values=np.arange(15.0, 70.1, 5.0), xlabel="distance at which the unlit car becomes visible [m]",
               title="visibility of the unlit car"),
    xlim=(-10, 120),
    record=dict(site="unlit straight 3-lane motorway at night; ego in lane 1 (middle lane), low beams",
                story="a two-car crash ahead: car 1 stands in the middle lane with hazard lights on, too close to stop "
                      "for; car 2, its lights smashed, lies in the left or the right lane just beyond and becomes visible "
                      "only inside the low-beam range - after the last moment to choose a side",
                information_contract="car2 is delivered to the ego only when its rear face is within d_vis m of the ego "
                                     "front (low beams); car1 is always delivered (hazard lights)",
                t0_rule="t0 = scenario start: ego at x = 0, car 1 already in view",
                carla_notes=["night weather preset (sun altitude < 0), no street lights on the chosen stretch",
                             "car2: vehicle with lights off; with camera/lidar the low-beam range emerges naturally - "
                             "with ground-truth detection gate car2 by d_vis",
                             "car1: hazard lights on (vehicle light state)",
                             "run each mode (per_mode) as a separate episode with identical setup otherwise"]))


### RC11-S2 — Fog pile-up: left lane or right lane?
**Story.** Dense fog (visibility 35 m) on a 3-lane motorway; the ego drives at 25 m/s in the middle lane. **Car 1 emerges from the
fog** stopped in the middle lane — too close to stop. **Car 2**, which swerved around it, stopped 18 m further on, **either in the
left or in the right lane** — and comes out of the fog (≈0.75 s) only after the last moment to choose a side (0.4–0.6 s).

**Modes:** L (car 2 in the left lane → go right) / R (car 2 in the right lane → go left).
**Clips (mode R):** brake, then commit at 0.4 s — to the right (into car 2, which appears out of the fog) vs to the left.
**Paired contrast (RC10):** visibility 80 m.
**Sweep:** fog visibility.

In [ ]:
def build_rc11_s2(mode="R", v=25.0, V=35.0, gap2=18.0, T=6.0):
    """Dense fog (visibility V) on a 3-lane motorway; ego in the middle lane (lane 1) at v. A pile-up: car 1 has stopped in
    the middle lane and comes out of the fog at t0 (its rear exactly V m ahead). Car 2 stopped gap2 m further on, either
    in the left lane (mode 'L') or in the right lane (mode 'R'). Every object is delivered to the ego only within V m."""
    x1 = HALF_L + V + 2.4
    acts = [Actor("car1", x=x1, lane=1, length=4.8, kind="static", note="pile-up car 1, stopped in the middle lane"),
            Actor("car2", x=x1 + gap2, lane=0 if mode == "L" else 2, length=4.8, kind="static",
                  note=f"pile-up car 2, stopped {gap2} m beyond car 1 in the {'left' if mode == 'L' else 'right'} lane")]
    return Scene("RC11-S2", lanes=3, ego_lane=1, ego_speed=v, ego_x=0.0, actors=acts, T=T, T_term=2.0,
                 visible=range_gate_all(V), world=dict(mode=mode, v=v, V=V, gap2=gap2),
                 meta=dict(overlay=compose_overlays(hide_undelivered(["car1", "car2"]), fog_overlay(V))))


def evidence_s2(o):
    c = o.actors.get("car2")
    return None if c is None else ("L" if c["y"] < LANE_W else "R")


OPTIONS_11S2 = {"left": 0.0, "right": 2 * LANE_W, "stop": "stop"}
ANSWER_11S2 = {"L": "right", "R": "left"}
df_11s2, clips_11s2 = rc11_scenario(
    "RC11-S2", build_rc11_s2, ("L", "R"), ANSWER_11S2, OPTIONS_11S2, evidence_s2,
    info_var="V", info_early=80.0, nominal_mode="R", gif=dict(b_s=6.0, t_commit=0.4),
    sweep=dict(var="V", values=np.arange(25.0, 85.1, 5.0), xlabel="fog visibility [m]", title="fog visibility"),
    xlim=(-10, 120),
    record=dict(site="straight 3-lane motorway in dense fog; ego in lane 1 (middle lane)",
                story="a pile-up in fog: car 1 emerges from the fog stopped in the middle lane, too close to stop for; car "
                      "2, which swerved around it, stopped further on in either the left or the right lane, and comes out "
                      "of the fog only after the last moment to choose a side",
                information_contract="every actor is delivered to the ego only when its rear face is within V m of the "
                                     "ego front (fog visibility)",
                t0_rule="t0 = car 1 emerges from the fog (its rear exactly V m ahead)",
                carla_notes=["fog via WeatherParameters (fog_density / fog_distance) tuned so a car is detectable at V m; "
                             "with ground-truth detection gate every actor by V",
                             "run each mode (per_mode) as a separate episode"]))


### RC11-S3 — Tunnel exit glare
**Story.** The ego drives at 25 m/s in the middle lane of a 3-lane motorway **tunnel**. Just inside the exit portal, **car 1** has
stopped in the middle lane (lit by the tunnel lights, 38 m ahead). **Car 2** stands just outside, in bright sunlight, in **either
the left or the right lane** — but the ego's camera, exposed for the dark tunnel, is **saturated by the glare** for 1.0 s, after
the last moment to choose a side (0.55–0.8 s).

**Modes:** L (→ go right) / R (→ go left). **Clips (mode R):** brake, then commit at 0.6 s — left vs right; the glare is drawn
white beyond the portal until the camera adapts.
**Paired contrast (RC10):** an HDR camera that adapts in 0.2 s.
**Sweep:** glare / exposure-adaptation time.

In [ ]:
def build_rc11_s3(mode="L", v=25.0, D=38.0, gap2=9.0, t_glare=1.0, T=6.0):
    """3-lane motorway tunnel, ego in the middle lane (lane 1) at v. Just inside the exit portal, car 1 has stopped in the
    middle lane (lit by the tunnel lighting, D m ahead). Car 2 stopped gap2 m further on, just outside the portal in
    bright sunlight, in the left (mode 'L') or the right lane (mode 'R'). The ego's camera, exposed for the tunnel,
    is saturated by the glare beyond the portal: car 2 is delivered only from t_glare (exposure adaptation time)."""
    x1 = HALF_L + D + 2.4
    x_portal = x1 + 2.4 + 3.0
    acts = [Actor("car1", x=x1, lane=1, length=4.8, kind="static", note=f"car 1 stopped in the middle lane inside the tunnel, {D} m ahead"),
            Actor("car2", x=x1 + gap2, lane=0 if mode == "L" else 2, length=4.8, kind="static",
                  note=f"car 2 stopped just outside the portal in the {'left' if mode == 'L' else 'right'} lane; "
                       f"hidden by glare until t0+{t_glare} s")]
    return Scene("RC11-S3", lanes=3, ego_lane=1, ego_speed=v, ego_x=0.0, actors=acts, T=T, T_term=2.0,
                 visible=lambda t, n: n != "car2" or t >= t_glare - 1e-9,
                 world=dict(mode=mode, v=v, D=D, gap2=gap2, t_glare=t_glare),
                 meta=dict(x_portal=x_portal, overlay=compose_overlays(hide_undelivered(["car2"]), tunnel_overlay(x_portal, t_glare))))


def evidence_s3(o):
    c = o.actors.get("car2")
    return None if c is None else ("L" if c["y"] < LANE_W else "R")


OPTIONS_11S3 = {"left": 0.0, "right": 2 * LANE_W, "stop": "stop"}
ANSWER_11S3 = {"L": "right", "R": "left"}
df_11s3, clips_11s3 = rc11_scenario(
    "RC11-S3", build_rc11_s3, ("L", "R"), ANSWER_11S3, OPTIONS_11S3, evidence_s3,
    info_var="t_glare", info_early=0.2, nominal_mode="R", gif=dict(b_s=6.0, t_commit=0.6),
    sweep=dict(var="t_glare", values=np.arange(0.0, 1.61, 0.1), xlabel="camera glare / exposure adaptation time [s]",
               title="glare duration at the tunnel exit"),
    xlim=(-10, 120),
    record=dict(site="3-lane motorway tunnel exit into bright sunlight; ego in lane 1 (middle lane)",
                story="a crash at a tunnel exit: car 1 has stopped in the middle lane just inside the portal; car 2 stands "
                      "just outside in the left or the right lane, but the ego's camera, exposed for the tunnel, is "
                      "blinded by the glare and sees it only after the last moment to choose a side",
                information_contract="car2 (beyond the portal) is delivered only from t_glare s after t0 (exposure "
                                     "adaptation); car1 inside the tunnel is always delivered",
                t0_rule="t0 = scenario start: ego at x = 0, car 1 in view inside the tunnel",
                carla_notes=["needs a map with a tunnel exit (or a covered section) and a high sun facing the exit",
                             "camera-based perception reproduces the glare only with an LDR camera / auto-exposure; with "
                             "ground-truth detection gate car2 until t_glare",
                             "run each mode (per_mode) as a separate episode"]))


### RC11-S4 — The car ahead spins: which way will it slide?
**Story.** The ego follows a car at only 10 m (0.4 s) at 25 m/s in the middle lane. That car **blows a rear tyre**: it skids at
10 m/s², starts to yaw and **slides straight along its lane for 1.4 s** — identically in both worlds — before it slides to rest
across the middle lane and **either the left or the right lane**. The ego is too close to stop and must pick a side before the
slide direction shows.

**Modes:** L (→ go right) / R (→ go left). **Clips (mode L):** brake, then commit at 0.6 s — left vs right.
**Paired contrast (RC10):** the same slide shows its direction after 0.4 s.
**Sweep:** time until the slide direction shows.

In [ ]:
def spin_path(x0, y0, v, b, t_div, T_slide, y_rest, psi_common, psi_rest):
    """Car losing control: brakes/skids at b to rest; yaws to psi_common along its lane until t_div (identical for
    every outcome), then slides laterally to y_rest while rotating to psi_rest over T_slide."""
    t_s = v / b

    def f(t):
        tt = min(max(t, 0.0), t_s)
        x, vx = x0 + v * tt - 0.5 * b * tt * tt, max(v - b * max(t, 0.0), 0.0)
        u = min(max((t - t_div) / T_slide, 0.0), 1.0)
        y = y0 + (y_rest - y0) * quintic(u)
        vy = (y_rest - y0) * 30 * u * u * (1 - u) ** 2 / T_slide
        psi = psi_common * min(max(t, 0.0) / t_div, 1.0) + (psi_rest - psi_common) * quintic(u)
        return x, y, psi, vx, vy
    f.params = dict(x0=x0, y0=y0, v=v, b=b, t_div=t_div, T_slide=T_slide, y_rest=y_rest, psi_common=psi_common, psi_rest=psi_rest)
    return f


def build_rc11_s4(mode="L", v=25.0, gap=10.0, b_spin=10.0, t_div=1.4, T=6.0):
    """3-lane motorway, ego in the middle lane (lane 1) at v, following a car at a gap of `gap` m (bumper). At t0 that
    car blows a rear tyre: it skids at b_spin and starts to yaw, sliding straight along its lane until t_div - identical
    in both modes - then slides to rest across the middle and the left lane (mode 'L') or the right lane (mode 'R')."""
    x0 = HALF_L + gap + 2.4
    y_rest, psi_rest = (LANE_W - 2.6, 1.3) if mode == "L" else (LANE_W + 2.6, -1.3)
    acts = [Actor("spin_car", x=0, lane=1, length=4.8, kind="scripted",
                  law=spin_path(x0, LANE_W, v, b_spin, t_div, 1.2, y_rest, 0.25, psi_rest),
                  note=f"lead car, tyre blowout at t0: skids at {b_spin} m/s^2, yaws 0.25 rad along its lane until "
                       f"t0+{t_div} s, then slides to rest across the middle and {'left' if mode == 'L' else 'right'} lane")]
    return Scene("RC11-S4", lanes=3, ego_lane=1, ego_speed=v, ego_x=0.0, actors=acts, T=T, T_term=2.0,
                 world=dict(mode=mode, v=v, gap=gap, b_spin=b_spin, t_div=t_div),
                 meta=dict(overlay=label_overlay(["lead car: tyre blowout, spinning"])))


def evidence_s4(o):
    c = o.actors.get("spin_car")
    if c is None or abs(c["y"] - LANE_W) < 0.3:
        return None
    return "L" if c["y"] < LANE_W else "R"


OPTIONS_11S4 = {"left": 0.0, "right": 2 * LANE_W, "stop": "stop"}
ANSWER_11S4 = {"L": "right", "R": "left"}
df_11s4, clips_11s4 = rc11_scenario(
    "RC11-S4", build_rc11_s4, ("L", "R"), ANSWER_11S4, OPTIONS_11S4, evidence_s4,
    info_var="t_div", info_early=0.4, nominal_mode="L", gif=dict(b_s=6.0, t_commit=0.6),
    sweep=dict(var="t_div", values=np.arange(0.2, 1.81, 0.2), xlabel="time until the spinning car's slide direction shows [s]",
               title="when the slide direction becomes visible"),
    xlim=(-10, 90),
    record=dict(site="straight 3-lane motorway; ego in lane 1 (middle lane) following a car at 10 m (0.4 s)",
                story="the car ahead blows a rear tyre and spins; it first skids straight along its lane while it starts "
                      "to yaw, and only then slides to rest across the middle lane and either the left or the right lane; "
                      "the ego is too close to stop and must pick a side before the slide direction shows",
                information_contract="the spinning car is always delivered; the two modes are physically identical until "
                                     "t_div, so the evidence is the car's own lateral motion (|dy| > 0.3 m)",
                t0_rule="t0 = the tyre blows (the car ahead starts to skid)",
                carla_notes=["replay spin_car from scripted_trajectories (pose + yaw per mode); a physics-driven spin is "
                             "not repeatable enough for a paired evaluation",
                             "the paired contrast (t_div earlier) is the same slide revealed earlier, not a different car",
                             "run each mode (per_mode) as a separate episode"]))


### RC11-S5 — Drowsy oncoming driver: will they wake up?
**Story.** On a rural two-way road (22 m/s each way) an **oncoming driver has fallen asleep** and drifts across the centre line,
90 m ahead. At 1.7 s **either they wake and jerk back** into their own lane (mode W: the ego should brake hard and stay in its
lane — swerving into the oncoming lane would meet them there) **or they stay asleep** and the drift steepens until their car
straddles the ego lane's right edge and the verge (mode D: braking is useless and only the vacated oncoming lane is free).

**Clips — both worlds, same prefix (brake 6 m/s²) and commit time (0.9 s):**
* *keeps drifting* (mode D): braking to a stop in lane → hit; swerving into the vacated oncoming lane → safe.
* *wakes up* (mode W): swerving into the oncoming lane → meets the driver steering back; braking in lane → safe.
The right action in one world is exactly the wrong one in the other.
**Paired contrast (RC10):** the driver wakes (or the drift steepens) at 0.8 s.
**Sweep:** when the driver's outcome shows. The information-infeasible band is narrow here (≈1.6–1.7 s): earlier, waiting
works; much later, the wake-up world itself no longer has a solution.

In [ ]:
def drift_path(x0, v, y0, vy, t_wake, mode, T_corr=0.9, vy_sleep=2.6, y_end=6.0):
    """Oncoming car (heading -x) drifting across the centre line at vy. Mode 'W': the driver wakes at t_wake and steers
    back to the centre of its own lane (y = 0) over T_corr. Mode 'D': the driver stays asleep; from t_wake the drift
    steepens to vy_sleep until y_end (straddling the ego lane's edge and the verge). Identical until t_wake."""
    y_w = y0 + vy * t_wake

    def f(t):
        x = x0 - v * t
        if t <= t_wake:
            y, vyy = y0 + vy * t, vy
        elif mode == "W":
            u = min((t - t_wake) / T_corr, 1.0)
            y, vyy = y_w + (0.0 - y_w) * quintic(u), (0.0 - y_w) * 30 * u * u * (1 - u) ** 2 / T_corr
        else:
            y = min(y_w + vy_sleep * (t - t_wake), y_end)
            vyy = vy_sleep if y < y_end else 0.0
        return x, y, math.atan2(vyy, -v), -v, vyy
    f.params = dict(x0=x0, v=v, y0=y0, vy=vy, t_wake=t_wake, mode=mode, T_corr=T_corr, vy_sleep=vy_sleep, y_end=y_end)
    return f


def build_rc11_s5(mode="W", v=22.0, v_onc=22.0, x_onc=90.0, y0=1.0, vy=1.6, t_wake=1.7, verge=2.5, T=6.0):
    """Rural two-lane two-way road (lane 1 = ego direction, lane 0 = opposing) with a 2.5 m gravel verge on the ego's
    right; ego at v. At t0 an oncoming car (centre x_onc, v_onc) is drifting across the centre line - its driver has
    fallen asleep. Mode 'W': the driver wakes at t_wake and jerks back into the own lane; mode 'D': the drift steepens
    and the car ends up straddling the ego lane's right edge and the verge. Identical until t_wake."""
    acts = [Actor("oncoming", x=0, lane=0, length=4.8, kind="scripted",
                  law=drift_path(x_onc, v_onc, y0, vy, t_wake, mode),
                  note=f"oncoming car {v_onc} m/s drifting toward the ego lane at {vy} m/s; "
                       + ("the driver wakes at t0+%.1f s and steers back" % t_wake if mode == "W"
                          else "the driver stays asleep; from t0+%.1f s the drift steepens across the ego lane to its edge" % t_wake))]
    return Scene("RC11-S5", lanes=2, ego_lane=1, ego_speed=v, ego_x=0.0, actors=acts, T=T, T_term=2.0,
                 road_bounds=(-LANE_W / 2, 1.5 * LANE_W + verge),
                 world=dict(mode=mode, v=v, v_onc=v_onc, x_onc=x_onc, y0=y0, vy=vy, t_wake=t_wake, verge=verge),
                 meta=dict(overlay=label_overlay(["rural two-way road: lane 0 = oncoming direction",
                                                  "oncoming driver asleep, drifting across the centre line"])))


def evidence_s5(o):
    if len(o.history) < 2:
        return None
    (t1, s1), (t2, s2) = o.history[-2], o.history[-1]
    a, c = s1.get("oncoming"), s2.get("oncoming")
    if a is None or c is None:
        return None
    vy = (c["y"] - a["y"]) / (t2 - t1)
    return "W" if vy < 0.0 else ("D" if vy > 2.2 else None)


OPTIONS_11S5 = {"keep": None, "left": 0.0, "right": 1.5 * LANE_W + 1.25, "stop": "stop"}
ANSWER_11S5 = {"W": "stop", "D": "left"}
df_11s5, clips_11s5 = rc11_scenario(
    "RC11-S5", build_rc11_s5, ("W", "D"), ANSWER_11S5, OPTIONS_11S5, evidence_s5,
    info_var="t_wake", info_early=0.8, nominal_mode="D",
    gif=[dict(mode="D", b_s=6.0, t_commit=0.9, wrong="stop", tag="drifts_"),
         dict(mode="W", b_s=6.0, t_commit=0.9, wrong="left", tag="wakes_")],
    sweep=dict(var="t_wake", values=np.arange(0.6, 2.21, 0.2), xlabel="time at which the drowsy driver's outcome shows [s]",
               title="when the drowsy driver wakes (or does not)"),
    xlim=(-10, 110),
    record=dict(site="rural two-lane two-way road with a 2.5 m gravel verge on the ego's right; ego in lane 1",
                story="an oncoming driver has fallen asleep and drifts across the centre line; either they wake and jerk "
                      "back (the ego should brake hard and stay in lane) or they keep drifting until the car straddles "
                      "the ego lane's edge (only the vacated oncoming lane is free); the two cannot be told apart until "
                      "the driver wakes - or doesn't",
                information_contract="the oncoming car is always delivered; the two modes are physically identical until "
                                     "t_wake, so the evidence is its lateral velocity (steering back vs drifting faster)",
                t0_rule="t0 = scenario start: the oncoming car is already over the centre line, 90 m ahead",
                carla_notes=["replay the oncoming car from scripted_trajectories (heading ~pi, per mode)",
                             "the verge must be drivable but is not a refuge here (the drifting car ends on it)",
                             "run each mode (per_mode) as a separate episode"]))


### RC11-S6 — Child between parked cars: freeze or sprint?
**Story.** The ego drives at 50 km/h (14 m/s) in the right traffic lane of a one-way street with a parking lane full of cars. A
**child runs out between two parked cars**, 15 m ahead. Reacting to the car, the child **either freezes** in the ego lane (mode F:
the ego must swerve into the left lane) **or sprints on** across the street (mode C: the ego must brake in its lane — swerving
left would meet the child there). Which one only shows once the child is already in the ego lane.

**Clips (mode F):** brake 3 m/s², then commit at 0.3 s — stay in the lane (moderate braking) vs swerve left. Braking
hard in the lane fails too, but only just: the ego stops within centimetres of the frozen child, inside the clearance margin;
a late swerve once the child freezes also misses only narrowly, inside the margin.
**Paired contrast (RC10):** the child reacts as soon as it steps out.
**Sweep:** where the child reacts (larger = earlier).

In [ ]:
def child_path(x, y0, v_run, mode, y_slow=LANE_W + 1.0, y_end=-LANE_W / 2 - 1.5, b=12.0, v_sprint=5.5, a_sprint=8.0):
    """Child running out across the road (toward -y) at v_run. At y_slow it reacts to the car: mode 'F' stops hard (b)
    and freezes in the ego lane; mode 'C' sprints on (to v_sprint) to the far side (y_end). Identical until y_slow."""
    t_slow = (y0 - y_slow) / v_run
    t_stop, t_acc = v_run / b, (v_sprint - v_run) / a_sprint

    def f(t):
        if t <= t_slow:
            y, vy = y0 - v_run * max(t, 0.0), (-v_run if t >= 0 else 0.0)
        elif mode == "F":
            tt = min(t - t_slow, t_stop)
            y, vy = y_slow - v_run * tt + 0.5 * b * tt * tt, -(v_run - b * tt)
        else:
            tt = t - t_slow
            ta = min(tt, t_acc)
            y = y_slow - v_run * ta - 0.5 * a_sprint * ta * ta - v_sprint * max(tt - t_acc, 0.0)
            vy = -(v_run + a_sprint * ta)
            if y <= y_end:
                y, vy = y_end, 0.0
        return x, y, -math.pi / 2, 0.0, vy
    f.params = dict(x=x, y0=y0, v_run=v_run, mode=mode, y_slow=y_slow, y_end=y_end, b=b, v_sprint=v_sprint, a_sprint=a_sprint)
    return f


def build_rc11_s6(mode="F", v=14.0, d_c=15.0, v_run=3.5, y_slow=LANE_W + 1.0, T=6.0):
    """Urban one-way street with two traffic lanes (lane 0, lane 1 = ego, 50 km/h) and a parking lane (lane 2) full of
    parked cars. At t0 a child runs out from between two parked cars, d_c m ahead. Mode 'F': seeing the car, the child
    stops dead in the ego lane; mode 'C': the child sprints on across the street. Identical until the child reaches
    y_slow (where it reacts to the car)."""
    x_c = HALF_L + d_c + 0.25
    acts = [Actor("child", x=0, y=2 * LANE_W - 1.0, length=0.5, width=0.5, kind="scripted",
                  law=child_path(x_c, 2 * LANE_W - 1.0, v_run, mode, y_slow=y_slow),
                  note=f"child running out from between parked cars {d_c} m ahead at {v_run} m/s; "
                       + ("stops dead in the ego lane" if mode == "F" else "sprints on across the street"))]
    acts += [Actor(f"parked_{i}", x=x_c + dx, lane=2, length=4.6, kind="static", note="parked car")
             for i, dx in enumerate((-33.0, -27.0, -21.0, -15.0, -9.0, -3.5, 3.5, 9.0, 15.0, 21.0, 27.0, 33.0, 39.0))]
    return Scene("RC11-S6", lanes=3, ego_lane=1, ego_speed=v, ego_x=0.0, actors=acts, T=T, T_term=2.0,
                 road_bounds=(-LANE_W / 2, 1.5 * LANE_W),
                 world=dict(mode=mode, v=v, d_c=d_c, v_run=v_run, y_slow=y_slow),
                 meta=dict(overlay=compose_overlays(mark_actor("child"),
                                                    label_overlay(["one-way street, 50 km/h; lane 2 = parking lane",
                                                                   "child (ringed) runs out between parked cars"]))))


def evidence_s6(o):
    """F: the child decelerates (|vy| < 2 m/s after it has started running); C: it sprints (|vy| > 4.2 m/s)."""
    if len(o.history) < 2:
        return None
    a, c = o.history[-2][1].get("child"), o.history[-1][1].get("child")
    if a is None or c is None or c["y"] > 2 * LANE_W - 1.3:
        return None
    vy = (c["y"] - a["y"]) / (o.history[-1][0] - o.history[-2][0])
    return "F" if vy > -2.0 else ("C" if vy < -4.2 else None)


OPTIONS_11S6 = {"left": 0.0, "stop": "stop", "keep": None}
ANSWER_11S6 = {"F": "left", "C": "stop"}
df_11s6, clips_11s6 = rc11_scenario(
    "RC11-S6", build_rc11_s6, ("F", "C"), ANSWER_11S6, OPTIONS_11S6, evidence_s6,
    info_var="y_slow", info_early=2 * LANE_W - 1.2, nominal_mode="F", gif=dict(b_s=3.0, t_commit=0.3, wrong="keep"),
    sweep=dict(var="y_slow", values=np.arange(LANE_W + 0.4, 2 * LANE_W - 1.09, 0.4),
               xlabel="lateral position where the child reacts [m] (larger = earlier)", title="where the child reacts"),
    xlim=(-10, 60),
    record=dict(site="urban one-way street, two traffic lanes (50 km/h) and a parking lane full of parked cars; ego in "
                     "lane 1 (right traffic lane)",
                story="a child runs out from between parked cars 15 m ahead; reacting to the car it either freezes in the "
                      "ego lane (the ego must swerve into the left lane) or sprints on across the street (the ego must "
                      "brake in its lane - swerving left would meet the child there); which one shows only once the child "
                      "is already in the ego lane",
                information_contract="the child is always delivered (from t0, when it appears between the parked cars); "
                                     "the two modes are physically identical until it reaches y_slow; the evidence is its "
                                     "speed (stopping vs sprinting)",
                t0_rule="t0 = the child appears between the parked cars",
                carla_notes=["child: a walker blueprint driven by scripted_trajectories (per mode); parked cars as static "
                             "vehicles in the parking lane",
                             "the gap between the parked cars must hide the child until t0 from the ego's sensors",
                             "run each mode (per_mode) as a separate episode"]))


---
## RC15 — Yielding must be observed, not assumed  (`R_REF` only with a common causal fallback)
**Card (recap).** A merge works when the adjacent actor yields but fails if that favourable response is *assumed* in a
non-yielding world. Yield and non-yield modes have bounded reaction delay and acceleration and both respond to the **actual** ego
state. *Witness:* observe a sufficient gap before merging, otherwise stay in the supported fallback; `always_merge` supplies the
non-yield counterexample. *Accept only when* the common fallback passes before the mode is known; a positive label from a recorded
yielding trajectory alone is invalid. *Paired contrast:* remove the fallback room (or reveal the response too late); keep separate
labels for favourable-world and robust success. **Required profile contrast:** `r(always_merge) < ε`; `r(observe_yield_else_fallback) ≥ ε`.

**How the scenarios implement it.**
* **Adjacent driver** (reactive, `law_adjacent`): it notices the ego's intent once the ego has nudged 0.4 m toward its lane and
  reacts 0.5 or 1.0 s later. Three exogenous modes — **yield** (stays slower than the ego to open a gap ahead of itself, down to a
  floor speed; gives up if the ego drops back alongside), **hold** (keeps its speed whatever the ego does), **match** (speeds up to
  stay level with the ego — or, pronounced, to pull its nose ahead and close the door; never brakes for it). In every mode it
  follows its own lane with IDM, perceived with the same delay, so a forced cut-in is not absorbed instantly (a car that cut in
  and pulls away only makes it lift off).
* **Two response styles** (`STYLES`), so the clips show both levels of movement:

| style | yield | match | ego probe / fallback | scenarios |
|---|---|---|---|---|
| **pronounced** | 7 m/s slower than the ego (≤ 3.5 m/s², floor −9 m/s): its gap ahead clearly opens, the car behind closes up | up to 3 m/s², nose 3 m ahead of the ego, 0.6 s gap to its leader: gap ahead clearly shrinks, the car behind falls back | +6 m/s at 2 m/s², passes 8 m clear before moving over / drops 4 m/s below it | S1, S4 |
| **subtle** | 3 m/s slower (≤ 2.5 m/s², floor −6 m/s) | up to 2.5 m/s², level with the ego | +3 m/s at 1.5 m/s², 3 m clear / 1.5 m/s below it | S2, S3 |

  A **hold** is the same in both: its gaps to the cars ahead and behind stay constant (the car behind starts at its IDM equilibrium gap).
* **Witness `observe_then_merge`**: signal and nudge; if the ego lane ahead is free, **speed up** (the probe); merge **in front**
  once the gap behind is open and the adjacent car is not faster — unless it is seen **speeding up** (matching); keep watching
  while it is seen **slowing** (yielding) or while the ego is visibly gaining on it; otherwise, after 3 s (5 s while yielding /
  gaining), **fall back**: slow, let it pass, merge **behind** into a real gap (protective stop before the lane end if needed).
  This produces the four requested behaviours:

| adjacent driver | ego has room ahead (S1, S2) | ego has no room ahead (S3, S4) |
|---|---|---|
| **yield** — slows, signals "go ahead" | merges in front | merges in front |
| **hold** — keeps its speed | speeds up, passes it → merges in front | waits, merges behind |
| **match** — keeps level with the ego | speeds up, sees it matched → drops back, merges behind | waits, merges behind |

* **Library**: `always_merge` (assumes the yield: merges at 2.5 s, when a yielder would have opened the gap), the witness, the
  **common fallback** (merge behind, committed at 1.0 s — before any mode can be told apart), and an **open-loop replay** of the
  yield-world witness (a diagnostic: a replayed cooperative log is never evidence for the label).
* **Worlds**: mode ∈ {yield, hold, match} × reaction delay ∈ {0.5, 1.0} s — the same exogenous mode for every candidate policy.
* **Checks** per scenario: the card's two inequalities, favourable-world vs robust success, the common fallback, the intended
  outcome per mode, **the witness completes the merge in every world** (never left waiting in the ending lane), and the contrast.
* **Clips** (reaction 0.5 s): the witness in each mode, and `always_merge` in its worst non-yield world. The adjacent car is
  ringed and its gaps to the cars ahead / behind are drawn in blue; the cone taper runs from the ending lane's outer edge toward
  the lane the traffic merges into.

### RC15 — toolkit and runner
`law_adjacent` (reactive adjacent driver), `MergeObserve` (the witness; `assume='front'/'behind'` turns it into the open-loop
assumptions), `ReplayActions` (open-loop replay of a recorded ego command log), `law_lead_zipper` (a vehicle ahead of the ego in
the ending lane that zips into the target lane itself instead of braking for the lane end), `STYLES`, `build_merge(...)` (the
generic merge scene: ego lane blocked from `end_x`, target-lane traffic, optional vehicle ahead of / behind the ego),
`lane_end_overlay`, `gap_overlay`. `rc15_scenario(...)` runs the profile over the six worlds, the per-mode traces, a sweep of the
fallback room, the card's checks, the paired contrast and the CARLA hand-off (per mode: actors, the adjacent driver's law and its
executed trace, the witness trace).

In [ ]:
# ---------- RC15: reactive adjacent driver, observe-then-merge policy, merge scene ----------
def law_adjacent(mode, react, y_ego_lane, side, v_cruise, b_yield=2.5, dv_yield=3.0, dv_floor=6.0, a_match=2.5,
                 d_match=-2.0, T_match=1.2, nudge=0.4):
    """Adjacent driver in the target lane; every mode responds to the *actual* ego state.
    The driver notices the ego's intent once the ego has moved `nudge` m toward the target lane (side = +1/-1) and
    reacts `react` s later:  'yield' stays dv_yield m/s slower than the ego (braking at most b_yield, but not below v_floor)
    to open a gap ahead of itself;  'hold' keeps v_cruise;
    'match' speeds up (never brakes for it, at most a_match) to pull level with the ego and stay there (its centre d_match m
    from the ego centre; d_match > 0 = it pulls its nose ahead to close the door) - blocking the merge. A yielding driver who sees the ego drop back alongside instead (its rear > 3 m
    behind the yielder's front) gives up yielding and drives on.
    In every mode the driver also follows whatever is ahead in its own lane with IDM (incl. the ego once it has
    merged), perceived with the same delay `react` - so a forced cut-in is not absorbed instantly. While matching it
    accepts a shorter time gap T_match to its own leader (it closes up to block the ego)."""
    v_floor = max(v_cruise - dv_floor, 0.0)

    def f(t, me, sim):
        e = sim.ego
        key = ("intent", id(me))
        if key not in sim.mem and side * (e.position[1] - y_ego_lane) > nudge:
            sim.mem[key] = t
        seen = key in sim.mem and t >= sim.mem[key] + react
        ego_in_lane = abs(e.position[1] - me.position[1]) < 2.0
        acc = speed_cmd(me.speed, v_cruise, a_lo=-1.0, a_hi=2.0 if sim.mem.get(("gave_up", id(me))) else 1.0)
        if seen and not ego_in_lane:
            if mode == "yield":
                rear_gap = (e.position[0] - HALF_L) - (me.position[0] + me.LENGTH / 2)
                if rear_gap < -3.0:                              # the ego drops back alongside instead of going ahead
                    sim.mem[("gave_up", id(me))] = True
                if not sim.mem.get(("gave_up", id(me))):            # stay dv_yield slower than the ego (even to a stop)
                    acc = speed_cmd(me.speed, min(v_cruise, max(e.speed - dv_yield, v_floor)), a_lo=-b_yield, a_hi=0.5)
            elif mode == "match":
                acc = float(np.clip(0.8 * ((e.position[0] - me.position[0]) + d_match) + 1.2 * (e.speed - me.speed), 0.0, a_match))
        s, ld = leader_of(me, sim)
        buf = sim.mem.setdefault(("idm_buf", id(me)), [])
        buf.append((1e3, 0.0) if ld is None else (s, me.speed - ld.velocity[0]))
        s_p, dv_p = buf[max(0, len(buf) - 1 - int(round(react / DT)))]       # perceived `react` s ago
        if s_p < 1e3:
            # car-following only as a safety cap (no free-road term): the mode chooses the speed
            T_f = T_match if (mode == "match" and seen) else 1.2
            cap = idm_accel(me.speed, 1e3, s_p, dv_p, T=T_f, s0=2.0, a=2.5, b=2.0)
            if dv_p < 0.0 and s_p > 2.0:            # a car that cut in but pulls away: lift off, no hard braking
                cap = max(cap, -0.8)
            acc = min(acc, cap)
        return float(np.clip(acc, -6.0, a_match))
    return f


class MergeObserve(Policy):
    """Observe-then-merge (the RC15 witness) - causal: uses only delivered observations.
    probe: signal intent with an in-lane nudge toward the target lane; if the ego lane ahead is free (`probe` > 0),
           accelerate gently (a_probe) up to v0 + dv_probe, else keep following the ego-lane traffic.
    decide: merge in front as soon as the gap behind (ego rear to adjacent front) is >= g_front (+1 s of any closing
           speed), the adjacent car is not faster than the ego and the gap to the target-lane car ahead is >= g_ahead;
           if the adjacent car is seen speeding up (> 0.5 m/s^2 over 0.5 s) it is matching - never merge in front of it;
           if it is seen slowing down while the ego is not (< -0.7 m/s^2, >= 0.5 m/s slower), it is yielding - keep watching
           up to T_obs + 2 s (also while the ego is visibly gaining on a car that does not match); if no gap has been
           accepted by then, fall back.
    fallback: return to the lane centre, slow down (b_fall, to dv_fall m/s below the adjacent car) and let it pass; merge
           behind it once the gap next to the ego is usable (car ahead >= g_behind ahead, car behind >= 5 m + 2.5 s of
           closing speed behind); if instead the adjacent car keeps holding back behind the ego with a clear gap in front
           of it, take that gap (no deadlock with a very cautious yielder); IDM
           toward the lane end throughout (protective stop if needed).
    assume='front' / 'behind' turns it into an open-loop assumption (always merge in front at t_assume / fall back
    at t_assume without looking)."""
    def __init__(self, tgt_lane, adj="adj", end="lane_end", a_probe=1.5, dv_probe=3.0, T_obs=3.0, g_front=3.0,
                 g_ahead=8.0, b_fall=3.0, dv_fall=1.5, g_behind=5.0, T_lc=3.0, nudge=0.7, assume=None, t_assume=1.0, name=None):
        # g_front: required gap (ego rear to adjacent front) before merging in front
        self.tgt_y, self.adj, self.end = tgt_lane * LANE_W, adj, end
        self.a_probe, self.dv_probe, self.T_obs, self.g_front, self.g_ahead = a_probe, dv_probe, T_obs, g_front, g_ahead
        self.b_fall, self.dv_fall, self.g_behind, self.T_lc, self.nudge = b_fall, dv_fall, g_behind, T_lc, nudge
        self.assume, self.t_assume = assume, t_assume
        self.name = name or ("observe_then_merge" if assume is None else f"assume_{assume}@{t_assume:g}s")
        self.desc = dict(tgt_lane=tgt_lane, a_probe=a_probe, dv_probe=dv_probe, T_obs=T_obs, g_front=g_front,
                         g_ahead=g_ahead, b_fall=b_fall, dv_fall=dv_fall, g_behind=g_behind, T_lc=T_lc, nudge=nudge, assume=assume,
                         t_assume=t_assume)
        self.reset()

    def reset(self):
        self.phase, self.home, self.v0, self.lc, self.log, self.matching, self.yielding = "probe", None, None, None, [], False, False

    def _go(self, o, phase, y1):
        self.phase = phase
        self.lc = (o.ego["y"], y1, o.t, self.T_lc)
        self.log.append((round(o.t, 2), phase))

    def _yref(self, t, default):
        if self.lc is None:
            return default
        y0, y1, t0, T = self.lc
        return y0 + (y1 - y0) * quintic((t - t0) / T)

    def _gap_ok(self, o):
        """A usable gap in the target lane next to the ego: the car ahead's rear >= g_behind ahead of the ego front and
        the car behind's front >= 5 m + 2.5 s of closing speed behind the ego rear (the adjacent car must be ahead)."""
        e, ahead, behind = o.ego, 1e3, 1e3
        a = o.actors.get(self.adj)
        if a is None or a["x"] < e["x"]:
            return False
        for b in o.actors.values():
            if abs(b["y"] - self.tgt_y) > 2.3:
                continue
            L = extent(b)[0]
            if b["x"] >= e["x"]:
                ahead = min(ahead, (b["x"] - L / 2) - (e["x"] + HALF_L))
            else:
                behind = min(behind, (e["x"] - HALF_L) - (b["x"] + L / 2) - 2.5 * max(0.0, b["v"] - e["v"]))
        return ahead >= self.g_behind and behind >= 5.0

    def _leader(self, o, y, vehicles_only=False):
        s, dv = 1e3, 0.0
        for n, a in o.actors.items():
            if vehicles_only and n == self.end:
                continue
            if abs(a["y"] - y) < 2.3 and a["x"] > o.ego["x"]:
                gap = a["x"] - o.ego["x"] - (extent(a)[0] + EGO_L) / 2
                if gap < s:
                    s, dv = gap, o.ego["v"] - a["v"]
        return s, dv

    def act(self, o):
        e = o.ego
        if self.home is None:
            self.home, self.v0 = round(e["y"] / LANE_W) * LANE_W, e["v"]
        side = np.sign(self.tgt_y - self.home)
        a = o.actors.get(self.adj)
        s_home, dv_home = self._leader(o, self.home)
        acc_home = idm_accel(e["v"], self.v0 + 2.0, s_home, dv_home, T=1.0, s0=3.0, a=self.a_probe, b=2.5)
        room = s_home > max(40.0, 4.0 * e["v"])                   # free road ahead (> 4 s) in the ego lane: may speed up
        s_veh, dv_veh = self._leader(o, self.home, vehicles_only=True)
        acc_cap = idm_accel(e["v"], 1e3, s_veh, dv_veh, T=1.0, s0=3.0, a=self.a_probe, b=2.5)   # vehicles ahead only
        acc_probe = min(self.a_probe if (room and e["v"] < self.v0 + self.dv_probe) else 0.0, acc_cap)
        if self.phase == "probe":
            y_t = self.home + side * self.nudge
            acc = acc_probe if room else acc_home
            if self.assume is not None:
                if o.t >= self.t_assume:
                    self._go(o, "merge_front" if self.assume == "front" else "fallback",
                             self.tgt_y if self.assume == "front" else self.home)
            elif a is not None:
                rear_gap = (e["x"] - HALF_L) - (a["x"] + a["length"] / 2)
                s_tgt, _ = self._leader(o, self.tgt_y)
                need = self.g_front + 1.0 * max(0.0, a["v"] - e["v"])          # more room if it is still closing
                past = o.history[-11][1].get(self.adj) if len(o.history) > 11 else None
                a_adj = (a["v"] - past["v"]) / (o.history[-1][0] - o.history[-11][0]) if past else 0.0
                if self.matching or a_adj > 0.5:                              # it speeds up as the ego does: matching
                    self.matching = True
                if a_adj < -0.7 and a["v"] < e["v"] - 0.5:                     # it slows down for the ego: yielding
                    self.yielding = True
                gaining = not self.matching and e["v"] > a["v"] + 0.5            # visibly pulling ahead of it
                T_wait = self.T_obs + (2.0 if (self.yielding or gaining) else 0.0)
                if (not self.matching and rear_gap >= need and a["v"] <= e["v"] + 0.3 and s_tgt >= self.g_ahead
                        and o.t >= 0.5):
                    self._go(o, "merge_front", self.tgt_y)
                elif o.t >= T_wait or (self.matching and o.t >= 1.5):
                    self._go(o, "fallback", self.home)
        if self.phase == "fallback":
            y_t = self._yref(o.t, self.home)
            acc = min(acc_home, -self.b_fall if a is None or e["v"] > a["v"] - self.dv_fall else 0.0)
            if self._gap_ok(o):
                self._go(o, "merge_behind", self.tgt_y)
            elif self.assume is None and a is not None and not self.matching and o.t >= self.T_obs + 1.0:
                # still watching: the adjacent car is holding back behind the ego (a late or very cautious yield)
                rear_gap = (e["x"] - HALF_L) - (a["x"] + a["length"] / 2)
                if rear_gap >= self.g_front + 2.0 and a["v"] <= e["v"] + 0.3 and self._leader(o, self.tgt_y)[0] >= self.g_ahead:
                    self._go(o, "merge_front", self.tgt_y)
        if self.phase in ("merge_front", "merge_behind"):
            y_t = self._yref(o.t, self.tgt_y)
            s_t, dv_t = self._leader(o, self.tgt_y)
            acc = idm_accel(e["v"], self.v0 + (self.dv_probe if self.phase == "merge_front" else 0.0), min(s_t, s_home if
                            abs(e["y"] - self.home) < 2.0 else 1e3), dv_t, T=1.0, s0=3.0, a=1.5, b=2.5)
        return float(np.clip(acc, -6.0, 2.0)), track_y(e["x"], e["y"], e["psi"], e["v"], y_t)


class ReplayActions(Policy):
    """Open-loop replay of a recorded ego command sequence (what a log-replay 'policy' does)."""
    def __init__(self, tr, name="replay_of_yield_world_witness"):
        pol = tr.rec["phase"] == "policy"
        self.cmds = list(zip(tr.rec["a_req"][pol], tr.rec["steer_req"][pol]))
        self.name, self.desc = name, dict(source=tr.policy.name, world=tr.scene.world)
        self.reset()

    def reset(self):
        self.k = 0

    def act(self, o):
        c = self.cmds[min(self.k, len(self.cmds) - 1)]
        self.k += 1
        return float(c[0]), float(c[1])


def lane_end_overlay(x_end, text, lane, side):
    """Mark the start of the blocked part of the ego lane (cones / end of lane) and label it. The taper runs from the
    outer edge of the ending lane toward the lane the traffic merges into (side = +1: toward higher y, -1: lower y)."""
    def draw(d, px, py, t, k, tr, ov):
        y0, y1 = py(lane * LANE_W - LANE_W / 2), py(lane * LANE_W + LANE_W / 2)
        for i in range(12):                                     # cone taper
            xx = x_end - 30 + 2.5 * i
            yy = lane * LANE_W - side * LANE_W / 2 + side * (LANE_W * i / 12.0)
            d.ellipse([px(xx) - 3, py(yy) - 3, px(xx) + 3, py(yy) + 3], fill=(255, 120, 0, 255))
        y_txt = min(y0, y1) - 14 if side > 0 else max(y0, y1) + 3       # on the outer side, away from the traffic
        d.text((px(x_end - 30), y_txt), text, fill=(255, 200, 120, 255))
    return frame_overlay(draw)


def law_lead_zipper(v0, lane_y, tgt_y, x_end, react=0.5, T=1.2, t_look=4.0, g_front=6.0, g_rear=8.0, T_lc=3.0):
    """A vehicle ahead of the ego in the ending lane: it follows its own lane (IDM on vehicles; the lane end only
    within its last 30 m, and not once it is merging) and, once within t_look s of the end, merges into the first gap next to it (front >= g_front,
    rear >= g_rear + 1.5 s of closing speed) - it zips in like any driver instead of braking for the lane end."""
    def f(t, me, sim):
        key = ("zip", id(me))
        x = me.position[0]
        if key not in sim.mem and x > x_end - t_look * max(me.speed, 5.0):
            ahead, behind = 1e3, 1e3
            for ob in [sim.ego] + list(sim.objs.values()):
                if ob is me or abs(ob.position[1] - tgt_y) > 1.8:
                    continue
                dx = ob.position[0] - x
                if dx >= 0:
                    ahead = min(ahead, dx - (ob.LENGTH + me.LENGTH) / 2)
                else:
                    behind = min(behind, -dx - (ob.LENGTH + me.LENGTH) / 2 - 1.5 * max(0.0, ob.speed - me.speed))
            if ahead >= g_front and behind >= g_rear:
                sim.mem[key] = (t, me.position[1])
        if key in sim.mem:
            t0, y0 = sim.mem[key]
            y_t = y0 + (tgt_y - y0) * quintic((t - t0) / T_lc)
        else:
            y_t = lane_y
        buf = sim.mem.setdefault(("zip_buf", id(me)), [])
        s, ld = leader_of(me, sim)
        if ld is not None and ld.LENGTH > 20 and (key in sim.mem or s > 30.0):     # the static lane end: only near it
            s, ld = None, None
        buf.append((1e3, 0.0) if ld is None else (s, me.speed - ld.velocity[0]))
        s_p, dv_p = buf[max(0, len(buf) - 1 - int(round(react / DT)))]
        return float(np.clip(idm_accel(me.speed, v0, s_p, dv_p, T=T, s0=2.0, a=1.5, b=2.0), -6.0, 1.5)), y_t
    return f


def gap_overlay(name="adj", colour=(120, 220, 255, 255)):
    """Draw the adjacent car's bumper gaps to the nearest vehicle ahead and behind it in its lane (ego included), in
    metres, so a yield (front gap grows, rear gap shrinks), a hold (both steady) and a match (front gap shrinks, rear
    gap grows) are visible in the clip."""
    lens = {}

    def draw(d, px, py, t, k, tr, ov):
        if not lens:
            lens.update({a.name: a.length for a in tr.scene.actors if a.kind != "static"})
            lens["ego"] = EGO_L
        if name not in tr.actors or k >= len(tr.actors[name]):
            return
        _, xa, ya, _, _ = tr.actors[name][k]
        pos = {n: tr.actors[n][k][1:3] for n in lens if n in tr.actors and n != name and k < len(tr.actors[n])}
        pos["ego"] = (tr.rec["x"][k], tr.rec["y"][k])
        same = {n: x for n, (x, y) in pos.items() if abs(y - ya) < 1.5}
        yl = py(ya) - 13
        for sgn in (1, -1):
            cand = [n for n, x in same.items() if sgn * (x - xa) > 0]
            if not cand:
                continue
            o = min(cand, key=lambda n: abs(same[n] - xa))
            x1, x2 = xa + sgn * lens[name] / 2, same[o] - sgn * lens[o] / 2
            d.line([px(x1), yl, px(x2), yl], fill=colour, width=1)
            for xx in (x1, x2):
                d.line([px(xx), yl - 3, px(xx), yl + 3], fill=colour, width=1)
            d.text(((px(x1) + px(x2)) / 2 - 12, yl - 12), f"{sgn * (x2 - x1):.0f} m", fill=colour)
    return frame_overlay(draw)


STYLES = {   # how visible the adjacent driver's response and the ego's probe / fallback are
    "subtle": dict(adj=dict(b_yield=2.5, dv_yield=3.0, dv_floor=6.0, a_match=2.5, d_match=-2.0, T_match=1.2), pol=dict()),
    "pronounced": dict(adj=dict(b_yield=3.5, dv_yield=7.0, dv_floor=9.0, a_match=3.0, d_match=3.0, T_match=0.6),
                       pol=dict(a_probe=2.0, dv_probe=6.0, g_front=8.0, dv_fall=4.0, b_fall=3.5)),
}


def build_merge(sid, mode="yield", react=0.5, v=17.0, v_adj=None, adj_dx=-4.0, lanes=2, ego_lane=1, tgt_lane=0,
                end_x=170.0, end_len=80.0, end_kind="cones", end_label="LANE CLOSED", ego_lead=None, ego_follow=None,
                tgt_lead_dx=45.0, tgt_follow_dx=-48.0, style="subtle", T=20.0, world_extra=None):
    """Generic RC15 merge scene: the ego lane is blocked from end_x (cones, ramp end, lane drop ...); the ego must
    merge into tgt_lane, where the adjacent car 'adj' (centre adj_dx m from the ego centre at t0) drives with a car
    ahead (tgt_lead_dx) and one behind (tgt_follow_dx). ego_lead = (dx, v[, L]) puts a vehicle close ahead in the ego
    lane (no room to speed up); it zips into the target lane itself near the end. ego_follow = (dx, react, b_max) puts
    a car behind the ego in its lane (a tailgater: a sudden stop in lane is then no longer protected).
    style = 'subtle' / 'pronounced' sets how strongly the adjacent driver yields or matches (STYLES). The adjacent
    driver's mode is exogenous and fixed per world; its reaction is causal."""
    v_adj = v if v_adj is None else v_adj
    side = 1.0 if tgt_lane > ego_lane else -1.0
    ka = STYLES[style]["adj"]
    gap0, s_eq = (adj_dx - tgt_follow_dx) - 4.8, 2.0 + 1.2 * v_adj     # rear car: desired speed with gap0 as equilibrium
    v_follow = v_adj / (1.0 - (s_eq / gap0) ** 2) ** 0.25 if gap0 > 1.05 * s_eq else v_adj + 3.0
    acts = [Actor("adj", x=adj_dx - v_adj * 1.0, lane=tgt_lane, speed=v_adj, length=4.8, kind="vehicle",
                  law=law_adjacent(mode, react, ego_lane * LANE_W, side, v_adj, **ka),
                  note=f"adjacent driver, mode={mode} ({style}): notices the ego's nudge (>0.4 m toward its lane), reacts "
                       f"after {react} s; yield = stays {ka['dv_yield']:g} m/s slower than the ego (brakes <= "
                       f"{ka['b_yield']:g} m/s^2, not below {max(v_adj - ka['dv_floor'], 0):g} m/s), hold = keeps {v_adj} m/s, match = speeds up (<= {ka['a_match']:g} "
                       f"m/s^2) to keep its centre {ka['d_match']:+g} m from the ego's (time gap {ka['T_match']:g} s to its leader); IDM on its own lane"),
            Actor("tgt_lead", x=tgt_lead_dx - v_adj * 1.0, lane=tgt_lane, speed=v_adj, kind="vehicle",
                  law=law_idm(v_adj, T=1.2, react=0.5), note="target-lane car ahead of the adjacent car"),
            Actor("tgt_follow", x=tgt_follow_dx - v_adj * 1.0, lane=tgt_lane, speed=v_adj, kind="vehicle",
                  law=law_idm(v_follow, T=1.2, react=0.8),
                  note=f"target-lane car behind the adjacent car (IDM, desired {v_follow:.1f} m/s: its start gap is its "
                       f"equilibrium gap at {v_adj:g} m/s)"),
            Actor("lane_end", x=end_x + end_len / 2, lane=ego_lane, length=end_len, width=LANE_W - 0.6, kind="static",
                  note=f"{end_kind}: the ego lane is blocked from {end_x} m")]
    if ego_lead is not None:
        dx, vl, Ll = (*ego_lead, 4.8)[:3]
        acts.append(Actor("ego_lead", x=dx - vl * 1.0, lane=ego_lane, speed=vl, length=Ll, width=2.55 if Ll > 6 else 2.0,
                          kind="vehicle", law=law_lead_zipper(vl, ego_lane * LANE_W, tgt_lane * LANE_W, end_x),
                          note=f"vehicle ({Ll} m) close ahead in the ego lane at {vl} m/s (no room to speed up); IDM, "
                               f"zips into the first gap in the target lane within 4 s of the lane end"))
    if ego_follow is not None:
        dx, rf, bf = ego_follow
        acts.append(Actor("ego_follow", x=dx - v * 1.0, lane=ego_lane, speed=v, kind="vehicle",
                          law=law_idm(v, T=0.8, react=rf, b_max=bf),
                          note=f"tailgater {abs(dx)} m behind the ego, perception delay {rf} s, max braking {bf} m/s^2"))
    return Scene(sid, lanes=lanes, ego_lane=ego_lane, ego_speed=v, ego_x=-v * 1.0, actors=acts, T=T, T_term=2.0,
                 prefix_policy=cruise(), prefix_T=1.0,
                 world=dict(mode=mode, react=react, v=v, v_adj=v_adj, adj_dx=adj_dx, end_x=end_x, style=style,
                            **(world_extra or {})),
                 meta=dict(overlay=compose_overlays(lane_end_overlay(end_x, end_label, ego_lane, side), mark_actor("adj"),
                                                    gap_overlay(),
                                                    label_overlay([f"adjacent driver (ringed): {mode} ({style})",
                                                                   "blue: its gaps to the cars ahead / behind"]))))


print("RC15 toolkit ok")


In [ ]:
def merge_order(tr, tgt_lane):
    """'front' / 'behind' the adjacent car if the ego ends in the target lane, else 'not_merged'."""
    if abs(tr.rec["y"][-1] - tgt_lane * LANE_W) > 1.0:
        return "not_merged"
    return "front" if tr.rec["x"][-1] > np.array(tr.actors["adj"])[-1, 1] else "behind"


def rc15_scenario(sid, build, tgt_lane, expected, reacts=(0.5, 1.0), clip_react=0.5, contrast=None, sweep=None,
                  xlim=(-30, 220), record=None, T_commit=1.0, style="subtle"):
    """Run one RC15 scenario: profile of the library over modes {yield, hold, match} x reaction delays -> per-mode
    traces -> sweep of the fallback room -> the card's checks -> paired contrast -> CARLA hand-off.
    expected: mode -> 'front'/'behind', the witness outcome the story intends (checked at clip_react).
    style: 'subtle' / 'pronounced' (STYLES) - the ego's probe and fallback strength; the scene builder takes the same."""
    modes = ("yield", "hold", "match")
    worlds = [dict(mode=m, react=r) for m in modes for r in reacts]
    key = lambda p, w: (p, json.dumps(w, sort_keys=True))
    kp = STYLES[style]["pol"]
    wit = MergeObserve(tgt_lane, **kp)
    amerge = MergeObserve(tgt_lane, assume="front", t_assume=2.5, name="always_merge", **kp)
    fallback = MergeObserve(tgt_lane, assume="behind", t_assume=T_commit, name=f"common_fallback(merge_behind@{T_commit:g}s)", **kp)
    tr_y = simulate(build(mode="yield", react=clip_react), wit)
    replay = ReplayActions(tr_y)
    lib = [amerge, wit, fallback, replay]
    df, trs = evaluate(build, lib, worlds, keep=True)
    prof = robust_profile(df)
    fav = df[df.world.str.contains('"yield"')].groupby("policy", sort=False)["r"].max()
    display(df.pivot_table(index="policy", columns="world", values="r", sort=False).round(2))
    display(pd.DataFrame({"robust r (all modes)": prof.round(2), "favourable-world r (best yield world)": fav.round(2)}))
    outcome = {m: merge_order(trs[key(wit.name, dict(mode=m, react=clip_react))], tgt_lane) for m in modes}
    print("witness outcome per mode (react %.1f s): %s" % (clip_react, outcome))
    plot_traces([trs[key(wit.name, dict(mode=m, react=clip_react))] for m in modes] +
                [trs[key(amerge.name, dict(mode="match", react=clip_react))]],
                f"{sid}: observe_then_merge in the yield / hold / match worlds, and always_merge (match)", xlim=xlim)

    var, xs = sweep["var"], np.asarray(sweep["values"], dtype=float)
    r_w = [min(simulate(build(**{**w, var: x}), wit).r for w in worlds) for x in xs]
    r_f = [min(simulate(build(**{**w, var: x}), fallback).r for w in worlds) for x in xs]
    r_a = [min(simulate(build(**{**w, var: x}), amerge).r for w in worlds) for x in xs]
    plot_sweep(xs, {"observe_then_merge (min over worlds)": r_w, "common fallback (min)": r_f, "always_merge (min)": r_a},
               sweep["xlabel"], f"{sid}: {sweep['title']}")

    check(sid, "r(always_merge) < eps (a non-yielding world is the counterexample)", prof["always_merge"] < EPS)
    check(sid, "r(observe_then_merge) >= eps in every mode and reaction delay", prof[wit.name] >= EPS)
    check(sid, "always_merge's favourable-world success is recorded separately (it passes a yield world)", fav["always_merge"] >= EPS)
    check(sid, f"the common fallback, committed at {T_commit:g} s before the mode is known, passes every world",
          prof[fallback.name] >= EPS)
    print(f"diagnostic: open-loop replay of the yield-world witness - robust r = {prof[replay.name]:+.2f} "
          f"(a replayed cooperative log is never evidence for the label, whether or not it happens to survive)")
    check(sid, f"the witness does what the story intends in each mode: {expected}", outcome == expected)
    stuck = [w for w in worlds if merge_order(trs[key(wit.name, w)], tgt_lane) == "not_merged"]
    check(sid, f"the witness completes the merge in every world (never left waiting in the ending lane){': ' + str(stuck) if stuck else ''}",
          not stuck)
    dfc = evaluate(build, [amerge, wit, fallback], [{**w, **contrast[1]} for w in worlds])
    labc = label_from_profile(robust_profile(dfc), library_complete=True)
    print(f"paired contrast ({contrast[0]}): label from this finite library = {labc}")
    check(sid, "paired contrast: removing the fallback room leaves no robust policy in the library (U_LIBRARY)", labc == "U_LIBRARY")

    per_mode = {}
    for m in modes:
        tw = trs[key(wit.name, dict(mode=m, react=clip_react))]
        per_mode[m] = {"actors": actors_rel(tw.scene, tw), "actor_laws": {a.name: a.note for a in tw.scene.actors},
                       "world": tw.scene.world, "witness_outcome": outcome[m], "witness_trace": ego_keyframes(tw),
                       "adjacent_trace": [dict(t=round(float(r[0] - tw.t0), 2), dx=round(float(r[1] - tw.measured_t0["ego"]["x"]), 2),
                                               dy=round(float(r[2] - tw.measured_t0["ego"]["y"]), 2), v=round(float(r[4]), 2))
                                          for r in tw.actors["adj"][::2]]}
    sc0 = trs[key(wit.name, dict(mode="yield", react=clip_react))].scene
    handoff(sid, parent="RC15", label=("R_REF" if prof[wit.name] >= EPS else "UNRESOLVED"), **(record or {}),
            evidence=f"executed witness in HighwayEnv reference dynamics over {len(worlds)} worlds; not CARLA-validated",
            ego={"v": sc0.ego_speed, "lane": sc0.ego_lane, "target_lane": tgt_lane, "envelope": asdict(sc0.env)},
            road={"lanes": sc0.lanes, "lane_width": LANE_W, "drivable_y": list(sc0.bounds)},
            worlds=worlds, per_mode=per_mode, witness=wit.name, witness_descriptor=wit.descriptor(),
            refuted=["always_merge"], replay_diagnostic={replay.name: round(float(prof[replay.name]), 3)},
            favorable_world_success=fav.round(3).to_dict(),
            robust_success=prof.round(3).to_dict(), common_fallback=fallback.descriptor(),
            sweep={"var": var, "values": xs.tolist(), "r_witness_min": r_w, "r_fallback_min": r_f, "r_always_merge_min": r_a},
            paired_contrast={"change": contrast[0], "kwargs": contrast[1], "label": labc})
    clips = {f"{m}_observe_merges_{outcome[m]}": (dict(mode=m, react=clip_react), wit) for m in modes}
    am = df[(df.policy == "always_merge") & ~df.world.str.contains('"yield"')].sort_values("r").iloc[0]
    w_am = json.loads(am["world"])
    clips[f"always_merge_{w_am['mode']}_crash"] = (w_am, amerge)
    return df, clips


print("RC15 scenario runner ok")


### RC15-S1 — Motorway on-ramp  *(room ahead · pronounced responses)*
**Story.** The ego joins a motorway at 80 km/h on a 200 m acceleration lane; a car is alongside in the motorway's right lane. The
acceleration lane ahead is empty, so the ego accelerates firmly while it signals. Every response is unmistakable:
**Yield:** the motorway driver brakes well down — its gap to the car ahead opens and the car behind closes up → the ego slots in
ahead. **Hold:** it keeps 80 km/h (both gaps constant) and the ego clearly overtakes it, moving in once 8 m clear. **Match:** it
floors it to stay level — its gap ahead shrinks and the car behind falls back → the ego brakes and joins behind it.
**Paired contrast:** a 60 m acceleration lane and a tailgater behind the ego.

In [ ]:
build_rc15_s1 = lambda **kw: build_merge("RC15-S1", **{**dict(v=22.0, lanes=3, ego_lane=2, tgt_lane=1, end_x=200.0,
                                                              end_kind="end of the acceleration lane", end_label="RAMP ENDS",
                                                              style="pronounced"), **kw})
df_15s1, clips_15s1 = rc15_scenario(
    "RC15-S1", build_rc15_s1, tgt_lane=1, expected={"yield": "front", "hold": "front", "match": "behind"}, style="pronounced",
    contrast=("a short acceleration lane (60 m) and a tailgater behind the ego: no room to observe and no protected stop",
              dict(end_x=60.0, ego_follow=(-12.0, 1.2, 4.0))),
    sweep=dict(var="end_x", values=np.arange(50.0, 230.1, 20.0), xlabel="distance to the end of the acceleration lane [m]",
               title="length of the acceleration lane"),
    record=dict(site="motorway on-ramp: 2 through lanes + an acceleration lane on the right; ego in lane 2 (acceleration lane)",
                story="joining a motorway at 80 km/h with a car alongside in the right lane, everything clearly visible: the "
                      "ego signals, drifts toward the line and accelerates firmly (the acceleration lane ahead is empty). "
                      "A courteous driver brakes well down (its gap to the car ahead opens, the car behind closes up) and "
                      "the ego slots in ahead; a driver who holds 80 km/h is clearly overtaken (its gaps stay constant) and "
                      "the ego moves in ahead once it is 8 m clear; a driver who floors it to stay level (its gap ahead "
                      "shrinks, the car behind falls back) makes the ego brake and join behind it",
                t0_rule="t0 = the ego starts to signal on the acceleration lane (200 m before its end)",
                carla_notes=["needs a map with an on-ramp (Town04 / Town06); the acceleration-lane end is the lane_end actor",
                             "the adjacent car must be driven by its reactive law (actor_laws), never replayed",
                             "style = pronounced: large yield / match responses and a firm probe (see STYLES in the notebook)",
                             "the three modes are separate episodes with identical spawns"]))


### RC15-S2 — End of a 2+1 road's passing lane  *(room ahead · subtle responses)*
**Story.** On a rural 2+1 road at 90 km/h the ego is overtaking and is level with the car it is passing when the passing lane
starts to end (220 m). It signals right and edges ahead. The cues are small, as they usually are:
**Yield:** the overtaken driver backs off ~3 m/s → the ego completes the pass. **Hold:** the driver keeps its speed; the ego's
extra 3 m/s completes the pass. **Match:** the driver speeds up a little as soon as it is overtaken — a common reaction — so the
ego gives up, drops back and tucks in behind it.
**Paired contrast:** the passing lane ends 70 m ahead and a car follows the ego closely.

In [ ]:
build_rc15_s2 = lambda **kw: build_merge("RC15-S2", **{**dict(v=25.0, adj_dx=-2.5, lanes=2, ego_lane=0, tgt_lane=1, end_x=220.0,
                                                              end_kind="end of the passing lane", end_label="PASSING LANE ENDS",
                                                              style="subtle"), **kw})
df_15s2, clips_15s2 = rc15_scenario(
    "RC15-S2", build_rc15_s2, tgt_lane=1, expected={"yield": "front", "hold": "front", "match": "behind"}, style="subtle",
    contrast=("the passing lane ends 70 m ahead and a car follows the ego closely: no room to observe and no protected stop",
              dict(end_x=70.0, ego_follow=(-12.0, 1.2, 4.0))),
    sweep=dict(var="end_x", values=np.arange(60.0, 250.1, 20.0), xlabel="distance to the end of the passing lane [m]",
               title="passing-lane length left"),
    record=dict(site="rural 2+1 road: a passing lane (lane 0) next to the normal lane (lane 1); ego in the passing lane",
                story="the ego is overtaking on a 2+1 road at 90 km/h and is level with the car it is passing when the "
                      "passing lane starts to end; it signals right and edges ahead. The cues are small, as on a real road: "
                      "the other driver eases off by ~3 m/s, or keeps its speed, and the ego completes the pass and moves in "
                      "ahead; or it speeds up a little as soon as it is being overtaken (a common reaction) and the ego "
                      "gives up, drops back and tucks in behind it",
                t0_rule="t0 = the ego starts to signal (220 m before the passing lane ends)",
                carla_notes=["a 2+1 road can be built in OpenDRIVE (a lane that ends with a taper); the end is the lane_end actor",
                             "style = subtle: small yield / match responses (see STYLES in the notebook)",
                             "the overtaken car must be driven by its reactive law (actor_laws), never replayed"]))


### RC15-S3 — Toll-plaza funnel in a queue  *(no room ahead · subtle responses)*
**Story.** Leaving a toll plaza in a slow queue (36 km/h), two lanes funnel into one in 110 m; the ego's lane ends and the car ahead
of it leaves no room to speed up (it zips into the other lane itself near the funnel). A car is alongside in the other lane.
**Yield:** it eases off a little → the ego goes in front. **Hold / match:** it keeps rolling (or creeps up) → the ego waits and
merges behind it.
**Paired contrast:** the funnel 45 m ahead and a tailgater behind the ego.

In [ ]:
build_rc15_s3 = lambda **kw: build_merge("RC15-S3", **{**dict(v=10.0, lanes=2, ego_lane=0, tgt_lane=1, end_x=110.0,
                                                              end_kind="lane funnel after the toll plaza", end_label="LANES MERGE",
                                                              ego_lead=(18.0, 10.0), tgt_lead_dx=30.0, tgt_follow_dx=-35.0,
                                                              style="subtle"), **kw})
df_15s3, clips_15s3 = rc15_scenario(
    "RC15-S3", build_rc15_s3, tgt_lane=1, expected={"yield": "front", "hold": "behind", "match": "behind"}, style="subtle",
    contrast=("a tailgater close behind the ego (reacts after 1.2 s, brakes at most 4 m/s^2) and the funnel 45 m ahead: no "
              "room to observe and no protected stop in the closing lane", dict(ego_follow=(-11.0, 1.2, 4.0), end_x=45.0)),
    sweep=dict(var="end_x", values=np.arange(40.0, 140.1, 20.0), xlabel="distance to the funnel [m]",
               title="room before the lanes merge"),
    xlim=(-30, 140),
    record=dict(site="toll-plaza exit: two lanes funnel into one, slow queued traffic (36 km/h); ego in lane 0 (left, ending)",
                story="leaving a toll plaza in a slow queue, the two lanes funnel into one; the car ahead of the ego zips in "
                      "first and leaves no room to speed up; the ego signals and edges right next to a car in the other "
                      "lane. The cues are small: that driver eases off a little and the ego goes ahead of it, or it keeps "
                      "rolling (or creeps up) and the ego waits and merges behind it",
                t0_rule="t0 = the ego starts to signal (110 m before the funnel)",
                carla_notes=["a funnel can be built as a lane that ends (taper) in OpenDRIVE; queue cars via actor_laws (IDM)",
                             "the queue car ahead of the ego zips into the other lane itself (actor_laws: ego_lead)",
                             "style = subtle: small yield / match responses (see STYLES in the notebook)",
                             "the adjacent car must be driven by its reactive law (actor_laws), never replayed"]))


### RC15-S4 — Lane drop behind a truck  *(no room ahead · pronounced responses)*
**Story.** On a 2-lane highway (80 km/h) the right lane ends in 200 m and the ego follows a truck in it, so it cannot speed up past
the car alongside in the left lane; the truck itself zips into the left lane ahead of the traffic near the end.
**Yield:** the driver brakes hard (gap ahead opens, the car behind closes up) → the ego moves in ahead without slowing.
**Hold:** constant speed (gaps constant) → the ego brakes clearly, lets it go and merges behind it. **Match:** it accelerates to
close the door (gap ahead shrinks, the car behind falls back) → the ego drops back and merges behind it.
**Paired contrast:** the lane ends 70 m ahead and a car follows the ego closely.

In [ ]:
build_rc15_s4 = lambda **kw: build_merge("RC15-S4", **{**dict(v=22.0, lanes=2, ego_lane=1, tgt_lane=0, end_x=200.0,
                                                              end_kind="lane drop", end_label="LANE ENDS",
                                                              ego_lead=(60.0, 22.0, 12.0), tgt_lead_dx=40.0,
                                                              style="pronounced"), **kw})
df_15s4, clips_15s4 = rc15_scenario(
    "RC15-S4", build_rc15_s4, tgt_lane=0, expected={"yield": "front", "hold": "behind", "match": "behind"}, style="pronounced",
    contrast=("the lane ends 70 m ahead and a car follows the ego closely: no room to observe and no protected stop",
              dict(end_x=70.0, ego_follow=(-12.0, 1.2, 4.0))),
    sweep=dict(var="end_x", values=np.arange(60.0, 240.1, 20.0), xlabel="distance to the lane drop [m]",
               title="room before the lane drop"),
    record=dict(site="2-lane highway with a lane drop on the right, 80 km/h; ego in lane 1 (right, ending) behind a truck",
                story="the right lane ends ahead and the ego follows a truck in it, so it cannot speed up past the car "
                      "alongside in the left lane (the truck itself zips into the left lane ahead of the traffic near the "
                      "end). The ego signals and edges left; the responses are unmistakable: a courteous driver brakes "
                      "hard (gap ahead of it opens, the car behind closes up) and the ego moves in ahead; a driver who holds "
                      "its speed (gaps constant) or accelerates to close the door (gap ahead shrinks, the car behind falls "
                      "back) makes the ego brake clearly, let it go and merge behind it",
                t0_rule="t0 = the ego starts to signal (200 m before the lane drop)",
                carla_notes=["truck ahead: a heavy-truck blueprint in the ego lane following actor_laws (it zips left itself)",
                             "style = pronounced: large yield / match responses and a firm fallback (see STYLES in the notebook)",
                             "the adjacent car must be driven by its reactive law (actor_laws), never replayed"]))


---
## RC19 — Benign and label-preserving nuisance controls  (`R_REF` must not move)
**Card (recap).** *Benign anchor:* a clear scene where nominal driving must pass and must not be out-scored on progress by an
always-stop policy (track unnecessary interventions). *Nuisance pair:* the same scene plus a distant, non-interacting object
(after a visibility check) and a render-only change; the profiles must agree within tolerance. A deliberate **semantic** change
(an actor moved into the conflict corridor) is kept separate — it must change the profile.

**How the scenarios implement it.** Each scenario fixes one **safety-relevant** scene (a hazard that forces the ego to act) and
runs it under **four benign cases** plus **one semantic control**:

| case | what changes | HighwayEnv | CARLA |
|---|---|---|---|
| **A anchor** | nothing (clear noon, plain scene) | the fixed scene | clear weather, layered map with optional layers unloaded |
| **B appearance** | colours, scenery, sun / cloud | render-only colours + backdrop drawn under the scene | `color` attribute, map layers (Buildings, Foliage, Decals), sun angle ≥ 25°, cloudiness |
| **C distant / non-interacting** | traffic or people that never come near | extra actors outside the corridor | background traffic / walkers on fixed routes, fixed seed |
| **D roadside** | parked or stopped objects beside the road | parked vehicles ≥ 2 m from every used lane | parked actors (no controller), props |
| **semantic control** | an object that matters (in the corridor, occluding, or a resized relevant actor) | — | — |

**Benign is checked, not declared** — every case runs an admissibility test: (1) every extra object keeps ≥ 2 m from the
corridor box (all lanes a library policy uses, from 30 m behind to 80 m ahead of the ego) at every tick; (2) no extra object ever
blocks the ego's sight line to a safety-relevant actor (relevant actors are delivered only with a clear line of sight in *every*
RC19 run); (3) the CARLA weather keeps precipitation, puddles, wetness and fog at zero and the sun ≥ 25° high (friction,
visibility and glare are **not** nuisances); (4) the fixed actors are untouched (same spawn, size, class, speed).
**Never benign:** rain / wet road, fog, night, low sun ahead, a different size or class of a relevant actor, anything within
2 m of a used lane (e.g. kerb-side parking: door zone), anything on a sight line to a relevant actor, changed timing.

**Library** (per scenario): the **witness** (a corridor-based causal driver; in S4 the RC15 observe-then-merge policy), an
**unsafe** policy that ignores the hazard (shows the fixed scene is safety-relevant), **always_stop** (must lose on progress), and
a **distracted** variant of the witness that phantom-brakes (2 m/s²) for any slow object within 40 m ahead and 9 m to the side —
a deliberately benign-*sensitive* policy, kept as a contrast.

**Checks** (per scenario): the witness passes in all four benign cases (same label `R_REF`); the unsafe policy fails in all four;
always_stop is caught by progress (< 0.6 × the witness); every benign case is admissible; **invariance** — the witness, the unsafe
policy and always_stop have identical profiles in all four cases (|Δr| ≤ 1e-6, trajectory within 0.01 m); the test has teeth — the
distracted policy's trajectory changes in some benign case; the semantic control is flagged as not benign *and* changes the
witness's profile.

> In HighwayEnv the ego sees ground truth, so render-only changes (B) are invariant by construction: here they verify that the
> scene specification is truly render-only. Their real test is in CARLA, with camera perception in the loop — the hand-off lists
> the CARLA settings of every case, and the admissibility check must be re-run there with the actual sensor rig.

### RC19 — toolkit and runner
`BenignCase` (one variant: render-only colours and scenery, physical non-interacting extras, the CARLA settings, and — semantic
controls only — `modify` for a fixed actor), `apply_case` (fixed scene + case + line-of-sight gating of the relevant actors),
`los_gate`, `CorridorDriver` (the witness), `Distracted` (the benign-sensitive contrast), `law_event` (fixed traffic: IDM plus a
scripted brake / lane change), `parked`, `walker`, `scenery_overlay` (drawn *under* the scene), `outline_actors`, `progress`,
`interventions`. `rc19_scenario(...)` runs the library in every case, the admissibility check, the invariance table against the
anchor, a speed plot (all benign cases should coincide), the checks and the CARLA hand-off (per case: render-only settings,
physical extras with their laws, CARLA settings, admissibility, profile and deviation from the anchor).

In [ ]:
# ---------- RC19: benign (label-preserving) settings around a fixed safety-relevant scene ----------
from shapely.geometry import LineString

# CARLA weather fields a benign case may NOT change (friction / visibility / glare are safety-relevant, not benign)
WEATHER_FIXED = dict(precipitation=0.0, precipitation_deposits=0.0, wetness=0.0, fog_density=0.0)
SUN_ALT_MIN = 25.0          # deg; a lower sun can blind a forward camera (RC11-S3) -> not benign
CLEAR_MIN = 2.0             # m; a benign physical object keeps this lateral clearance to every lane a library policy uses


@dataclass
class BenignCase:
    """One benign variant of a fixed scene. Everything here must leave the recoverability profile unchanged.
    colours: actor name ('ego' included) -> RGB, render-only.   scenery: backdrop drawn by the overlay, render-only.
    extras:  physical but non-interacting actors (outside the drivable corridor, never occluding a relevant actor).
    carla:   the same case in CARLA terms (weather, map layers, colours, walkers, props, background traffic).
    modify:  {actor: {field: value}} changes to a FIXED actor - only ever used by a semantic control."""
    name: str
    title: str
    colours: dict = field(default_factory=dict)
    scenery: tuple = ()
    extras: list = field(default_factory=list)
    carla: dict = field(default_factory=dict)
    modify: dict = field(default_factory=dict)


def parked(name, x, y, length=4.6, width=1.9, colour=None, note="parked car (benign)"):
    """A stationary vehicle drawn as a car (not an obstacle block)."""
    return Actor(name, x=x, y=y, length=length, width=width, speed=0.0, kind="vehicle", law=law_const(0.0), note=note)


def walker(name, x0, y0, vx, vy=0.0, note="pedestrian walking along the sidewalk (benign)"):
    def f(t):
        return x0 + vx * t, y0 + vy * t, math.atan2(vy, vx) if (vx or vy) else 0.0, vx, vy
    f.params = dict(x0=x0, y0=y0, vx=vx, vy=vy)
    return Actor(name, x=x0, y=y0, length=0.6, width=0.6, kind="scripted", law=f, note=note)


def los_clear(p0, p1, polys):
    seg = LineString([p0, p1])
    return not any(seg.intersects(P) for P in polys)


def los_gate(targets, eye_dx=0.5):
    """Scene.visible: a relevant actor is delivered only if a sight line from the ego's eye (eye_dx m behind the front
    centre) to its centre or one of its corners is not blocked by another actor's footprint (pedestrians do not block)."""
    def vis(t, n, sim):
        if n not in targets:
            return True
        e, o = sim.ego, sim.objs[n]
        eye = (e.position[0] + HALF_L - eye_dx, e.position[1])
        polys = [ShPolygon(obb(b.position[0], b.position[1], b.heading, b.LENGTH, b.WIDTH))
                 for m, b in sim.objs.items() if m != n and b.WIDTH > 1.0]
        pts = [tuple(o.position)] + obb(o.position[0], o.position[1], o.heading, o.LENGTH, o.WIDTH)
        return any(los_clear(eye, p, polys) for p in pts)
    vis.needs_sim = True
    return vis


def law_event(v0, t_ev=None, b=0.0, v_end=0.0, y_to=None, T_lc=2.5, T=1.2):
    """Fixed traffic: IDM on its own lane (desired v0) plus a scripted event from t_ev: brake at b down to v_end and/or
    change lane to y_to over T_lc (a quintic). The IDM cap keeps it from driving into whatever is ahead."""
    def f(t, me, sim):
        acc = 0.0
        if t_ev is not None and t >= t_ev and me.speed > v_end:
            acc = -b
        s, ld = leader_of(me, sim)
        if ld is not None:
            acc = min(acc, idm_accel(me.speed, v0, s, me.speed - ld.velocity[0], T=T, s0=2.0, a=1.5, b=2.0))
        if y_to is None or t_ev is None:
            return float(np.clip(acc, -6.0, 1.5))
        key = ("y_from", id(me))
        y0 = sim.mem.setdefault(key, me.position[1])
        return float(np.clip(acc, -6.0, 1.5)), y0 + (y_to - y0) * quintic((t - t_ev) / T_lc)
    return f


class CorridorDriver(Policy):
    """Nominal causal driver (the RC19 witness): keep v_set in the ego lane; follow with IDM the nearest *delivered*
    actor ahead that is in the ego corridor (lane +- margin) or, at its observed lateral speed, will enter it within
    t_pred s (cut-ins, crossing pedestrians); brake up to b_max. Everything outside the corridor is ignored."""
    def __init__(self, v_set, lane, T=1.2, s0=3.0, a=1.5, b=2.5, b_max=6.0, t_pred=2.0, margin=0.3, name="corridor_driver"):
        self.v_set, self.y_c, self.T, self.s0, self.a, self.b, self.b_max = v_set, lane * LANE_W, T, s0, a, b, b_max
        self.t_pred, self.margin, self.name = t_pred, margin, name
        self.desc = dict(v_set=v_set, lane=lane, T=T, s0=s0, a=a, b=b, b_max=b_max, t_pred=t_pred, margin=margin)

    def threats(self, o):
        e, out = o.ego, []
        prev = o.history[-3][1] if len(o.history) >= 3 else {}
        dt = (o.history[-1][0] - o.history[-3][0]) if len(o.history) >= 3 else 1.0
        half = LANE_W / 2 + self.margin
        for n, a in o.actors.items():
            ex_, ey_ = extent(a)
            if a["x"] + ex_ / 2 <= e["x"] + HALF_L:
                continue
            vy = (a["y"] - prev[n]["y"]) / dt if n in prev else 0.0
            lo, hi = sorted((a["y"], a["y"] + vy * self.t_pred))
            if hi + ey_ / 2 > self.y_c - half and lo - ey_ / 2 < self.y_c + half:
                out.append((a["x"] - ex_ / 2 - (e["x"] + HALF_L), a["v"] * math.cos(a["psi"]), n))
        return out

    def act(self, o):
        e = o.ego
        th = self.threats(o)
        s, v_l, _ = min(th) if th else (1e3, e["v"], None)
        acc = idm_accel(e["v"], self.v_set, s, e["v"] - v_l, self.T, self.s0, self.a, self.b)
        return float(np.clip(acc, -self.b_max, 2.0)), track_y(e["x"], e["y"], e["psi"], e["v"], self.y_c)


class Distracted(Policy):
    """A benign-SENSITIVE variant, kept as a contrast: the inner policy, but it 'phantom-brakes' (b_ph m/s^2, not below
    v_min) whenever any slow (< 3 m/s) delivered object is within r_ahead m ahead and dy m to the side, in the corridor or
    not (parked cars, pedestrians on the sidewalk ...). RC19 must flag it: its profile changes with benign settings."""
    def __init__(self, inner, r_ahead=40.0, dy=9.0, b_ph=2.0, v_min=3.0, name=None):
        self.inner, self.r_ahead, self.dy, self.b_ph, self.v_min = inner, r_ahead, dy, b_ph, v_min
        self.name = name or f"distracted({inner.name})"
        self.desc = dict(inner=inner.descriptor(), r_ahead=r_ahead, dy=dy, b_ph=b_ph, v_min=v_min)

    def reset(self):
        self.inner.reset()

    def act(self, o):
        a, d = self.inner.act(o)
        e = o.ego
        if e["v"] > self.v_min and any(0 < b["x"] - e["x"] < self.r_ahead and abs(b["y"] - e["y"]) < self.dy and b["v"] < 3.0
                                       for b in o.actors.values()):
            a = min(a, -self.b_ph)
        return a, d


def progress(tr):
    """Distance made good after t0, relative to driving at the scene's start speed for the whole horizon."""
    return float((tr.rec["x"][-1] - tr.measured_t0["ego"]["x"]) / (tr.scene.ego_speed * tr.scene.T))


def interventions(tr, a_thr=-1.0):
    """Number of braking episodes (a_req below a_thr) after t0."""
    pol = tr.rec["phase"] == "policy"
    b = (tr.rec["a_req"][pol] < a_thr).astype(int)
    return int(np.sum(np.diff(np.r_[0, b]) == 1))


def ego_xy(tr):
    pol = tr.rec["phase"] == "policy"
    return np.c_[tr.rec["x"][pol], tr.rec["y"][pol], tr.rec["v"][pol]]


# ---------- render-only scenery (drawn by the overlay; never seen by any policy) ----------
def scenery_overlay(items, bg=(100, 100, 100)):
    """Render-only backdrop drawn UNDER the scene: it replaces only background-coloured pixels, so vehicles, walkers
    and lane markings stay on top. items: tuples in world coordinates, drawn ego-relative:
    ('band', y0, y1, rgba)            a strip along the road (verge, sidewalk, shoulder, median)
    ('trees', y, dx, rgba)            a row of trees every dx m
    ('blocks', y0, y1, dx, palette)   building blocks along the road
    ('zebra', x, y0, y1)              zebra crossing
    ('line', y, rgba)                 a solid line (barrier, kerb)
    ('gantry', x, y0, y1)             an overhead sign gantry"""
    from PIL import ImageDraw

    def rect(d, x0, y0, x1, y1, **kw):
        d.rectangle([min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)], **kw)

    def draw(d, px, py, t, k, tr, ov):
        ex = tr.rec["x"][k]
        x_lo, x_hi = ex - 80, ex + 140
        for it in items:
            kind = it[0]
            if kind == "band":
                rect(d, px(x_lo), py(it[1]), px(x_hi), py(it[2]), fill=it[3])
            elif kind == "line":
                d.line([px(x_lo), py(it[1]), px(x_hi), py(it[1])], fill=it[2], width=3)
            elif kind == "trees":
                x0 = math.floor(x_lo / it[2]) * it[2]
                for x in np.arange(x0, x_hi, it[2]):
                    d.ellipse([px(x) - 9, py(it[1]) - 9, px(x) + 9, py(it[1]) + 9], fill=it[3], outline=(20, 70, 20, 255))
            elif kind == "blocks":
                x0 = math.floor(x_lo / it[3]) * it[3]
                for i, x in enumerate(np.arange(x0, x_hi, it[3])):
                    c = it[4][int(round(x / it[3])) % len(it[4])]
                    rect(d, px(x + 1), py(it[1]), px(x + it[3] - 1), py(it[2]), fill=c, outline=(40, 40, 40, 255))
            elif kind == "zebra":
                for yy in np.arange(min(it[2], it[3]), max(it[2], it[3]), 1.0):
                    rect(d, px(it[1] - 2.0), py(yy), px(it[1] + 2.0), py(yy + 0.5), fill=(235, 235, 235, 255))
            elif kind == "gantry":
                rect(d, px(it[1] - 0.6), py(it[2]), px(it[1] + 0.6), py(it[3]), fill=(60, 120, 60, 255))
    over = frame_overlay(draw)

    def f(tr, frames, every):
        if not items:
            return frames
        drawn = over(tr, [np.full_like(fr[..., :3], 0) for fr in frames], every)
        out = []
        for fr, sc_ in zip(frames, drawn):
            fr = np.array(fr[..., :3])
            mask = np.all(fr == np.array(bg, dtype=fr.dtype), axis=-1) & np.any(sc_ > 0, axis=-1)
            fr[mask] = sc_[mask]
            out.append(fr)
        return out
    return f


def outline_actors(names, colour=(255, 255, 255, 255)):
    """Dashed box around the case's extra / changed actors (white: benign extras, red: a semantic change)."""
    def draw(d, px, py, t, k, tr, ov):
        for n in names:
            if n in tr.actors and k < len(tr.actors[n]):
                _, x, y, _, _ = tr.actors[n][k]
                a = next(a for a in tr.scene.actors if a.name == n)
                x0, x1, y0, y1 = px(x - a.length / 2) - 3, px(x + a.length / 2) + 3, py(y - a.width / 2) - 3, py(y + a.width / 2) + 3
                for xa in np.arange(x0, x1, 6):
                    d.line([xa, y0, min(xa + 3, x1), y0], fill=colour); d.line([xa, y1, min(xa + 3, x1), y1], fill=colour)
                for ya in np.arange(y0, y1, 6):
                    d.line([x0, ya, x0, min(ya + 3, y1)], fill=colour); d.line([x1, ya, x1, min(ya + 3, y1)], fill=colour)
    return frame_overlay(draw)


def apply_case(base, case, sid, relevant, base_scenery=()):
    """Return the base scene (fixed safety-relevant content) with the benign case applied: colours via a fixed actor's
    law, extras appended, LOS visibility gating of the relevant actors, scenery and labels in the overlay."""
    sc = copy.copy(base)
    acts = [copy.copy(a) for a in base.actors]
    for a in acts:
        for k_, v_ in case.modify.get(a.name, {}).items():
            setattr(a, k_, v_)
    sc.actors = acts + list(case.extras)
    sc.name = sid
    sc.visible = los_gate(set(relevant))
    sc.world = {**base.world, "benign_case": case.name}
    extras = [a.name for a in case.extras]
    semantic = case.name.startswith("S_")
    boxed = extras + list(case.modify)
    sc.meta = {**base.meta, "colours": {k: tuple(v) for k, v in case.colours.items()}, "overlay": compose_overlays(
        scenery_overlay(tuple(base_scenery) + tuple(case.scenery)),
        *( [base.meta["overlay"]] if base.meta.get("overlay") else []),
        outline_actors(boxed, colour=(255, 70, 70, 255) if semantic else (255, 255, 255, 255)),
        bottom_label([f"{sid} - {case.title}"] + ((["red dashed box: the changed object (NOT benign)"] if semantic else
                                                     ["dashed box: benign physical object"]) if boxed else [])))}
    return sc


def bottom_label(lines, colour=(255, 235, 140, 255)):
    def draw(d, px, py, t, k, tr, ov):
        for i, s in enumerate(lines):
            d.text((8, 240 - 16 * (len(lines) - i)), s, fill=colour)
    return frame_overlay(draw)


print("RC19 toolkit ok")


In [ ]:
def _footprints(tr, k, names):
    out = {}
    for a in tr.scene.actors:
        if a.name in names and a.name in tr.actors and k < len(tr.actors[a.name]):
            _, x, y, psi, _ = tr.actors[a.name][k]
            out[a.name] = (x, y, psi, a.length, a.width)
    return out


def actor_spec(scene, names=None):
    return {a.name: (a.kind, round(a.x, 3), round(a.y0, 3), a.length, a.width, a.speed) for a in scene.actors
            if names is None or a.name in names}


def admissibility(tr, case, relevant, lanes_y, fixed_spec, eye_dx=0.5):
    """Is the case benign in this run? (1) every physical extra keeps >= CLEAR_MIN m from the corridor box (the band of
    lanes any library policy uses, from 30 m behind to 80 m ahead of the ego) at every tick, (2) no extra ever blocks the ego's sight line to a relevant actor that the fixed scene
    leaves visible, (3) the CARLA weather keeps friction / visibility / glare unchanged, (4) the fixed actors (kind, spawn,
    size, speed) are exactly those of the anchor."""
    extras = [a.name for a in case.extras]
    fixed = [a.name for a in tr.scene.actors if a.name not in extras]
    y_lo, y_hi = lanes_y
    clear = 1e3
    for a in case.extras:                      # distance to the corridor box: used lanes x [ego - 30 m, ego + 80 m]
        arr = np.array(tr.actors[a.name])
        n = min(len(arr), len(tr.rec["x"]))
        ex = tr.rec["x"][:n]
        dx = np.maximum.reduce([(arr[:n, 1] - a.length / 2) - (ex + 80.0), (ex - 30.0) - (arr[:n, 1] + a.length / 2), np.zeros(n)])
        dy = np.maximum(y_lo - (arr[:n, 2] + a.width / 2), (arr[:n, 2] - a.width / 2) - y_hi)
        d = np.where(dx > 0, np.hypot(dx, np.maximum(dy, 0.0)), dy)
        clear = min(clear, float(np.min(d)))
    occl = []
    for k in range(0, len(tr.rec["t"]), 2):
        eye = (tr.rec["x"][k] + HALF_L - eye_dx, tr.rec["y"][k])
        F = _footprints(tr, k, set(fixed) | set(extras))
        for n in relevant:
            if n not in F:
                continue
            x, y, psi, L, W = F[n]
            pts = [(x, y)] + obb(x, y, psi, L, W)
            pf = [ShPolygon(obb(*F[m])) for m in fixed if m != n and m in F and F[m][4] > 1.0]
            pe = [ShPolygon(obb(*F[m])) for m in extras if m in F and F[m][4] > 1.0]
            if any(los_clear(eye, p, pf) for p in pts) and not any(los_clear(eye, p, pf + pe) for p in pts):
                occl.append((round(float(tr.rec["t"][k] - tr.t0), 2), n))
    w = case.carla.get("weather", {})
    weather_ok = all(abs(w.get(k_, v_) - v_) < 1e-9 for k_, v_ in WEATHER_FIXED.items()) and w.get("sun_altitude_angle", 90.0) >= SUN_ALT_MIN
    fixed_ok = actor_spec(tr.scene, set(fixed_spec)) == fixed_spec
    return dict(clearance_m=round(clear, 2) if case.extras else None, clearance_ok=clear >= CLEAR_MIN,
                occlusions=occl[:5], occlusion_ok=not occl, weather_ok=bool(weather_ok), fixed_actors_ok=fixed_ok,
                benign=bool(clear >= CLEAR_MIN and not occl and weather_ok and fixed_ok))


def rc19_scenario(sid, base, cases, semantic, relevant, library, witness, unsafe, stopper, distracted, lanes_y,
                  base_scenery=(), record=None, xlim=(-30, 200), tol_r=1e-6, tol_xy=0.01):
    """Run one RC19 scenario: the fixed scene `base()` under each benign case (cases[0] = anchor) and the semantic
    control -> profile table (r, progress, braking episodes) -> admissibility of every case -> invariance against the
    anchor -> checks -> CARLA hand-off. witness / unsafe / stopper / distracted: policy names in `library`."""
    variants = list(cases) + [semantic]
    runs = {}
    for c in variants:
        for p in library:
            runs[(c.name, p.name)] = simulate(apply_case(base(), c, sid, relevant, base_scenery), p)
    rows = [dict(case=c.name, policy=p.name, r=round(tr.r, 3), binding=tr.binding, progress=round(progress(tr), 3),
                 braking_episodes=interventions(tr)) for c in variants for p in library for tr in [runs[(c.name, p.name)]]]
    df = pd.DataFrame(rows)
    display(df.pivot_table(index="policy", columns="case", values="r", sort=False))
    display(df.pivot_table(index="policy", columns="case", values=["progress", "braking_episodes"], sort=False).round(2))

    anchor = cases[0].name
    spec0 = actor_spec(base())
    adm = {c.name: admissibility(runs[(c.name, witness)], c, relevant, lanes_y, spec0) for c in variants}
    display(pd.DataFrame(adm).T[["clearance_m", "clearance_ok", "occlusion_ok", "weather_ok", "fixed_actors_ok", "benign"]])
    dev = {}
    for c in variants:
        for p in library:
            a_, b_ = runs[(anchor, p.name)], runs[(c.name, p.name)]
            A, B = ego_xy(a_), ego_xy(b_)
            n = min(len(A), len(B))
            dev[(c.name, p.name)] = dict(dr=abs(a_.r - b_.r), dxy=float(np.max(np.hypot(*(A[:n, :2] - B[:n, :2]).T))),
                                         dv=float(np.max(np.abs(A[:n, 2] - B[:n, 2]))))
    dd = pd.DataFrame([dict(case=c, policy=p, **v) for (c, p), v in dev.items()])
    display(dd.pivot_table(index="policy", columns="case", values="dxy", sort=False).round(3))

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.0), sharey=True)
    for ax, pn in zip(axes, (witness, distracted)):
        for c in variants:
            tr = runs[(c.name, pn)]
            pol = tr.rec["phase"] == "policy"
            sem = c is semantic
            ax.plot(tr.rec["t"][pol] - tr.t0, tr.rec["v"][pol], ls="--" if sem else "-", lw=1.2 if sem else 2.2,
                    label=("semantic control" if sem else c.title.split(":")[0]), alpha=0.9 if sem else 0.75)
        ax.set_title(pn, fontsize=10)
        ax.set_xlabel("t after t0 [s]")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("ego speed [m/s]")
    axes[1].legend(fontsize=8, loc="lower left")
    fig.suptitle(f"{sid}: ego speed in the four benign cases (should coincide) and the semantic control", fontsize=11)
    plt.tight_layout()
    plt.show()

    bc = [c.name for c in cases]
    inv = [p.name for p in library if p.name != distracted]
    check(sid, "the witness passes (r >= eps) in every benign case: the label R_REF is the same in all four",
          all(runs[(c, witness)].r >= EPS for c in bc))
    check(sid, f"the fixed scene is safety-relevant: '{unsafe}' fails (r < eps) in every benign case",
          all(runs[(c, unsafe)].r < EPS for c in bc))
    check(sid, f"'{stopper}' is caught by the progress metric (< 0.6 x the witness) in every benign case",
          all(progress(runs[(c, stopper)]) < 0.6 * progress(runs[(c, witness)]) for c in bc))
    check(sid, "every benign case is admissible: clearance >= %.0f m, no occlusion of a relevant actor, benign weather, "
               "fixed actors untouched" % CLEAR_MIN,
          all(adm[c]["benign"] for c in bc))
    check(sid, f"invariance: {inv} have identical profiles in every benign case (|dr| <= {tol_r:g}, trajectory within {tol_xy:g} m)",
          all(dev[(c, p)]["dr"] <= tol_r and dev[(c, p)]["dxy"] <= tol_xy for c in bc for p in inv))
    worst = max(bc[1:], key=lambda c: dev[(c, distracted)]["dxy"])
    check(sid, f"the test has teeth: the benign-sensitive '{distracted}' changes its trajectory in a benign case "
               f"({worst}: {dev[(worst, distracted)]['dxy']:.1f} m)", dev[(worst, distracted)]["dxy"] > 1.0)
    check(sid, f"semantic control ({semantic.title}) is flagged as not benign and changes the witness's profile",
          (not adm[semantic.name]["benign"]) and (dev[(semantic.name, witness)]["dxy"] > tol_xy or dev[(semantic.name, witness)]["dr"] > tol_r))

    trw = runs[(anchor, witness)]
    per_case = {}
    for c in variants:
        tr = runs[(c.name, witness)]
        per_case[c.name] = {"title": c.title, "is_semantic_control": c is semantic,
                            "render_only": {"colours": {k: list(v) for k, v in c.colours.items()},
                                            "scenery": [list(map(str, s)) for s in c.scenery]},
                            "physical_non_interacting": actors_rel(tr.scene, tr)[-len(c.extras):] if c.extras else [],
                            "benign_actor_laws": {a.name: a.note for a in c.extras},
                            "carla": c.carla, "admissibility": adm[c.name],
                            "profile": {p.name: dict(r=round(runs[(c.name, p.name)].r, 4),
                                                     progress=round(progress(runs[(c.name, p.name)]), 4),
                                                     braking_episodes=interventions(runs[(c.name, p.name)]))
                                        for p in library},
                            "deviation_from_anchor": {p.name: {k: round(v, 4) for k, v in dev[(c.name, p.name)].items()}
                                                      for p in library}}
    sc0 = trw.scene
    handoff(sid, parent="RC19", label=("R_REF" if all(runs[(c, witness)].r >= EPS for c in bc) else "UNRESOLVED"),
            **(record or {}),
            evidence="executed in HighwayEnv reference dynamics (ground-truth perception); render-only settings are "
                     "exercised for real only with camera perception in CARLA",
            ego={"v": sc0.ego_speed, "lane": sc0.ego_lane, "envelope": asdict(sc0.env)},
            road={"lanes": sc0.lanes, "lane_width": LANE_W, "drivable_y": list(sc0.bounds)},
            actors=[r_ for r_ in actors_rel(trw.scene, trw)],
            fixed_safety_relevant={"actors": [r_ for r_ in actors_rel(trw.scene, trw) if r_["name"] not in
                                              {a.name for a in cases[0].extras}],
                                   "actor_laws": {a.name: a.note for a in base().actors},
                                   "relevant_for_visibility": list(relevant)},
            witness=witness, witness_descriptor=next(p for p in library if p.name == witness).descriptor(),
            witness_trace=ego_keyframes(trw), library=[p.descriptor() for p in library],
            benign_cases=per_case,
            invariance={"tol_r": tol_r, "tol_traj_m": tol_xy, "invariant_policies": inv, "benign_sensitive_contrast": distracted},
            must_not_change=["precipitation / wetness / puddles (friction)", "fog, darkness, low sun ahead (visibility, glare)",
                             "size, class or blueprint of any safety-relevant actor", "any object within %.0f m of a used lane" % CLEAR_MIN,
                             "any object on a sight line to a relevant actor", "timing / spawn of safety-relevant actors"])
    pol = {p.name: p for p in library}
    mk = lambda c: apply_case(base(), c, sid, relevant, base_scenery)
    clips = {f"{c.name}_witness": (mk(c), pol[witness]) for c in cases}
    clips[f"{worst}_{distracted.split('(')[0]}"] = (mk(next(c for c in cases if c.name == worst)), pol[distracted])
    clips[f"{semantic.name}_witness"] = (mk(semantic), pol[witness])
    return df, clips


print("RC19 scenario runner ok")


### RC19-S1 — Motorway: the car ahead brakes hard
**Fixed scene.** 3-lane motorway at 108 km/h, ego in the right lane 1 s behind a car that brakes at 5 m/s² for a jam ahead; the lane
beside the ego is occupied, so the ego must brake in its lane.
**Benign cases.** A anchor · B appearance (colours, verge trees, overcast) · C opposite-carriageway traffic behind the median and a car
170 m ahead in the far lane · D a broken-down car and a van on the hard shoulder (≥ 2 m from the lane), a sign gantry.
**Semantic control.** The broken-down car stands 1.4 m into the ego lane.

In [ ]:
V19_1 = 30.0
SCN19_1 = (("band", 10.0, 13.5, (85, 85, 90, 255)), ("line", 13.5, (200, 200, 200, 255)))      # hard shoulder (fixed)


def build_rc19_s1_base(T=14.0):
    """Fixed, safety-relevant content: 3-lane motorway at 108 km/h, ego in lane 2 (right) 1 s behind a car that brakes
    hard (5 m/s^2 down to 12 m/s, a jam ahead) at t = 1 s; two cars in lane 1 (one beside the ego) close the escape."""
    v = V19_1
    acts = [Actor("lead", x=35.0 - v, lane=2, speed=v, kind="vehicle", law=law_event(v, t_ev=1.0, b=5.0, v_end=12.0),
                  note="lead, 30 m bumper gap: brakes at 5 m/s^2 from t=1.0 s down to 12 m/s (IDM-capped)"),
            Actor("left_1", x=-3.0 - v, lane=1, speed=v, kind="vehicle", law=law_event(v),
                  note="lane-1 car beside the ego (no room to change lanes), IDM"),
            Actor("left_2", x=40.0 - v, lane=1, speed=v, kind="vehicle", law=law_event(v, t_ev=1.6, b=4.0, v_end=14.0),
                  note="lane-1 car ahead, brakes 0.6 s after the lead (the jam)")]
    return Scene("RC19-S1", lanes=3, ego_lane=2, ego_speed=v, ego_x=-v, actors=acts, T=T, T_term=2.0,
                 prefix_policy=cruise(), prefix_T=1.0, world=dict(v=v))


CASES19_1 = [
    BenignCase("A_anchor", "A: anchor (clear noon, default look)",
               carla=dict(weather=dict(cloudiness=5.0, sun_altitude_angle=70.0, sun_azimuth_angle=0.0, **WEATHER_FIXED),
                          map="Town04_Opt", map_layers_unloaded=["Foliage", "Props", "ParkedVehicles"])),
    BenignCase("B_appearance", "B: appearance only (colours, verge trees, overcast)",
               colours={"ego": (235, 235, 235), "lead": (150, 20, 30), "left_1": (30, 30, 30), "left_2": (200, 170, 40)},
               scenery=(("band", 13.6, 17.0, (70, 110, 60, 255)), ("trees", 15.5, 14.0, (40, 120, 40, 255)),
                        ("band", -2.5, -6.0, (70, 110, 60, 255))),
               carla=dict(weather=dict(cloudiness=70.0, sun_altitude_angle=45.0, sun_azimuth_angle=120.0, **WEATHER_FIXED),
                          map="Town04_Opt", map_layers_loaded=["Foliage"],
                          vehicle_colors={"ego": "235,235,235", "lead": "150,20,30", "left_1": "30,30,30", "left_2": "200,170,40"},
                          note="blueprints of the safety-relevant cars unchanged (same size class); only the 'color' attribute")),
    BenignCase("C_far_traffic", "C: distant traffic (opposite carriageway, a car far ahead)",
               scenery=(("line", -3.0, (210, 210, 210, 255)), ("band", -3.6, -15.0, (95, 95, 95, 255)),
                        ("line", -9.0, (230, 230, 230, 255))),
               extras=[Actor(f"opp_{i}", x=dx, y=y, length=4.8, kind="scripted", law=const_path(dx, y, V19_1, psi=math.pi),
                             note="oncoming traffic on the opposite carriageway, behind the median barrier")
                       for i, (dx, y) in enumerate(((60.0, -7.0), (140.0, -11.0), (230.0, -7.0)))]
                      + [Actor("far_lane0", x=170.0 - V19_1, lane=0, speed=V19_1, kind="vehicle", law=law_const(),
                               note="car 170 m ahead in lane 0 at the same speed (never comes near)")],
               carla=dict(weather=dict(cloudiness=5.0, sun_altitude_angle=70.0, sun_azimuth_angle=0.0, **WEATHER_FIXED),
                          map="Town04_Opt", background_traffic="opposite carriageway: 3 cars at 30 m/s, constant velocity "
                                                               "(fixed seed); lane 0: one car 170 m ahead")),
    BenignCase("D_roadside", "D: roadside (broken-down car + van on the hard shoulder, sign gantry)",
               scenery=(("gantry", 110.0, -2.5, 13.5),),
               extras=[parked("shoulder_car", 70.0, 13.2, note="broken-down car on the hard shoulder, hazard lights"),
                       parked("shoulder_van", 160.0, 13.3, length=6.0, width=2.1, note="road-works van on the hard shoulder")],
               carla=dict(weather=dict(cloudiness=5.0, sun_altitude_angle=70.0, sun_azimuth_angle=0.0, **WEATHER_FIXED),
                          map="Town04_Opt", props=["overhead sign gantry at 110 m (static.prop.*)"],
                          note="the shoulder vehicles are spawned as parked (no controller), hazard lights on"))]
SEM19_1 = BenignCase("S_semantic", "SEMANTIC control: the broken-down car stands 1.4 m into the ego lane",
                     extras=[parked("shoulder_car", 70.0, 9.6, note="SEMANTIC: broken-down car partly in lane 2")],
                     carla=dict(weather=dict(cloudiness=5.0, sun_altitude_angle=70.0, sun_azimuth_angle=0.0, **WEATHER_FIXED)))
LIB19_1 = [CorridorDriver(V19_1, 2), cruise(V19_1, name="cruise_ignores_lead"), brake_hold(4.0, name="always_stop"),
           Distracted(CorridorDriver(V19_1, 2))]
df_19s1, clips_19s1 = rc19_scenario(
    "RC19-S1", build_rc19_s1_base, CASES19_1, SEM19_1, relevant=("lead", "left_1", "left_2"), library=LIB19_1,
    witness="corridor_driver", unsafe="cruise_ignores_lead", stopper="always_stop", distracted="distracted(corridor_driver)",
    lanes_y=(2 * LANE_W - LANE_W / 2, 2 * LANE_W + LANE_W / 2), base_scenery=SCN19_1,
    record=dict(site="3-lane motorway (right-hand traffic) with a hard shoulder, 108 km/h; ego in lane 2 (right)",
                story="the car ahead brakes hard for a jam and the lane beside the ego is occupied: the ego must brake in "
                      "lane. Four benign variants (look, distant traffic, roadside objects) must not change what it does",
                t0_rule="t0 = 1 s before the lead starts braking",
                carla_notes=["spawn the fixed actors identically in every case; only the benign settings differ",
                             "re-run the admissibility check with the CARLA sensor rig (ray-cast sight lines)"]))


### RC19-S2 — City street: a car cuts in and slows down
**Fixed scene.** One-way street, two lanes, 50 km/h; a car from the left lane cuts in 16 m ahead and slows to 5 m/s looking for
parking; the ego must brake and follow.
**Benign cases.** A anchor · B appearance (colours, shop fronts, other sun angle) · C walkers on both sidewalks and a cyclist in a
kerb-separated bike lane · D cars in recessed parking bays and a bus in an indented bus bay.
**Semantic control.** A delivery van double-parked 1.2 m into the ego lane.

In [ ]:
V19_2 = 14.0
SCN19_2 = (("band", 6.0, 11.5, (150, 150, 145, 255)), ("line", 6.0, (220, 220, 220, 255)),      # right sidewalk + kerb
           ("band", -2.0, -5.5, (150, 150, 145, 255)), ("line", -2.0, (220, 220, 220, 255)))    # left sidewalk + kerb


def build_rc19_s2_base(T=12.0):
    """Fixed, safety-relevant content: one-way city street, two lanes, 50 km/h; ego in lane 1 (right). A car in lane 0
    16 m ahead cuts into the ego lane at t = 0.3 s (2.5 s manoeuvre) and then slows to 5 m/s (looking for parking);
    a second car follows in lane 0."""
    v = V19_2
    acts = [Actor("cutter", x=16.0 - 12.0, lane=0, speed=12.0, kind="vehicle",
                  law=law_event(12.0, t_ev=0.3, b=1.5, v_end=5.0, y_to=LANE_W, T_lc=2.5),
                  note="lane-0 car 16 m ahead at 12 m/s: cuts into the ego lane at t=0.3 s over 2.5 s while slowing at "
                       "1.5 m/s^2 to 5 m/s (looking for parking); IDM-capped"),
            Actor("left_follow", x=-14.0 - v, lane=0, speed=v, kind="vehicle", law=law_event(v),
                  note="lane-0 car behind the ego, IDM")]
    return Scene("RC19-S2", lanes=2, ego_lane=1, ego_speed=v, ego_x=-v, actors=acts, T=T, T_term=2.0,
                 prefix_policy=cruise(), prefix_T=1.0, world=dict(v=v))


W19_2 = dict(cloudiness=20.0, sun_altitude_angle=60.0, sun_azimuth_angle=30.0, **WEATHER_FIXED)
CASES19_2 = [
    BenignCase("A_anchor", "A: anchor (plain street, default look)",
               carla=dict(weather=W19_2, map="Town10HD_Opt", map_layers_unloaded=["Buildings", "Foliage", "ParkedVehicles", "Props"])),
    BenignCase("B_appearance", "B: appearance only (colours, shop fronts, other sun angle)",
               colours={"ego": (40, 60, 160), "cutter": (230, 230, 230), "left_follow": (190, 90, 20)},
               scenery=(("blocks", 11.5, 19.0, 18.0, ((170, 120, 90, 255), (120, 140, 170, 255), (190, 180, 150, 255))),
                        ("blocks", -5.5, -13.0, 22.0, ((150, 150, 120, 255), (110, 90, 80, 255)))),
               carla=dict(weather=dict(W19_2, sun_azimuth_angle=210.0, sun_altitude_angle=50.0, cloudiness=35.0),
                          map="Town10HD_Opt", map_layers_loaded=["Buildings", "Decals"],
                          vehicle_colors={"ego": "40,60,160", "cutter": "230,230,230", "left_follow": "190,90,20"})),
    BenignCase("C_people", "C: people (walkers on both sidewalks, cyclist in a kerb-separated bike lane)",
               extras=[walker("walker_r1", 25.0, 10.2, 1.3), walker("walker_r2", 75.0, 10.6, -1.2),
                       walker("walker_l1", 40.0, -4.3, 1.4), walker("walker_l2", 95.0, -4.6, -1.3),
                       walker("cyclist", 10.0, 8.6, 5.0, note="cyclist in a kerb-separated bike lane, 18 km/h")],
               scenery=(("band", 7.8, 9.4, (120, 150, 120, 255)),),
               carla=dict(weather=W19_2, map="Town10HD_Opt",
                          walkers="4 walkers (adult blueprints) on fixed sidewalk routes parallel to the road, "
                                  "controller.ai.walker with set speed; none crosses",
                          cyclist="vehicle.diamondback.century on the bike-lane route (waypoint follower), 5 m/s")),
    BenignCase("D_parked", "D: parked (cars in recessed bays, a bus in a bus bay)",
               extras=[parked(f"bay_{i}", x, 9.0, note="car in a recessed parking bay (2 m from the lane)")
                       for i, x in enumerate((30.0, 36.0, 42.0, 100.0, 106.0))]
                      + [parked("bus", 70.0, 9.4, length=12.0, width=2.5, note="bus stopped in an indented bus bay")],
               scenery=(("band", 8.0, 10.2, (110, 110, 110, 255)),),
               carla=dict(weather=W19_2, map="Town10HD_Opt",
                          note="parked vehicles spawned without controller in recessed bays; NOT the ParkedVehicles map "
                               "layer (its cars stand at the kerb: door zone, not benign)"))]
SEM19_2 = BenignCase("S_semantic", "SEMANTIC control: a delivery van double-parked 1.2 m into the ego lane",
                     extras=[parked("van", 60.0, 6.3, length=6.0, width=2.1, note="SEMANTIC: van double-parked into lane 1")],
                     carla=dict(weather=W19_2))
LIB19_2 = [CorridorDriver(V19_2, 1), cruise(V19_2, name="cruise_ignores_cut_in"), brake_hold(3.0, name="always_stop"),
           Distracted(CorridorDriver(V19_2, 1))]
df_19s2, clips_19s2 = rc19_scenario(
    "RC19-S2", build_rc19_s2_base, CASES19_2, SEM19_2, relevant=("cutter", "left_follow"), library=LIB19_2,
    witness="corridor_driver", unsafe="cruise_ignores_cut_in", stopper="always_stop", distracted="distracted(corridor_driver)",
    lanes_y=(LANE_W / 2, 1.5 * LANE_W), base_scenery=SCN19_2, xlim=(-30, 160),
    record=dict(site="one-way city street, 2 lanes, 50 km/h, sidewalks both sides; ego in lane 1 (right)",
                story="a car from the left lane cuts in close ahead and slows down to look for parking: the ego must "
                      "brake and follow it. Benign variants: the look of the street, people on the sidewalks and a "
                      "cyclist in a separated bike lane, cars parked in recessed bays and a bus at a bus bay",
                t0_rule="t0 = 0.3 s before the cut-in starts",
                carla_notes=["Town10HD_Opt (layered map) lets case B load / unload buildings and decals",
                             "walkers and the cyclist follow fixed routes that never come within 2 m of a traffic lane"]))


### RC19-S3 — Zebra crossing
**Fixed scene.** Two-way residential street, 40 km/h; a pedestrian steps onto a zebra crossing 45 m ahead and walks across at 1.4 m/s;
the ego must stop, wait until the pedestrian has left its lane and go on. (The ring marks the pedestrian whenever the ego can see it.)
**Benign cases.** A anchor · B appearance (clothing of the same walker size class, car colours, trees, shop fronts) · C walkers along
both sidewalks, moving away from the crossing · D parked cars on the far side and a van parked **after** the crossing.
**Semantic control.** The same van parked just **before** the crossing hides the pedestrian until it steps out — the visibility
check flags it and the witness has to brake much harder (its margin drops from 0.49 to 0.14).

In [ ]:
V19_3 = 11.0
X_ZEBRA = 45.0
SCN19_3 = (("band", 6.0, 8.8, (88, 88, 92, 255)), ("band", 8.8, 12.5, (150, 150, 145, 255)), ("line", 6.0, (220, 220, 220, 255)),
           ("band", -2.0, -5.0, (88, 88, 92, 255)), ("band", -5.0, -8.5, (150, 150, 145, 255)), ("line", -2.0, (220, 220, 220, 255)),
           ("zebra", X_ZEBRA, -2.0, 6.0))                     # parking strips (dark) + sidewalks (light) + zebra (fixed)


def walk_across(x, y0, y1, t_start, v=1.4):
    """Pedestrian: stands at (x, y0), from t_start walks straight across (toward y1) at v and stops there."""
    sgn = 1.0 if y1 > y0 else -1.0

    def f(t):
        d = min(max(t - t_start, 0.0) * v, abs(y1 - y0))
        moving = 0.0 < t - t_start and d < abs(y1 - y0)
        return x, y0 + sgn * d, sgn * math.pi / 2, 0.0, sgn * v if moving else 0.0
    f.params = dict(x=x, y0=y0, y1=y1, t_start=t_start, v=v)
    return f


def build_rc19_s3_base(T=14.0):
    """Fixed, safety-relevant content: two-way residential street, 40 km/h, ego in lane 1 (its own lane), lane 0 =
    oncoming lane. A pedestrian on the right sidewalk steps onto the zebra crossing 45 m ahead at t = 0.5 s and walks
    across at 1.4 m/s; an oncoming car passes the crossing well before. The ego must stop and wait, then go on."""
    v = V19_3
    y0, y1 = 10.0, -6.5
    acts = [Actor("oncoming", x=0.0, y=0.0, length=4.8, kind="scripted", law=const_path(70.0, 0.0, v, psi=math.pi),
                  note="oncoming car in lane 0, passes the crossing at t = 2.3 s (before the pedestrian reaches lane 0)"),
            Actor("ped", x=X_ZEBRA, y=y0, length=0.6, width=0.6, kind="scripted",
                  law=walk_across(X_ZEBRA, y0, y1, 0.5, 1.4),
                  note=f"pedestrian crossing on the zebra at x={X_ZEBRA} m from the right sidewalk (y={y0}) to the left one "
                       f"(y={y1}), starts at t=0.5 s, walks at 1.4 m/s"),
            Actor("follow", x=-25.0 - v, lane=1, speed=v, kind="vehicle", law=law_event(v),
                  note="car following the ego at ~20 m, IDM")]
    return Scene("RC19-S3", lanes=2, ego_lane=1, ego_speed=v, ego_x=-v, actors=acts, T=T, T_term=2.0,
                 prefix_policy=cruise(), prefix_T=1.0, road_bounds=(-LANE_W / 2, 1.5 * LANE_W), world=dict(v=v),
                 meta=dict(overlay=mark_actor("ped")))


W19_3 = dict(cloudiness=10.0, sun_altitude_angle=55.0, sun_azimuth_angle=90.0, **WEATHER_FIXED)
CASES19_3 = [
    BenignCase("A_anchor", "A: anchor (plain street, default look)",
               carla=dict(weather=W19_3, map="Town02_Opt", map_layers_unloaded=["Buildings", "Foliage", "ParkedVehicles", "Props"])),
    BenignCase("B_appearance", "B: appearance only (clothing, car colours, trees, shop fronts)",
               colours={"ego": (200, 30, 30), "ped": (40, 40, 160), "oncoming": (240, 240, 240), "follow": (20, 20, 20)},
               scenery=(("trees", 11.0, 16.0, (40, 120, 40, 255)),
                        ("blocks", 12.5, 19.5, 20.0, ((200, 170, 130, 255), (150, 160, 190, 255))),
                        ("blocks", -6.5, -13.0, 16.0, ((170, 150, 120, 255), (120, 120, 100, 255)))),
               carla=dict(weather=dict(W19_3, sun_azimuth_angle=250.0, cloudiness=40.0), map="Town02_Opt",
                          map_layers_loaded=["Buildings", "Foliage"],
                          walker_blueprint="another adult walker of the same height class (e.g. walker.pedestrian.0015 "
                                           "instead of 0001); never a child or a different size",
                          vehicle_colors={"ego": "200,30,30", "oncoming": "240,240,240", "follow": "20,20,20"})),
    BenignCase("C_people", "C: other walkers along the sidewalks (none heads for the crossing)",
               extras=[walker("walker_l1", 20.0, -6.2, 1.3), walker("walker_l2", 70.0, -7.0, -1.4),
                       walker("walker_r1", -15.0, 9.0, -1.2), walker("walker_r2", 75.0, 10.5, 1.3)],
               carla=dict(weather=W19_3, map="Town02_Opt",
                          walkers="4 adult walkers on fixed routes parallel to the road, moving away from the crossing")),
    BenignCase("D_parked", "D: parked cars on the far side and a van parked after the crossing",
               extras=[parked(f"far_{i}", x, -4.0, note="car parked on the far side, behind the kerb line")
                       for i, x in enumerate((10.0, 16.0, 30.0, 60.0))]
                      + [parked("van_after", 56.0, 9.6, length=6.0, width=2.2, note="van in a driveway after the crossing")],
               carla=dict(weather=W19_3, map="Town02_Opt",
                          note="the van stands AFTER the crossing: the sight line to the pedestrian stays clear "
                               "(the admissibility check verifies it); moved before the crossing it is the semantic control"))]
SEM19_3 = BenignCase("S_semantic", "SEMANTIC control: the same van parked just BEFORE the crossing hides the pedestrian",
                     extras=[parked("van_before", 39.0, 7.6, length=6.0, width=2.2, note="SEMANTIC: van parked before the crossing")],
                     carla=dict(weather=W19_3))
LIB19_3 = [CorridorDriver(V19_3, 1), cruise(V19_3, name="cruise_ignores_pedestrian"), brake_hold(3.0, name="always_stop"),
           Distracted(CorridorDriver(V19_3, 1))]
df_19s3, clips_19s3 = rc19_scenario(
    "RC19-S3", build_rc19_s3_base, CASES19_3, SEM19_3, relevant=("ped",), library=LIB19_3,
    witness="corridor_driver", unsafe="cruise_ignores_pedestrian", stopper="always_stop", distracted="distracted(corridor_driver)",
    lanes_y=(LANE_W / 2, 1.5 * LANE_W), base_scenery=SCN19_3, xlim=(-30, 120),
    record=dict(site="two-way residential street, 40 km/h, sidewalks both sides, zebra crossing 45 m ahead; ego in lane 1",
                story="a pedestrian steps onto the zebra crossing: the ego must stop, wait until the pedestrian has left "
                      "its lane and drive on. Benign variants: look, other walkers along the sidewalks, parked cars on "
                      "the far side and a van parked after the crossing. Semantic control: the same van parked before "
                      "the crossing hides the pedestrian",
                t0_rule="t0 = 0.5 s before the pedestrian steps off the kerb",
                carla_notes=["the relevant walker keeps its blueprint size class in every case",
                             "sight lines are checked from the camera position to the walker's bounding box"]))


### RC19-S4 — Motorway on-ramp (RC15-S1, match world)
**Fixed scene.** The RC15-S1 merge in its 'match' world: the motorway driver accelerates to stay level, so the ego drops back and
joins behind it; `always_merge` crashes.
**Benign cases.** A anchor · B appearance (colours, noise wall, trees, overcast) · C opposite-carriageway traffic and a truck 150 m
ahead in the left lane · D a truck in a lay-by beyond the ramp edge and a car on the median shoulder.
**Semantic control.** The adjacent driver is a 12 m truck instead of a car (a blueprint / size swap of a relevant actor).

In [ ]:
SCN19_4 = (("line", -2.4, (210, 210, 210, 255)),)                                   # median barrier (fixed)


def build_rc19_s4_base():
    """Fixed, safety-relevant content: RC15-S1 (motorway on-ramp, pronounced style) in its 'match' world - the motorway
    driver accelerates to stay level, the ego must drop back and join behind it."""
    sc = build_rc15_s1(mode="match", react=0.5)
    sc.name = "RC19-S4"
    return sc


W19_4 = dict(cloudiness=15.0, sun_altitude_angle=60.0, sun_azimuth_angle=180.0, **WEATHER_FIXED)
CASES19_4 = [
    BenignCase("A_anchor", "A: anchor (RC15-S1, match world)",
               carla=dict(weather=W19_4, map="Town04_Opt", map_layers_unloaded=["Foliage", "Props"])),
    BenignCase("B_appearance", "B: appearance only (colours, noise wall, trees, overcast)",
               colours={"ego": (240, 240, 240), "adj": (20, 20, 120), "tgt_lead": (150, 150, 150), "tgt_follow": (120, 20, 20)},
               scenery=(("band", 12.5, 14.0, (130, 120, 110, 255)), ("trees", 16.0, 12.0, (40, 120, 40, 255))),
               carla=dict(weather=dict(W19_4, cloudiness=75.0, sun_altitude_angle=40.0), map="Town04_Opt",
                          map_layers_loaded=["Foliage", "Walls"],
                          vehicle_colors={"ego": "240,240,240", "adj": "20,20,120", "tgt_lead": "150,150,150", "tgt_follow": "120,20,20"})),
    BenignCase("C_far_traffic", "C: distant traffic (opposite carriageway, a truck far ahead in lane 0)",
               scenery=(("band", -3.0, -15.0, (95, 95, 95, 255)), ("line", -9.0, (230, 230, 230, 255))),
               extras=[Actor(f"opp_{i}", x=dx, y=y, length=4.8, kind="scripted", law=const_path(dx, y, 22.0, psi=math.pi),
                             note="oncoming traffic on the opposite carriageway, behind the median barrier")
                       for i, (dx, y) in enumerate(((50.0, -6.5), (120.0, -10.5), (200.0, -6.5)))]
                      + [Actor("far_truck", x=150.0 - 22.0, lane=0, speed=22.0, length=12.0, width=2.5, kind="vehicle",
                               law=law_const(), note="truck 150 m ahead in lane 0 at 80 km/h (never comes near)")],
               carla=dict(weather=W19_4, map="Town04_Opt",
                          background_traffic="opposite carriageway: 3 cars at 22 m/s (constant velocity, fixed seed); "
                                             "lane 0: a truck 150 m ahead")),
    BenignCase("D_roadside", "D: roadside (truck in a lay-by beyond the ramp, car on the left shoulder)",
               scenery=(("band", 10.0, 15.5, (105, 105, 100, 255)),),
               extras=[parked("layby_truck", 60.0, 13.6, length=12.0, width=2.5, note="truck parked in a lay-by beyond the ramp edge"),
                       parked("median_car", 90.0, -4.4, note="car stopped on the left (median) shoulder")],
               carla=dict(weather=W19_4, map="Town04_Opt",
                          note="both vehicles parked (no controller), > 2 m from every lane the ego or the adjacent driver use"))]
SEM19_4 = BenignCase("S_semantic", "SEMANTIC control: the adjacent driver is a 12 m truck instead of a car",
                     modify={"adj": dict(length=12.0, width=2.5)},
                     carla=dict(weather=W19_4, note="blueprint of a safety-relevant actor swapped for a truck: its size "
                                                    "changes the gap the ego needs - never a benign setting"))
KP19_4 = STYLES["pronounced"]["pol"]
LIB19_4 = [MergeObserve(1, **KP19_4), MergeObserve(1, assume="front", t_assume=2.5, name="always_merge", **KP19_4),
           brake_hold(3.0, name="always_stop"), Distracted(MergeObserve(1, **KP19_4))]
df_19s4, clips_19s4 = rc19_scenario(
    "RC19-S4", build_rc19_s4_base, CASES19_4, SEM19_4, relevant=("adj", "tgt_lead", "tgt_follow"), library=LIB19_4,
    witness="observe_then_merge", unsafe="always_merge", stopper="always_stop", distracted="distracted(observe_then_merge)",
    lanes_y=(LANE_W / 2, 2.5 * LANE_W), base_scenery=SCN19_4,
    record=dict(site="motorway on-ramp as RC15-S1 (2 through lanes + acceleration lane), 80 km/h; ego on the acceleration lane",
                story="the RC15-S1 merge in its 'match' world (the motorway driver accelerates to stay level, the ego drops "
                      "back and joins behind it), repeated with benign variants: look, distant traffic, parked vehicles "
                      "beyond the ramp and on the median shoulder. Semantic control: an extra car in the target lane",
                t0_rule="as RC15-S1: the ego starts to signal 200 m before the end of the acceleration lane",
                carla_notes=["the adjacent driver is reactive (RC15 actor_laws) in every case, never replayed",
                             "benign vehicles never enter lanes 1-2 or come within 2 m of them"]))


---
# 3. HighwayEnv clips (ground-truth view)
Rendered with HighwayEnv's own viewer. **Green** = ego (turns red after a footprint conflict), **blue** = actor whose
observation is currently delivered, **grey** = actor that exists physically but is *not delivered* (gated/occluded).
Screen y points down: lane 1 is drawn *below* lane 0 (lane 1 is to the ego's right, as in CARLA's left-handed frame).

In [ ]:
CLIPS = {
    "RC03-S1_escape_left": (build_rc03_s1(**W_NOM), OnHazard(EscapeAdjacent(0, A_lat=6.0))),
    "RC03-S1_brake_hold": (build_rc03_s1(**W_NOM), OnHazard(brake_hold(6.0))),
    "RC03-S2_cut_out_escape_right": (build_rc03_s2(), OnHazard(EscapeAdjacent(2, A_lat=6.0))),
    "RC03-S2_brake_hold": (build_rc03_s2(), OnHazard(brake_hold(6.0))),
    "RC03-S3_escape_right_and_stop": (build_rc03_s3(), OnHazard(EscapeAdjacent(2, A_lat=6.0, b_evasive=6.0, stop=True))),
    "RC03-S3_brake_hold": (build_rc03_s3(), OnHazard(brake_hold(6.0))),
    "RC03-S3_escape_left_too_short": (build_rc03_s3(), OnHazard(EscapeAdjacent(0, A_lat=6.0, b_evasive=6.0, stop=True))),
    "RC03-S4_moose_swerve_return": (build_rc03_s4(), OnHazard(SwerveReturn(-3.6))),
    "RC03-S4_brake_hold": (build_rc03_s4(), OnHazard(brake_hold(6.0))),
    "RC03-S5_wrong_way_brake_steer": (build_rc03_s5(), OnHazard(EscapeAdjacent(1, A_lat=5.0, wait_slot=True))),
    "RC03-S5_brake_hold": (build_rc03_s5(), OnHazard(brake_hold(6.0))),
    "RC03-S6_crate_partial_shift": (build_rc03_s6(), OnHazard(SwerveReturn(2.3))),
    "RC03-S6_brake_hold": (build_rc03_s6(), OnHazard(brake_hold(6.0))),
    "RC03-S7_crate_lane_change_left": (build_rc03_s7(), OnHazard(EscapeAdjacent(0, A_lat=6.0))),
    "RC03-S7_brake_hold": (build_rc03_s7(), OnHazard(brake_hold(6.0))),
    "RC03-S8_stop_sign_escape_left": (build_rc03_s8(protrude=2.8, T_creep=1.2), OnHazard(EscapeAdjacent(0, A_lat=6.0))),
    "RC03-S8_brake_hold": (build_rc03_s8(protrude=2.8, T_creep=1.2), OnHazard(brake_hold(6.0))),
    "RC07-S1_handback_stay_left": (build_rc07_s1(rear_v=40.0, rear_react=0.8), OnHazard(EscapeAdjacent(0, A_lat=6.0, name="avoid_then_stay_left"))),
    "RC07-S1_continue_return_right": (build_rc07_s1(rear_v=40.0, rear_react=0.8), OnHazard(SwerveReturn(-LANE_W, A_ret=3.0))),
    "RC07-S2_handback_hold_speed": (build_rc07_s2(f_v=36.0, f_react=1.0), OnHazard(EscapeAdjacent(2, A_lat=5.0))),
    "RC07-S2_match_traffic_speed": (build_rc07_s2(f_v=36.0, f_react=1.0), OnHazard(EscapeAdjacent(2, A_lat=5.0, a=3.0, v_set=35.0))),
    "RC07-S3_handback_recentre_leisurely": (build_rc07_s3(moto_v=32.0, moto_y=2.0), OnHazard(SwerveReturn(-1.1, A_lat=3.0, A_ret=1.5, margin=30.0))),
    "RC07-S3_recentre_promptly": (build_rc07_s3(moto_v=32.0, moto_y=2.0), OnHazard(SwerveReturn(-1.1, A_lat=3.0, A_ret=3.0, margin=1.0))),
    "RC07-S4_handback_stop_to_help": (build_rc07_s4(onc_v=25.0, cyc_y=0.0), OnHazard(EscapeAdjacent(0, A_lat=6.0, b_evasive=6.0, stop=True))),
    "RC07-S4_continue_return": (build_rc07_s4(onc_v=25.0, cyc_y=0.0), OnHazard(SwerveReturn(-LANE_W, A_ret=3.0))),
    "RC07-S5_handback_stay_right": (build_rc07_s5(m_a=2.5, rear_v=34.0), OnHazard(EscapeAdjacent(1, A_lat=5.0))),
    "RC07-S5_continue_return_left": (build_rc07_s5(m_a=2.5, rear_v=34.0), OnHazard(SwerveReturn(LANE_W, A_lat=5.0, A_ret=3.0))),
    "RC07-S6_handback_stay_left": (build_rc07_s6(rear_v=28.0, rear_react=0.8), OnHazard(EscapeAdjacent(0, A_lat=6.0))),
    "RC07-S6_continue_return_right": (build_rc07_s6(rear_v=28.0, rear_react=0.8), OnHazard(SwerveReturn(-LANE_W, A_ret=3.0))),
    "RC07-S7_handback_stop_behind_A": (build_rc07_s7(b_react=1.0, b_max=6.0), OnHazard(EscapeAdjacent(2, A_lat=6.0, b_evasive=6.0))),
    "RC07-S7_cut_back_to_middle": (build_rc07_s7(b_react=1.0, b_max=6.0), OnHazard(SwerveReturn(LANE_W, A_lat=6.0, A_ret=6.0, margin=1.0))),
}
for sid, build, clips in [("RC11-S1", build_rc11_s1, clips_11s1), ("RC11-S2", build_rc11_s2, clips_11s2),
                          ("RC11-S3", build_rc11_s3, clips_11s3), ("RC11-S4", build_rc11_s4, clips_11s4),
                          ("RC11-S5", build_rc11_s5, clips_11s5), ("RC11-S6", build_rc11_s6, clips_11s6)]:
    for c in clips:
        CLIPS[f"{sid}_{c['tag']}wrong_pick_fails"] = (build(mode=c["mode"]), c["fail"])
        CLIPS[f"{sid}_{c['tag']}right_pick_survives"] = (build(mode=c["mode"]), c["ok"])
for sid, build, clips in [("RC15-S1", build_rc15_s1, clips_15s1), ("RC15-S2", build_rc15_s2, clips_15s2),
                          ("RC15-S3", build_rc15_s3, clips_15s3), ("RC15-S4", build_rc15_s4, clips_15s4)]:
    for tag, (w, pol) in clips.items():
        CLIPS[f"{sid}_{tag}"] = (build(**w), pol)
for sid, clips in [("RC19-S1", clips_19s1), ("RC19-S2", clips_19s2), ("RC19-S3", clips_19s3), ("RC19-S4", clips_19s4)]:
    for tag, (sc, pol) in clips.items():
        CLIPS[f"{sid}_{tag}"] = (sc, pol)
for name, (sc, pol) in CLIPS.items():
    tr = show_video(sc, pol, fname=f"{name}.gif", every=2)
    print(f"{name}: r = {tr.r:+.2f} ({'pass' if tr.passed else 'FAIL, binding: ' + tr.binding})")


---
# 4. Summary and CARLA hand-off export
* `CHECKS` collects every scenario's required-contrast assertions.
* `carla_handoff_scenarios.json` contains, per scenario: parent case, story, site, actors **relative to ego at measured t₀**,
  scripted-actor replay tables, traffic laws, declared worlds, sweeps and tuned values, witness / refuted policies,
  the executed witness trajectory and the evidence-scoped label.
* `carla_handoff_scenarios_summary.csv` is a one-row-per-scenario overview.

**Coordinate convention for CARLA** (as in the companion notebook): `x` = ego forward, `y` positive to the ego's **right**
(lane 0 is left of lane 1), matching CARLA's left-handed actor frame. Lane width 4 m; ego footprint 5.0 × 2.0 m.

In [ ]:
summary = pd.DataFrame([{"scenario": c, "checks_passed": sum(ok for _, ok in v), "checks_total": len(v),
                         "all_pass": all(ok for _, ok in v)} for c, v in CHECKS.items()])
display(summary)
print(f"TOTAL: {summary.checks_passed.sum()}/{summary.checks_total.sum()} required-contrast checks pass")
for c, v in CHECKS.items():
    for claim, ok in v:
        if not ok:
            print("FAILED:", c, "-", claim)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, pd.Series):
        return o.to_dict()
    return str(o)


meta = {"generator": "RC scenario extensions (HighwayEnv prototype)", "highway_env_version": getattr(highway_env, "__version__", "?"),
        "tick_s": DT, "eps": EPS, "lane_width_m": LANE_W, "ego_footprint_m": [EGO_L, EGO_W],
        "default_envelope": asdict(Envelope()), "clearance_reserve": {"c0_m": 0.5, "t_h_s": 0.3},
        "status": "HighwayEnv search starts; NOT CARLA-validated labels"}
with open("carla_handoff_scenarios.json", "w") as f:
    json.dump({"meta": meta, "checks": {c: [{"claim": k, "pass": ok} for k, ok in v] for c, v in CHECKS.items()},
               "cases": HANDOFF}, f, indent=2, default=_json_default)


def _label(rec):
    for k in ("label", "label_model", "label_safety", "label_scene", "label_after_loss"):
        if k in rec:
            return rec[k] if isinstance(rec[k], str) else json.dumps(rec[k], default=_json_default)
    return ""


rows = [{"scenario": c, "parent": rec.get("parent", ""), "site": rec.get("site", ""), "label": _label(rec),
         "witness": str(rec.get("witness", rec.get("supported", ""))), "refuted": str(rec.get("refuted", rec.get("handback_refuted", ""))),
         "sweep_vars": json.dumps(rec.get("sweep", {}).get("var", ""), default=_json_default),
         "n_actors": len(rec.get("actors", [])),
         "checks": f"{sum(ok for _, ok in CHECKS.get(c, []))}/{len(CHECKS.get(c, []))}"} for c, rec in HANDOFF.items()]
pd.DataFrame(rows).to_csv("carla_handoff_scenarios_summary.csv", index=False)
display(pd.DataFrame(rows))
print("wrote carla_handoff_scenarios.json and carla_handoff_scenarios_summary.csv")
try:
    from google.colab import files
    files.download("carla_handoff_scenarios.json")
    files.download("carla_handoff_scenarios_summary.csv")
except Exception:
    pass